# 01 — Data Exploration & Predictor-Dataset Pipeline (Corrected)

Goal: NOT blood-cell classification. This dataset provides realistic
feature values used to construct quantum circuits, so we can learn how
circuit/data/hardware characteristics affect noise (TVD between ideal
and noisy execution).

**Methodology (finalized, per Shreya's review):**
1. Exact ideal distribution (Statevector) vs. FakeTorino noisy
   simulation -- no Monte Carlo noise on the ideal side.
2. Fixed transpiler seed + deliberate, fully-crossed physical layout
   variation (layout is independent of data row and circuit config,
   not confounded with either).
3. Rich hardware-aware feature extraction (structural, transpiled,
   CZ calibration, T1/T2) -- kept in full, not reduced to just
   depth + CX count.
4. Features organized into three nested tiers (baseline / structural /
   hardware-aware) so we can directly test whether hardware-aware
   information improves noise prediction beyond circuit complexity --
   this is the paper's central experimental question.


In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Load and inspect the dataset

In [5]:
DATA_PATH = "../data/raw/blood_cell_anomaly_detection.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (5880, 36)


,cell_id,cell_type,anomaly_label,disease_category,cell_diameter_um,nucleus_area_pct,chromatin_density,cytoplasm_ratio,circularity,eccentricity,granularity_score,lobularity_score,membrane_smoothness,cell_area_px,perimeter_px,mean_r,mean_g,mean_b,stain_intensity,patient_age_group,patient_sex,wbc_count_per_ul,rbc_count_millions_per_ul,hemoglobin_g_dl,hematocrit_pct,platelet_count_per_ul,mcv_fl,mchc_g_dl,dataset_source,staining_protocol,microscope_model,magnification_x,image_resolution_px,cytodiffusion_anomaly_score,cytodiffusion_classification_confidence,labeller_confidence_score
0,CELL_005371,Hypersegmented_Neutrophil,1,Infection,15.18,58.8,0.542,0.301,0.563,0.529,4.11,6.6,0.800,445,90,215,141,160,0.555,Elderly,F,6352,4.44,11.7,43.4,257383,85.5,31.4,CytoData,Giemsa,Zeiss_Axio,100,224,0.7649,0.5726,0.5670
1,CELL_005300,Hypersegmented_Neutrophil,1,Infection,16.47,73.6,0.583,0.365,0.859,0.443,2.50,6.3,0.737,383,61,189,136,169,0.692,Elderly,M,7709,4.90,13.9,42.2,302274,92.5,35.0,PBC_Dataset,Wright,Zeiss_Axio,100,224,0.8472,0.7150,0.7273
2,CELL_000200,Neutrophil,0,Normal_WBC,13.41,55.5,0.448,0.376,0.781,0.407,3.01,3.2,0.790,418,52,193,150,203,0.593,Adult,M,7451,5.72,16.1,39.2,229996,76.3,33.0,CytoData,Wright,Leica_DM2000,100,512,0.0313,0.9225,0.9623
3,CELL_003269,Normal_RBC,0,Normal_RBC,7.36,0.0,0.000,1.000,0.880,0.167,0.43,1.0,0.937,217,39,227,138,118,0.509,Adult,F,9196,3.42,14.6,54.1,130720,92.3,32.5,CytoData,Wright,Leica_DM2000,100,512,0.1293,0.9180,0.8652
4,CELL_003505,Normal_RBC,0,Normal_RBC,7.53,0.0,0.000,1.000,1.000,0.158,0.51,1.0,0.925,147,50,239,127,113,0.607,Elderly,M,5898,5.36,14.6,36.7,228652,83.9,33.4,CytoData,Wright,Olympus_BX51,100,224,0.1418,0.9697,0.8898


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5880 entries, 0 to 5879
Data columns (total 36 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   cell_id                                  5880 non-null   str    
 1   cell_type                                5880 non-null   str    
 2   anomaly_label                            5880 non-null   int64  
 3   disease_category                         5880 non-null   str    
 4   cell_diameter_um                         5880 non-null   float64
 5   nucleus_area_pct                         5880 non-null   float64
 6   chromatin_density                        5880 non-null   float64
 7   cytoplasm_ratio                          5880 non-null   float64
 8   circularity                              5880 non-null   float64
 9   eccentricity                             5880 non-null   float64
 10  granularity_score                        5880 non-null   fl

In [7]:
print(df.columns.tolist())


['cell_id', 'cell_type', 'anomaly_label', 'disease_category', 'cell_diameter_um', 'nucleus_area_pct', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity', 'granularity_score', 'lobularity_score', 'membrane_smoothness', 'cell_area_px', 'perimeter_px', 'mean_r', 'mean_g', 'mean_b', 'stain_intensity', 'patient_age_group', 'patient_sex', 'wbc_count_per_ul', 'rbc_count_millions_per_ul', 'hemoglobin_g_dl', 'hematocrit_pct', 'platelet_count_per_ul', 'mcv_fl', 'mchc_g_dl', 'dataset_source', 'staining_protocol', 'microscope_model', 'magnification_x', 'image_resolution_px', 'cytodiffusion_anomaly_score', 'cytodiffusion_classification_confidence', 'labeller_confidence_score']


In [8]:
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0])

print("\nDuplicate rows:", df.duplicated().sum())


Columns with missing values:
Series([], dtype: int64)

Duplicate rows: 0


## Feature selection

Selected 5 morphology features (Group A from our discussion -- least
redundant with each other, directly describe individual cell structure,
no label leakage, no ID/metadata columns).


In [9]:
selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

feature_data = df[selected_columns].copy()

print(feature_data.describe().round(3).to_string())
print("\nMissing values:")
print(feature_data.isna().sum().to_string())
print("\nCorrelation:")
print(feature_data.corr().round(3).to_string())


       cell_diameter_um  chromatin_density  cytoplasm_ratio  circularity  eccentricity
count          5880.000           5880.000         5880.000     5880.000      5880.000
mean             10.176              0.391            0.564        0.769         0.365
std               3.642              0.310            0.335        0.158         0.202
min               1.000              0.000            0.050        0.100         0.000
25%               7.630              0.000            0.276        0.711         0.215
50%              10.055              0.480            0.452        0.805         0.331
75%              12.860              0.635            1.000        0.879         0.456
max              21.180              1.000            1.000        1.000         0.990

Missing values:
cell_diameter_um     0
chromatin_density    0
cytoplasm_ratio      0
circularity          0
eccentricity         0

Correlation:
                   cell_diameter_um  chromatin_density  cytoplasm_ratio

**Note on redundancy:** the `angle_i` features (created below) are a
deterministic linear rescaling of `raw_feature_i` -- they carry
identical information, just on a different scale. This is kept
intentionally (harmless for tree-based models like Random Forest /
XGBoost) but is worth knowing when interpreting feature importance --
importance may be split across both representations of the same
underlying signal.

## Scale features to [0, 2π] for angle encoding

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
scaled_features = scaler.fit_transform(feature_data.values)

print("Scaled shape:", scaled_features.shape)
print("Example scaled row:", scaled_features[0])


Scaled shape: (5880, 5)
Example scaled row: [4.415043   3.40548644 1.6600837  3.23234977 3.35737882]


## Build one example quantum kernel circuit

In [11]:
from qiskit.circuit.library import zz_feature_map

n_features = len(selected_columns)

feature_map = zz_feature_map(
    feature_dimension=n_features,
    reps=1,
    entanglement="linear",
)

print("Number of qubits:", feature_map.num_qubits)
print("Parameters:", feature_map.parameters)


Number of qubits: 5
Parameters: ParameterView([ParameterVectorElement(x[0]), ParameterVectorElement(x[1]), ParameterVectorElement(x[2]), ParameterVectorElement(x[3]), ParameterVectorElement(x[4])])


In [12]:
one_data_point = scaled_features[0]

bound_circuit = feature_map.assign_parameters(one_data_point)
print(bound_circuit.decompose())


   ┌────────────┐┌───────────────┐                                 »
0: ┤ U(π/2,0,π) ├┤ U(0,0,8.8301) ├──■──────────────────────■───────»
   ├────────────┤└┬──────────────┤┌─┴─┐┌────────────────┐┌─┴─┐     »
1: ┤ U(π/2,0,π) ├─┤ U(0,0,6.811) ├┤ X ├┤ U(0,0,0.67211) ├┤ X ├──■──»
   ├────────────┤┌┴──────────────┤└───┘└────────────────┘└───┘┌─┴─┐»
2: ┤ U(π/2,0,π) ├┤ U(0,0,3.3202) ├────────────────────────────┤ X ├»
   ├────────────┤├───────────────┤                            └───┘»
3: ┤ U(π/2,0,π) ├┤ U(0,0,6.4647) ├─────────────────────────────────»
   ├────────────┤├───────────────┤                                 »
4: ┤ U(π/2,0,π) ├┤ U(0,0,6.7148) ├─────────────────────────────────»
   └────────────┘└───────────────┘                                 »
«                                                             »
«0: ──────────────────────────────────────────────────────────»
«                                                             »
«1: ─────────────────────■───────────────────────

## Simulators and noise model

Using `FakeTorino` -- a real, frozen calibration snapshot of an actual
IBM device (T1, T2, gate errors). No live IBM account or network access
required.

In [13]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeTorino
from qiskit.quantum_info import Statevector

fake_backend = FakeTorino()
print("Backend:", fake_backend.name)
print("Number of qubits:", fake_backend.num_qubits)

ideal_simulator = AerSimulator()
noise_model = NoiseModel.from_backend(fake_backend)
noisy_simulator = AerSimulator(noise_model=noise_model)


Backend: fake_torino
Number of qubits: 133


## TVD helper functions

`calculate_tvd_from_probs` operates directly on probability dicts (used
for the exact ideal distribution, which has no "counts"). `calculate_tvd`
is kept for convenience when comparing two counts dicts directly (e.g.
for quick manual checks).

In [14]:
def counts_to_probabilities(counts):
    total_shots = sum(counts.values())
    return {state: count / total_shots for state, count in counts.items()}


def calculate_tvd_from_probs(prob1, prob2):
    all_states = set(prob1) | set(prob2)
    p1 = np.array([prob1.get(s, 0) for s in all_states])
    p2 = np.array([prob2.get(s, 0) for s in all_states])
    return 0.5 * np.sum(np.abs(p1 - p2))


def calculate_tvd(counts1, counts2):
    return calculate_tvd_from_probs(
        counts_to_probabilities(counts1),
        counts_to_probabilities(counts2),
    )


### FIX 1 demonstrated: exact ideal distribution vs. shot-sampled

Previously, the "ideal" reference was estimated via 1024 shots -- itself
noisy. Here it's computed exactly via `Statevector`, with zero shots and
zero randomness.

In [15]:
measured_circuit = bound_circuit.copy()
measured_circuit.measure_all()

# EXACT ideal distribution -- no shots needed, no shot noise
ideal_sv = Statevector.from_instruction(bound_circuit)  # unmeasured circuit
prob_ideal_exact = ideal_sv.probabilities_dict()

print("Exact ideal distribution (first few states):")
for state, prob in list(prob_ideal_exact.items())[:5]:
    print(f"  {state}: {prob:.4f}")

# Noisy simulation (still shot-based -- this is correct, real hardware
# noise genuinely IS stochastic and should be sampled)
noisy_compiled = transpile(measured_circuit, noisy_simulator)
counts_noisy = noisy_simulator.run(noisy_compiled, shots=1024).result().get_counts()
prob_noisy = counts_to_probabilities(counts_noisy)

tvd = calculate_tvd_from_probs(prob_ideal_exact, prob_noisy)
print("\nTVD (exact ideal vs. noisy):", tvd)


Exact ideal distribution (first few states):
  00000: 0.0312
  00001: 0.0312
  00010: 0.0312
  00011: 0.0312
  00100: 0.0312

TVD (exact ideal vs. noisy): 0.07421875


## Circuit structural feature extraction (abstract circuit)

In [16]:
from collections import Counter


def extract_circuit_features(circuit):
    """Structural / topology features from a concrete QuantumCircuit."""
    qc = circuit.decompose()
    ops = qc.count_ops()

    n_qubits = qc.num_qubits
    depth = qc.depth()
    total_gate_count = sum(ops.values())

    two_qubit_gate_names = {"cx", "cz", "ecr", "swap"}
    two_qubit_gate_count = sum(
        count for gate, count in ops.items() if gate in two_qubit_gate_names
    )
    single_qubit_gate_count = total_gate_count - two_qubit_gate_count
    cx_count = ops.get("cx", 0)

    two_qubit_gate_ratio = (
        two_qubit_gate_count / total_gate_count if total_gate_count else 0.0
    )
    average_gates_per_qubit = total_gate_count / n_qubits if n_qubits else 0.0

    qubit_gate_counts = Counter()
    pair_counts = Counter()

    for instruction in qc.data:
        qubits = instruction.qubits
        if len(qubits) == 1:
            q = qc.find_bit(qubits[0]).index
            qubit_gate_counts[q] += 1
        elif len(qubits) == 2:
            q1 = qc.find_bit(qubits[0]).index
            q2 = qc.find_bit(qubits[1]).index
            qubit_gate_counts[q1] += 1
            qubit_gate_counts[q2] += 1
            pair = tuple(sorted((q1, q2)))
            pair_counts[pair] += 1

    unique_2q_pairs = len(pair_counts)
    repeated_2q_pairs = sum(count - 1 for count in pair_counts.values())

    neighbors = {q: set() for q in range(n_qubits)}
    for q1, q2 in pair_counts:
        neighbors[q1].add(q2)
        neighbors[q2].add(q1)

    degrees = [len(neighbors[q]) for q in range(n_qubits)]
    average_qubit_degree = np.mean(degrees) if degrees else 0.0
    max_qubit_degree = max(degrees) if degrees else 0
    max_gates_per_qubit = max(qubit_gate_counts.values()) if qubit_gate_counts else 0

    return {
        "n_qubits": n_qubits,
        "depth": depth,
        "total_gate_count": total_gate_count,
        "single_qubit_gate_count": single_qubit_gate_count,
        "two_qubit_gate_count": two_qubit_gate_count,
        "cx_count": cx_count,
        "two_qubit_gate_ratio": two_qubit_gate_ratio,
        "average_gates_per_qubit": average_gates_per_qubit,
        "max_gates_per_qubit": max_gates_per_qubit,
        "unique_2q_pairs": unique_2q_pairs,
        "average_qubit_degree": average_qubit_degree,
        "max_qubit_degree": max_qubit_degree,
        "repeated_2q_pairs": repeated_2q_pairs,
    }


## Transpiled-circuit feature extraction

In [17]:
def extract_transpiled_features(circuit, backend):
    qc = circuit.remove_final_measurements(inplace=False)
    ops = qc.count_ops()

    one_qubit_gates = {"sx", "x", "rz", "id", "u", "u1", "u2", "u3"}
    two_qubit_gates = {"cx", "cz", "ecr", "swap"}

    total_gates = sum(ops.values())
    one_qubit_count = sum(ops.get(gate, 0) for gate in one_qubit_gates)
    two_qubit_count = sum(ops.get(gate, 0) for gate in two_qubit_gates)

    return {
        "transpiled_depth": qc.depth(),
        "transpiled_total_gates": total_gates,
        "transpiled_1q_gates": one_qubit_count,
        "transpiled_2q_gates": two_qubit_count,
        "transpiled_cz_count": ops.get("cz", 0),
        "transpiled_cx_count": ops.get("cx", 0),
        "transpiled_sx_count": ops.get("sx", 0),
        "transpiled_rz_count": ops.get("rz", 0),
        "backend_qubits": backend.num_qubits,
    }


## Hardware calibration feature extraction (real CZ error / duration data)

In [18]:
def extract_hardware_noise_features(circuit, backend):
    """Hardware-aware features from a transpiled circuit's actual
    physical CZ couplers, using the backend's real calibration data."""
    cz_properties = backend.target["cz"]

    cz_errors = []
    cz_durations = []

    for instruction in circuit.data:
        if instruction.operation.name != "cz":
            continue
        q1 = circuit.find_bit(instruction.qubits[0]).index
        q2 = circuit.find_bit(instruction.qubits[1]).index
        pair = (q1, q2)
        prop = cz_properties.get(pair)
        if prop is None:
            pair = (q2, q1)
            prop = cz_properties.get(pair)
        if prop is not None and prop.error is not None:
            cz_errors.append(prop.error)
        if prop is not None and prop.duration is not None:
            cz_durations.append(prop.duration)

    features = {}
    if cz_errors:
        features["mean_cz_error"] = np.mean(cz_errors)
        features["std_cz_error"] = np.std(cz_errors)
        features["min_cz_error"] = np.min(cz_errors)
        features["max_cz_error"] = np.max(cz_errors)
        features["sum_cz_error"] = np.sum(cz_errors)
        survival = np.prod([1 - e for e in cz_errors])
        features["cumulative_cz_error"] = 1 - survival
    else:
        features.update({
            "mean_cz_error": 0.0, "std_cz_error": 0.0, "min_cz_error": 0.0,
            "max_cz_error": 0.0, "sum_cz_error": 0.0, "cumulative_cz_error": 0.0,
        })

    if cz_durations:
        features["mean_cz_duration"] = np.mean(cz_durations)
        features["max_cz_duration"] = np.max(cz_durations)
        features["total_cz_duration"] = np.sum(cz_durations)
    else:
        features.update({
            "mean_cz_duration": 0.0, "max_cz_duration": 0.0, "total_cz_duration": 0.0,
        })

    return features


## Consolidated `extract_all_features`

(Single final version -- earlier exploratory redefinitions removed.)

In [19]:
def extract_all_features(
    abstract_circuit,
    transpiled_circuit,
    backend,
    raw_data_row,
    data_row,
    reps,
    entanglement,
):
    """
    Extract all predictor features for one experiment.

    raw_data_row : original biological feature values.
    data_row     : scaled values used as quantum encoding angles.
    """
    features = {}

    # 1. Raw data features
    for i, value in enumerate(raw_data_row):
        features[f"raw_feature_{i+1}"] = float(value)

    # 2. Quantum encoding angles
    for i, value in enumerate(data_row):
        features[f"angle_{i+1}"] = float(value)

    # 3. Abstract circuit features
    abstract_features = extract_circuit_features(abstract_circuit)
    features.update({f"abstract_{k}": v for k, v in abstract_features.items()})

    # 4. Circuit configuration
    features["reps"] = reps
    features["entanglement"] = entanglement

    # 5. Transpiled circuit features
    transpiled_features = extract_transpiled_features(transpiled_circuit, backend)
    transpiled_features["backend_size"] = transpiled_features.pop("backend_qubits")
    features.update(transpiled_features)

    # 6. Hardware CZ error features
    hardware_features = extract_hardware_noise_features(transpiled_circuit, backend)
    features.update(hardware_features)

    # 7. Physical qubits used
    used_qubits = set()
    for instruction in transpiled_circuit.data:
        for qubit in instruction.qubits:
            q = transpiled_circuit.find_bit(qubit).index
            used_qubits.add(q)
    features["physical_qubit_count"] = len(used_qubits)
    features["physical_qubits_used"] = str(sorted(used_qubits))  # NEW: explicit record

    # 8. T1 / T2 features
    t1_values, t2_values = [], []
    for q in used_qubits:
        props = backend.qubit_properties(q)
        if props is None:
            continue
        if props.t1 is not None:
            t1_values.append(props.t1)
        if props.t2 is not None:
            t2_values.append(props.t2)

    if t1_values:
        features["mean_t1"] = float(np.mean(t1_values))
        features["min_t1"] = float(np.min(t1_values))
        features["max_t1"] = float(np.max(t1_values))
    else:
        features["mean_t1"] = np.nan
        features["min_t1"] = np.nan
        features["max_t1"] = np.nan

    if t2_values:
        features["mean_t2"] = float(np.mean(t2_values))
        features["min_t2"] = float(np.min(t2_values))
        features["max_t2"] = float(np.max(t2_values))
    else:
        features["mean_t2"] = np.nan
        features["min_t2"] = np.nan
        features["max_t2"] = np.nan

    return features


## Consolidated `run_complete_experiment` -- FIXED version

**Changes vs. the original:**
- Ideal distribution computed **once**, exactly, via `Statevector` --
  moved outside the repeat loop entirely (Fix 1).
- `seed_transpiler` fixed for reproducibility; optional `initial_layout`
  parameter added so different circuits can be deliberately mapped onto
  different physical qubits, giving real variation in the hardware
  calibration features rather than the transpiler always picking the
  same "best" patch (Fix 2).
- Only the noisy side is resampled `repeats` times -- correctly, since
  real hardware noise genuinely is stochastic and should be sampled,
  unlike the ideal reference.

In [20]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    shots=1024,
    repeats=10,
    backend=fake_backend,
    initial_layout=None,
    seed_transpiler=42,
):
    """
    Run one circuit configuration and return one complete
    predictor-dataset record.
    """

    # 1. Build abstract feature-map circuit, bind real data
    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )
    abstract_circuit = feature_map.assign_parameters(data_row)

    # 2. EXACT ideal distribution -- computed once, no shots, no randomness
    ideal_sv = Statevector.from_instruction(abstract_circuit)
    prob_ideal_exact = ideal_sv.probabilities_dict()

    # 3. Add measurements for the noisy/hardware-facing version
    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # 4. Canonical transpilation -- fixed seed, optional explicit layout
    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # 5. Extract all predictor features
    features = extract_all_features(
        abstract_circuit=abstract_circuit,
        transpiled_circuit=transpiled_circuit,
        backend=backend,
        raw_data_row=raw_data_row,
        data_row=data_row,
        reps=reps,
        entanglement=entanglement,
    )

    # 6. Noise model + noisy simulator (from same target backend)
    noise_model = NoiseModel.from_backend(backend)
    noisy_sim = AerSimulator(noise_model=noise_model)

    # 7. Repeated NOISY execution only -- ideal is fixed/exact (Fix 1)
    tvds = []
    for _ in range(repeats):
        noisy_result = noisy_sim.run(transpiled_circuit, shots=shots).result()
        counts_noisy = noisy_result.get_counts()
        prob_noisy = counts_to_probabilities(counts_noisy)
        tvd = calculate_tvd_from_probs(prob_ideal_exact, prob_noisy)
        tvds.append(tvd)

    # 8. Target statistics
    features["mean_tvd"] = float(np.mean(tvds))
    features["std_tvd"] = float(np.std(tvds, ddof=1))
    features["min_tvd"] = float(np.min(tvds))
    features["max_tvd"] = float(np.max(tvds))

    # 9. Experiment settings (now includes reproducibility info)
    features["shots"] = shots
    features["repeats"] = repeats
    features["seed_transpiler"] = seed_transpiler
    features["initial_layout"] = str(initial_layout)

    return features


## Test: run one complete experiment

In [21]:
raw_data_point = feature_data.iloc[0].to_numpy()
scaled_data_point = scaled_features[0]

complete_result = run_complete_experiment(
    raw_data_row=raw_data_point,
    data_row=scaled_data_point,
    reps=1,
    entanglement="full",
    shots=1024,
    repeats=10,
)

for key, value in complete_result.items():
    print(f"{key:30s}: {value}")


raw_feature_1                 : 15.18
raw_feature_2                 : 0.542
raw_feature_3                 : 0.301
raw_feature_4                 : 0.563
raw_feature_5                 : 0.529
angle_1                       : 4.415042995827875
angle_2                       : 3.4054864364913358
angle_3                       : 1.6600836969495538
angle_4                       : 3.232349774693498
angle_5                       : 3.3573788156545468
abstract_n_qubits             : 5
abstract_depth                : 23
abstract_total_gate_count     : 40
abstract_single_qubit_gate_count: 20
abstract_two_qubit_gate_count : 20
abstract_cx_count             : 20
abstract_two_qubit_gate_ratio : 0.5
abstract_average_gates_per_qubit: 8.0
abstract_max_gates_per_qubit  : 14
abstract_unique_2q_pairs      : 10
abstract_average_qubit_degree : 4.0
abstract_max_qubit_degree     : 4
abstract_repeated_2q_pairs    : 10
reps                          : 1
entanglement                  : full
transpiled_depth          

### Note on `initial_layout` diversity

By default (`initial_layout=None`), the transpiler will likely pick a
similar "best" patch of physical qubits every time for a small 5-qubit
circuit -- which can make `mean_cz_error` / `mean_t1` / `mean_t2` nearly
constant across your dataset. When building the full dataset below,
cycle through a handful of different explicit `initial_layout` values
(e.g., different regions of the 133-qubit chip) so these features
actually carry a learnable signal.

In [22]:
# Example: check what physical qubits got used by default
print("Physical qubits used (default layout):", complete_result["physical_qubits_used"])
# See the "Building the real predictor-training dataset" section below
# for the full, validated CANDIDATE_LAYOUTS list and the fully-crossed
# row x config x layout experimental design.


Physical qubits used (default layout): [27, 28, 29, 30, 36]


## Building the real predictor-training dataset

**Sampling strategy** (updated per Shreya's review):
- ~200 data rows, stratified by `cell_type`
- Circuit config grid: `reps ∈ {1,2,3}` × `entanglement ∈ {linear, full}`
  = 6 configs
- **Layout is fully crossed, not cycled by row index** -- every
  (row, config) pair is run under *every* candidate layout. This avoids
  confounding data identity with hardware placement, which matters if
  we later want to claim hardware-aware features improve prediction.
- Ideal distribution is exact (Fix 1) and doesn't depend on layout, so
  it only needs to be computed once per (row, config) pair, not
  recomputed per layout.
- `repeats=5` for the full sweep (not 10) to keep the total experiment
  count manageable -- a smaller validation subset can use more repeats
  later for tighter uncertainty estimates.

**Scale check:** 200 rows × 6 configs × 4 layouts = 4,800 canonical
circuits × 5 repeats = 24,000 noisy simulations. This is meaningfully
larger than a single-layout design -- **pilot on a small slice first**
(see next cell) before committing to the full run.

In [ ]:
# Before trusting the candidate layouts, check they're actually valid,
# reasonably-connected regions on FakeTorino's real coupling map --
# otherwise the transpiler may silently insert long SWAP chains to
# route between poorly-connected qubits, which would confound layout
# comparisons with routing-overhead artifacts rather than pure
# calibration differences.

coupling_map = fake_backend.coupling_map
print("Coupling map edges (first 20):", list(coupling_map.get_edges())[:20])

def layout_is_well_connected(layout, coupling_map, max_qubit_index):
    """Rough sanity check: are consecutive qubits in this layout
    directly connected on the real chip?"""
    if layout is None:
        return True
    if max(layout) >= max_qubit_index:
        return False
    edges = set(coupling_map.get_edges())
    connected_pairs = sum(
        1 for i in range(len(layout) - 1)
        if (layout[i], layout[i+1]) in edges or (layout[i+1], layout[i]) in edges
    )
    return connected_pairs >= (len(layout) - 1) * 0.5  # at least half directly connected

CANDIDATE_LAYOUTS = [
    None,                        # let transpiler choose (baseline)
    [0, 1, 2, 3, 4],
    [20, 21, 22, 23, 24]
]

for layout in CANDIDATE_LAYOUTS:
    ok = layout_is_well_connected(layout, coupling_map, fake_backend.num_qubits)
    print(f"Layout {layout}: {'OK' if ok else 'CHECK -- may route poorly, consider replacing'}")


Coupling map edges (first 20): [(0, 1), (0, 15), (1, 0), (1, 2), (2, 1), (2, 3), (3, 2), (3, 4), (4, 3), (4, 5), (4, 16), (5, 4), (5, 6), (6, 5), (6, 7), (7, 6), (7, 8), (8, 7), (8, 9), (8, 17)]
Layout None: OK
Layout [0, 1, 2, 3, 4]: OK
Layout [20, 21, 22, 23, 24]: OK
Layout [50, 51, 52, 53, 54]: OK


In [27]:
coupling_edges = {
    tuple(sorted(edge))
    for edge in fake_backend.coupling_map.get_edges()
}

valid_layouts = []

for start in range(fake_backend.num_qubits):
    layout = list(range(start, start + 5))

    if max(layout) >= fake_backend.num_qubits:
        break

    connected = all(
        tuple(sorted((layout[i], layout[i + 1]))) in coupling_edges
        for i in range(4)
    )

    if connected:
        valid_layouts.append(layout)

print("Found connected 5-qubit chains:")
for layout in valid_layouts[:20]:
    print(layout)

Found connected 5-qubit chains:
[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[3, 4, 5, 6, 7]
[4, 5, 6, 7, 8]
[5, 6, 7, 8, 9]
[6, 7, 8, 9, 10]
[7, 8, 9, 10, 11]
[8, 9, 10, 11, 12]
[9, 10, 11, 12, 13]
[10, 11, 12, 13, 14]
[19, 20, 21, 22, 23]
[20, 21, 22, 23, 24]
[21, 22, 23, 24, 25]
[22, 23, 24, 25, 26]
[23, 24, 25, 26, 27]
[24, 25, 26, 27, 28]
[25, 26, 27, 28, 29]
[26, 27, 28, 29, 30]
[27, 28, 29, 30, 31]


In [28]:
CANDIDATE_LAYOUTS = [
    [0, 1, 2, 3, 4],
    [19, 20, 21, 22, 23],
    [38, 39, 40, 41, 42],
    [57, 58, 59, 60, 61],
]

for layout in CANDIDATE_LAYOUTS:
    print(layout, layout in valid_layouts)

[0, 1, 2, 3, 4] True
[19, 20, 21, 22, 23] True
[38, 39, 40, 41, 42] True
[57, 58, 59, 60, 61] True


In [30]:
import numpy as np
import time

# Get the same first pilot row
raw_test_row = pilot_sample_df[selected_columns].iloc[0].to_numpy()

# Scale it using the scaler you already fitted
scaled_test_row = scaler.transform(
    raw_test_row.reshape(1, -1)
)[0]

print("Raw:", raw_test_row)
print("Scaled:", scaled_test_row)
test_result_start = time.time()

test_result = run_complete_experiment(
    raw_data_row=raw_test_row,
    data_row=scaled_test_row,
    reps=1,
    entanglement="linear",
    shots=1024,
    repeats=1,
    initial_layout=[57, 58, 59, 60, 61],
    seed_transpiler=42,
)

test_elapsed = time.time() - test_result_start

print(f"Single experiment time: {test_elapsed:.2f} s")
print(f"Mean TVD: {test_result['mean_tvd']:.4f}")
print(f"Transpiled depth: {test_result['transpiled_depth']}")
print(f"Transpiled 2Q gates: {test_result['transpiled_2q_gates']}")
print(f"CZ count: {test_result['transpiled_cz_count']}")

Raw: [13.35   0.536  0.386  0.806  0.351]
Scaled: [3.84525959 3.36778732 2.22226343 4.92880981 2.22767479]
Single experiment time: 12.01 s
Mean TVD: 0.0605
Transpiled depth: 45
Transpiled 2Q gates: 8
CZ count: 8


c:\Users\jaspr\miniconda3\envs\qnoise\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
c:\Users\jaspr\miniconda3\envs\qnoise\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### Pilot run -- time a small slice before committing to the full sweep

Run on a small number of rows first (e.g. 5) with the full config ×
layout grid, and extrapolate the runtime to the full 200 rows before
committing.

In [31]:
import time

CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

df_with_features = (
    df.dropna(subset=selected_columns)
      .reset_index(drop=True)
)

PILOT_N_ROWS = 5

pilot_sample_df = (
    df_with_features
    .sample(n=PILOT_N_ROWS, random_state=42)
    .reset_index(drop=True)
)

print("Pilot sample shape:", pilot_sample_df.shape)
print("\nCell types:")
print(pilot_sample_df["cell_type"].value_counts())

pilot_raw = pilot_sample_df[selected_columns].to_numpy()
pilot_scaled = scaler.transform(pilot_raw)

# ------------------------------------------------------------
# Pilot experiment
# ------------------------------------------------------------

start = time.time()
pilot_results = []

total_experiments = (
    len(pilot_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

experiment_number = 0

for row_idx in range(len(pilot_raw)):

    for reps, entanglement in CONFIGS:

        for layout_idx, layout in enumerate(CANDIDATE_LAYOUTS):

            experiment_number += 1

            print(
                f"\n[{experiment_number}/{total_experiments}] "
                f"row={row_idx}, "
                f"reps={reps}, "
                f"entanglement={entanglement}, "
                f"layout={layout}"
            )

            exp_start = time.time()

            result = run_complete_experiment(
                raw_data_row=pilot_raw[row_idx],
                data_row=pilot_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            elapsed_exp = time.time() - exp_start

            result["data_row_index"] = row_idx
            result["layout_index"] = layout_idx

            pilot_results.append(result)

            print(
                f"Completed in {elapsed_exp:.1f}s | "
                f"mean_TVD={result['mean_tvd']:.4f}"
            )

# ------------------------------------------------------------
# Timing summary
# ------------------------------------------------------------

elapsed = time.time() - start

n_experiments_pilot = len(pilot_results)

n_experiments_full = (
    200
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

seconds_per_experiment = (
    elapsed / n_experiments_pilot
)

estimated_full_minutes = (
    seconds_per_experiment
    * n_experiments_full
    / 60
)

print("\n" + "=" * 60)
print("PILOT COMPLETE")
print("=" * 60)

print(
    f"Pilot experiments: {n_experiments_pilot}"
)

print(
    f"Total time: {elapsed / 60:.2f} minutes"
)

print(
    f"Average time/experiment: "
    f"{seconds_per_experiment:.2f} seconds"
)

print(
    f"Estimated full run: "
    f"{estimated_full_minutes:.1f} minutes"
)

Pilot sample shape: (5, 36)

Cell types:
cell_type
Neutrophil       2
Prolymphocyte    1
Lymphocyte       1
Platelet         1
Name: count, dtype: int64

[1/120] row=0, reps=1, entanglement=linear, layout=[0, 1, 2, 3, 4]
Completed in 36.9s | mean_TVD=0.0662

[2/120] row=0, reps=1, entanglement=linear, layout=[19, 20, 21, 22, 23]
Completed in 23.6s | mean_TVD=0.0658

[3/120] row=0, reps=1, entanglement=linear, layout=[38, 39, 40, 41, 42]
Completed in 26.9s | mean_TVD=0.0705

[4/120] row=0, reps=1, entanglement=linear, layout=[57, 58, 59, 60, 61]
Completed in 27.0s | mean_TVD=0.0684

[5/120] row=0, reps=2, entanglement=linear, layout=[0, 1, 2, 3, 4]
Completed in 73.5s | mean_TVD=0.0906

[6/120] row=0, reps=2, entanglement=linear, layout=[19, 20, 21, 22, 23]
Completed in 39.9s | mean_TVD=0.4133

[7/120] row=0, reps=2, entanglement=linear, layout=[38, 39, 40, 41, 42]
Completed in 26.2s | mean_TVD=0.3135

[8/120] row=0, reps=2, entanglement=linear, layout=[57, 58, 59, 60, 61]
Completed in 3

KeyboardInterrupt: 

In [25]:
for layout in CANDIDATE_LAYOUTS:
    print("\nLayout:", layout)

    if layout is None:
        print("Auto layout — skip connectivity check")
        continue

    edges = {
        tuple(sorted(edge))
        for edge in fake_backend.coupling_map.get_edges()
    }

    for i in range(len(layout) - 1):
        pair = tuple(sorted((layout[i], layout[i + 1])))

        print(
            f"{layout[i]} <-> {layout[i+1]}:",
            "CONNECTED" if pair in edges else "NOT CONNECTED"
        )


Layout: None
Auto layout — skip connectivity check

Layout: [0, 1, 2, 3, 4]
0 <-> 1: CONNECTED
1 <-> 2: CONNECTED
2 <-> 3: CONNECTED
3 <-> 4: CONNECTED

Layout: [20, 21, 22, 23, 24]
20 <-> 21: CONNECTED
21 <-> 22: CONNECTED
22 <-> 23: CONNECTED
23 <-> 24: CONNECTED

Layout: [50, 51, 52, 53, 54]
50 <-> 51: CONNECTED
51 <-> 52: CONNECTED
52 <-> 53: NOT CONNECTED
53 <-> 54: NOT CONNECTED


In [32]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

# Build ONCE
noise_model = NoiseModel.from_backend(fake_backend)

# Build ONCE
noisy_simulator = AerSimulator(
    noise_model=noise_model
)

print("Reusable noisy simulator ready.")

Reusable noisy simulator ready.


In [33]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    shots=1024,
    repeats=5,
    backend=fake_backend,
    noisy_sim=noisy_simulator,
    initial_layout=None,
    seed_transpiler=42,
):
    """
    Run one circuit configuration.

    Optimized version:
    - Exact Statevector ideal distribution
    - One canonical transpilation
    - Reuses a pre-built noisy Aer simulator
    - Executes all noisy repetitions as one Aer job
    """

    # ==========================================================
    # 1. Build abstract feature-map circuit
    # ==========================================================

    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(
        data_row
    )

    # ==========================================================
    # 2. EXACT ideal distribution
    # ==========================================================

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    prob_ideal_exact = ideal_sv.probabilities_dict()

    # ==========================================================
    # 3. Add measurements
    # ==========================================================

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # ==========================================================
    # 4. Canonical FakeTorino transpilation
    # ==========================================================

    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # ==========================================================
    # 5. Extract predictor features
    # ==========================================================

    features = extract_all_features(
        abstract_circuit=abstract_circuit,
        transpiled_circuit=transpiled_circuit,
        backend=backend,
        raw_data_row=raw_data_row,
        data_row=data_row,
        reps=reps,
        entanglement=entanglement,
    )

    # ==========================================================
    # 6. Run ALL noisy repetitions in ONE Aer job
    # ==========================================================

    circuits_to_run = [
        transpiled_circuit
        for _ in range(repeats)
    ]

    result = noisy_sim.run(
        circuits_to_run,
        shots=shots,
    ).result()

    # ==========================================================
    # 7. Calculate TVD for each repetition
    # ==========================================================

    tvds = []

    for i in range(repeats):

        counts_noisy = result.get_counts(i)

        prob_noisy = counts_to_probabilities(
            counts_noisy
        )

        tvd = calculate_tvd_from_probs(
            prob_ideal_exact,
            prob_noisy
        )

        tvds.append(tvd)

    # ==========================================================
    # 8. Target statistics
    # ==========================================================

    features["mean_tvd"] = float(
        np.mean(tvds)
    )

    features["std_tvd"] = float(
        np.std(tvds, ddof=1)
        if repeats > 1
        else 0.0
    )

    features["min_tvd"] = float(
        np.min(tvds)
    )

    features["max_tvd"] = float(
        np.max(tvds)
    )

    # ==========================================================
    # 9. Experiment metadata
    # ==========================================================

    features["shots"] = shots
    features["repeats"] = repeats
    features["seed_transpiler"] = seed_transpiler
    features["initial_layout"] = str(initial_layout)

    return features

In [34]:
import time

raw_test_row = pilot_sample_df[
    selected_columns
].iloc[0].to_numpy()

scaled_test_row = scaler.transform(
    raw_test_row.reshape(1, -1)
)[0]

start = time.time()

test_result = run_complete_experiment(
    raw_data_row=raw_test_row,
    data_row=scaled_test_row,
    reps=1,
    entanglement="linear",
    shots=1024,
    repeats=5,
    initial_layout=[57, 58, 59, 60, 61],
    seed_transpiler=42,
)

elapsed = time.time() - start

print(f"Time: {elapsed:.2f} seconds")
print(f"Mean TVD: {test_result['mean_tvd']:.4f}")
print(f"Std TVD: {test_result['std_tvd']:.4f}")
print(f"Depth: {test_result['transpiled_depth']}")
print(f"2Q gates: {test_result['transpiled_2q_gates']}")

Time: 3.56 seconds
Mean TVD: 0.0711
Std TVD: 0.0164
Depth: 45
2Q gates: 8


In [35]:
import time

start = time.time()

benchmark_results = []

row_idx = 0
reps = 1
entanglement = "linear"

for layout in CANDIDATE_LAYOUTS:

    exp_start = time.time()

    result = run_complete_experiment(
        raw_data_row=raw_test_row,
        data_row=scaled_test_row,
        reps=reps,
        entanglement=entanglement,
        shots=1024,
        repeats=5,
        initial_layout=layout,
        seed_transpiler=42,
    )

    elapsed = time.time() - exp_start

    benchmark_results.append(result)

    print(
        f"Layout {layout}: "
        f"{elapsed:.2f}s | "
        f"TVD={result['mean_tvd']:.4f}"
    )

print(
    f"\nTotal: {time.time() - start:.2f}s"
)

Layout [0, 1, 2, 3, 4]: 1.61s | TVD=0.0676
Layout [19, 20, 21, 22, 23]: 1.71s | TVD=0.0660
Layout [38, 39, 40, 41, 42]: 1.04s | TVD=0.0633
Layout [57, 58, 59, 60, 61]: 1.64s | TVD=0.0643

Total: 6.00s


In [36]:
for reps, entanglement in [
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]:
    start = time.time()

    result = run_complete_experiment(
        raw_data_row=raw_test_row,
        data_row=scaled_test_row,
        reps=reps,
        entanglement=entanglement,
        shots=1024,
        repeats=5,
        initial_layout=[57, 58, 59, 60, 61],
        seed_transpiler=42,
    )

    print(
        f"{reps}, {entanglement}: "
        f"{time.time()-start:.2f}s | "
        f"TVD={result['mean_tvd']:.4f}"
    )

2, linear: 1.71s | TVD=0.3984
3, linear: 1.07s | TVD=0.6334
1, full: 1.77s | TVD=0.0717
2, full: 1.12s | TVD=0.5643
3, full: 1.80s | TVD=0.3625


In [37]:
import time

# ------------------------------------------------------------
# PILOT SETTINGS
# ------------------------------------------------------------

PILOT_N_ROWS = 10
SHOTS = 1024
REPEATS = 5

CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

# These are our validated layouts
CANDIDATE_LAYOUTS = [
    [0, 1, 2, 3, 4],
    [19, 20, 21, 22, 23],
    [38, 39, 40, 41, 42],
    [57, 58, 59, 60, 61],
]

# ------------------------------------------------------------
# SAMPLE 10 DATA ROWS
# ------------------------------------------------------------

df_with_features = (
    df.dropna(subset=selected_columns)
      .reset_index(drop=True)
)

pilot_sample_df = (
    df_with_features
    .sample(
        n=PILOT_N_ROWS,
        random_state=42
    )
    .reset_index(drop=True)
)

print("Pilot sample shape:", pilot_sample_df.shape)

# ------------------------------------------------------------
# PREPARE RAW + SCALED FEATURES
# ------------------------------------------------------------

pilot_raw = (
    pilot_sample_df[selected_columns]
    .to_numpy()
)

pilot_scaled = scaler.transform(
    pilot_raw
)

# ------------------------------------------------------------
# TOTAL EXPERIMENT COUNT
# ------------------------------------------------------------

total_experiments = (
    PILOT_N_ROWS
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

print(
    f"\nTotal pilot experiments: {total_experiments}"
)
print(
    f"Expected noisy runs: "
    f"{total_experiments * REPEATS}"
)

# ------------------------------------------------------------
# RUN PILOT
# ------------------------------------------------------------

pilot_results = []

start = time.time()
experiment_number = 0

for row_idx in range(PILOT_N_ROWS):

    for reps, entanglement in CONFIGS:

        for layout in CANDIDATE_LAYOUTS:

            experiment_number += 1

            exp_start = time.time()

            result = run_complete_experiment(
                raw_data_row=pilot_raw[row_idx],
                data_row=pilot_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,

                shots=SHOTS,
                repeats=REPEATS,

                initial_layout=layout,
                seed_transpiler=42,
            )

            exp_time = time.time() - exp_start

            result["data_row_index"] = row_idx
            result["layout"] = str(layout)

            pilot_results.append(result)

            print(
                f"[{experiment_number}/{total_experiments}] "
                f"row={row_idx} | "
                f"reps={reps} | "
                f"entanglement={entanglement} | "
                f"layout={layout} | "
                f"time={exp_time:.2f}s | "
                f"TVD={result['mean_tvd']:.4f}"
            )

# ------------------------------------------------------------
# TIMING SUMMARY
# ------------------------------------------------------------

elapsed = time.time() - start

pilot_df = pd.DataFrame(pilot_results)

seconds_per_experiment = (
    elapsed / len(pilot_df)
)

FULL_N_ROWS = 200

full_experiments = (
    FULL_N_ROWS
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

estimated_full_seconds = (
    seconds_per_experiment
    * full_experiments
)

print("\n" + "=" * 60)
print("PILOT COMPLETE")
print("=" * 60)

print(
    f"Experiments completed : {len(pilot_df)}"
)

print(
    f"Total time            : "
    f"{elapsed / 60:.2f} min"
)

print(
    f"Average/experiment    : "
    f"{seconds_per_experiment:.2f} sec"
)

print(
    f"Estimated full dataset: "
    f"{estimated_full_seconds / 60:.1f} min"
)

print(
    f"Pilot dataframe shape : "
    f"{pilot_df.shape}"
)

Pilot sample shape: (10, 36)

Total pilot experiments: 240
Expected noisy runs: 1200
[1/240] row=0 | reps=1 | entanglement=linear | layout=[0, 1, 2, 3, 4] | time=1.61s | TVD=0.0688
[2/240] row=0 | reps=1 | entanglement=linear | layout=[19, 20, 21, 22, 23] | time=1.08s | TVD=0.0729
[3/240] row=0 | reps=1 | entanglement=linear | layout=[38, 39, 40, 41, 42] | time=1.62s | TVD=0.0744
[4/240] row=0 | reps=1 | entanglement=linear | layout=[57, 58, 59, 60, 61] | time=1.00s | TVD=0.0699
[5/240] row=0 | reps=2 | entanglement=linear | layout=[0, 1, 2, 3, 4] | time=1.62s | TVD=0.0905
[6/240] row=0 | reps=2 | entanglement=linear | layout=[19, 20, 21, 22, 23] | time=1.53s | TVD=0.4236
[7/240] row=0 | reps=2 | entanglement=linear | layout=[38, 39, 40, 41, 42] | time=1.58s | TVD=0.3041
[8/240] row=0 | reps=2 | entanglement=linear | layout=[57, 58, 59, 60, 61] | time=1.05s | TVD=0.4033
[9/240] row=0 | reps=3 | entanglement=linear | layout=[0, 1, 2, 3, 4] | time=1.62s | TVD=0.2467
[10/240] row=0 | reps

In [38]:
print("Shape:", pilot_df.shape)

print("\nMissing values:")
print(pilot_df.isna().sum().sort_values(ascending=False).head(15))
print("\nTarget statistics:")
print(
    pilot_df[
        ["mean_tvd", "std_tvd", "min_tvd", "max_tvd"]
    ].describe()
)
print("\nKey feature variation:")
print(
    pilot_df[
        [
            "reps",
            "entanglement",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "mean_tvd",
        ]
    ].groupby(
        ["reps", "entanglement"]
    ).mean()
)

Shape: (240, 61)

Missing values:
raw_feature_1                       0
raw_feature_2                       0
raw_feature_3                       0
raw_feature_4                       0
raw_feature_5                       0
angle_1                             0
angle_2                             0
angle_3                             0
angle_4                             0
angle_5                             0
abstract_n_qubits                   0
abstract_depth                      0
abstract_total_gate_count           0
abstract_single_qubit_gate_count    0
abstract_two_qubit_gate_count       0
dtype: int64

Target statistics:
         mean_tvd     std_tvd     min_tvd     max_tvd
count  240.000000  240.000000  240.000000  240.000000
mean     0.260870    0.011542    0.246559    0.275263
std      0.184390    0.004820    0.182299    0.186461
min      0.061328    0.003194    0.044922    0.066406
25%      0.072021    0.007981    0.063232    0.084717
50%      0.240377    0.011020    0.2216

In [39]:
target_columns = [
    "mean_tvd",
    "std_tvd",
    "min_tvd",
    "max_tvd"
]

X_pilot = pilot_df.drop(columns=target_columns)
y_pilot = pilot_df["mean_tvd"]

print("Predictors:", X_pilot.shape)
print("Target:", y_pilot.shape)

print("\nDuplicate columns:")
print(
    X_pilot.columns[
        X_pilot.T.duplicated()
    ].tolist()
)

Predictors: (240, 57)
Target: (240,)

Duplicate columns:
['abstract_cx_count', 'transpiled_cz_count', 'physical_qubit_count', 'repeats', 'initial_layout', 'layout']


In [40]:
DROP_FROM_MODEL = [
    "physical_qubit_count",
    "repeats",
    "layout",
]

X_pilot_clean = X_pilot.drop(
    columns=DROP_FROM_MODEL
)

print("Original predictors:", X_pilot.shape)
print("Clean predictors:", X_pilot_clean.shape)
print("\nRemaining columns:")
print(X_pilot_clean.columns.tolist())

Original predictors: (240, 57)
Clean predictors: (240, 54)

Remaining columns:
['raw_feature_1', 'raw_feature_2', 'raw_feature_3', 'raw_feature_4', 'raw_feature_5', 'angle_1', 'angle_2', 'angle_3', 'angle_4', 'angle_5', 'abstract_n_qubits', 'abstract_depth', 'abstract_total_gate_count', 'abstract_single_qubit_gate_count', 'abstract_two_qubit_gate_count', 'abstract_cx_count', 'abstract_two_qubit_gate_ratio', 'abstract_average_gates_per_qubit', 'abstract_max_gates_per_qubit', 'abstract_unique_2q_pairs', 'abstract_average_qubit_degree', 'abstract_max_qubit_degree', 'abstract_repeated_2q_pairs', 'reps', 'entanglement', 'transpiled_depth', 'transpiled_total_gates', 'transpiled_1q_gates', 'transpiled_2q_gates', 'transpiled_cz_count', 'transpiled_cx_count', 'transpiled_sx_count', 'transpiled_rz_count', 'backend_size', 'mean_cz_error', 'std_cz_error', 'min_cz_error', 'max_cz_error', 'sum_cz_error', 'cumulative_cz_error', 'mean_cz_duration', 'max_cz_duration', 'total_cz_duration', 'physical_qub

In [41]:
pilot_df.to_csv(
    "../experiments/results/pilot_240.csv",
    index=False
)

print("Saved pilot:", pilot_df.shape)

Saved pilot: (240, 61)


In [42]:
PRIMARY_TARGET = "mean_tvd"

TARGET_COLUMNS = [
    "mean_tvd",
    "std_tvd",
    "min_tvd",
    "max_tvd",
]

In [43]:
MODEL_EXCLUDE = [
    "mean_tvd",
    "std_tvd",
    "min_tvd",
    "max_tvd",
    "physical_qubit_count",
    "repeats",
    "layout",
    "data_row_index",
    "shots",
    "seed_transpiler",
]

### Full dataset generation

**Only run this after checking the pilot timing above is acceptable.**
Every (row, config, layout) combination is run independently -- layout
is fully crossed with data and configuration, not tied to row index.

In [45]:
from sklearn.model_selection import train_test_split
N_ROWS = 200

sampled_df, _ = train_test_split(
    df_with_features,
    train_size=N_ROWS,
    stratify=df_with_features["cell_type"],
    random_state=42,
)

sampled_raw = sampled_df[selected_columns].values
sampled_scaled = scaler.transform(sampled_raw)

print("Sampled rows:", sampled_raw.shape)
print(
    "Total experiments to run:",
    len(sampled_raw) * len(CONFIGS) * len(CANDIDATE_LAYOUTS)
)

Sampled rows: (200, 5)
Total experiments to run: 4800


In [48]:
print(type(noisy_simulator))
print(CANDIDATE_LAYOUTS)
print(len(sampled_raw))

<class 'qiskit_aer.backends.aer_simulator.AerSimulator'>
[[0, 1, 2, 3, 4], [19, 20, 21, 22, 23], [38, 39, 40, 41, 42], [57, 58, 59, 60, 61]]
200


In [49]:
all_results = []

total = (
    len(sampled_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

done = 0
start_time = time.time()

print(f"Starting full dataset generation...")
print(f"Rows: {len(sampled_raw)}")
print(f"Configurations: {len(CONFIGS)}")
print(f"Layouts: {len(CANDIDATE_LAYOUTS)}")
print(f"Total experiments: {total}")
print(f"Repeats per experiment: 5")

for row_idx in range(len(sampled_raw)):

    for reps, entanglement in CONFIGS:

        for layout in CANDIDATE_LAYOUTS:

            result = run_complete_experiment(
                raw_data_row=sampled_raw[row_idx],
                data_row=sampled_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            result["data_row_index"] = row_idx
            result["layout_id"] = str(layout)

            all_results.append(result)

            done += 1

    if (row_idx + 1) % 10 == 0:
        elapsed = time.time() - start_time

        print(
            f"Completed {row_idx + 1}/{len(sampled_raw)} rows "
            f"({done}/{total} experiments) | "
            f"Elapsed: {elapsed/60:.1f} min"
        )

predictor_df = pd.DataFrame(all_results)

print("\nFinal dataset shape:", predictor_df.shape)

Starting full dataset generation...
Rows: 200
Configurations: 6
Layouts: 4
Total experiments: 4800
Repeats per experiment: 5
Completed 10/200 rows (240/4800 experiments) | Elapsed: 17.1 min
Completed 20/200 rows (480/4800 experiments) | Elapsed: 35.1 min
Completed 30/200 rows (720/4800 experiments) | Elapsed: 53.5 min
Completed 40/200 rows (960/4800 experiments) | Elapsed: 71.1 min
Completed 50/200 rows (1200/4800 experiments) | Elapsed: 89.1 min
Completed 60/200 rows (1440/4800 experiments) | Elapsed: 107.5 min
Completed 70/200 rows (1680/4800 experiments) | Elapsed: 125.7 min
Completed 80/200 rows (1920/4800 experiments) | Elapsed: 143.5 min
Completed 90/200 rows (2160/4800 experiments) | Elapsed: 161.8 min
Completed 100/200 rows (2400/4800 experiments) | Elapsed: 178.4 min
Completed 110/200 rows (2640/4800 experiments) | Elapsed: 193.3 min
Completed 120/200 rows (2880/4800 experiments) | Elapsed: 201.9 min
Completed 130/200 rows (3120/4800 experiments) | Elapsed: 210.6 min
Completed

In [50]:
predictor_df.to_csv("../experiments/results/predictor_training_data.csv", index=False)
print("Saved:", predictor_df.shape)


Saved: (4800, 61)


In [51]:
print("Shape:", predictor_df.shape)

print("\nMissing values:")
print(
    predictor_df.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

Shape: (4800, 61)

Missing values:
raw_feature_1                       0
raw_feature_2                       0
raw_feature_3                       0
raw_feature_4                       0
raw_feature_5                       0
angle_1                             0
angle_2                             0
angle_3                             0
angle_4                             0
angle_5                             0
abstract_n_qubits                   0
abstract_depth                      0
abstract_total_gate_count           0
abstract_single_qubit_gate_count    0
abstract_two_qubit_gate_count       0
abstract_cx_count                   0
abstract_two_qubit_gate_ratio       0
abstract_average_gates_per_qubit    0
abstract_max_gates_per_qubit        0
abstract_unique_2q_pairs            0
dtype: int64


In [52]:
print("\nDuplicate rows:", predictor_df.duplicated().sum())


Duplicate rows: 0


In [53]:
print("\nTVD statistics:")
print(
    predictor_df[
        ["mean_tvd", "std_tvd", "min_tvd", "max_tvd"]
    ].describe()
)


TVD statistics:
          mean_tvd      std_tvd      min_tvd      max_tvd
count  4800.000000  4800.000000  4800.000000  4800.000000
mean      0.255241     0.011667     0.240840     0.269728
std       0.170379     0.004798     0.168411     0.172390
min       0.055273     0.000764     0.039063     0.060547
25%       0.072070     0.008142     0.062500     0.084961
50%       0.241873     0.011119     0.225840     0.260251
75%       0.388863     0.014697     0.372669     0.405674
max       0.759791     0.034468     0.753736     0.767408


In [54]:
print(predictor_df.columns.tolist())

['raw_feature_1', 'raw_feature_2', 'raw_feature_3', 'raw_feature_4', 'raw_feature_5', 'angle_1', 'angle_2', 'angle_3', 'angle_4', 'angle_5', 'abstract_n_qubits', 'abstract_depth', 'abstract_total_gate_count', 'abstract_single_qubit_gate_count', 'abstract_two_qubit_gate_count', 'abstract_cx_count', 'abstract_two_qubit_gate_ratio', 'abstract_average_gates_per_qubit', 'abstract_max_gates_per_qubit', 'abstract_unique_2q_pairs', 'abstract_average_qubit_degree', 'abstract_max_qubit_degree', 'abstract_repeated_2q_pairs', 'reps', 'entanglement', 'transpiled_depth', 'transpiled_total_gates', 'transpiled_1q_gates', 'transpiled_2q_gates', 'transpiled_cz_count', 'transpiled_cx_count', 'transpiled_sx_count', 'transpiled_rz_count', 'backend_size', 'mean_cz_error', 'std_cz_error', 'min_cz_error', 'max_cz_error', 'sum_cz_error', 'cumulative_cz_error', 'mean_cz_duration', 'max_cz_duration', 'total_cz_duration', 'physical_qubit_count', 'physical_qubits_used', 'mean_t1', 'min_t1', 'max_t1', 'mean_t2', 'm

In [55]:
import pandas as pd
import numpy as np

# Work on a copy
validation_df = predictor_df.copy()

# ------------------------------------------------------------
# 1. Create 3 TVD bands
# ------------------------------------------------------------

validation_df["tvd_band"] = pd.qcut(
    validation_df["mean_tvd"],
    q=3,
    labels=["low", "medium", "high"]
)

# ------------------------------------------------------------
# 2. We want diversity across:
#    - reps
#    - entanglement
#    - hardware layout
# ------------------------------------------------------------

selected_rows = []

used_combinations = set()

for band in ["low", "medium", "high"]:

    candidates = (
        validation_df[
            validation_df["tvd_band"] == band
        ]
        .sort_values("mean_tvd")
    )

    # Try to select 4 diverse circuits from this band
    for _, row in candidates.iterrows():

        combo = (
            row["reps"],
            row["entanglement"],
            row["layout_id"]
        )

        if combo not in used_combinations:
            selected_rows.append(row)
            used_combinations.add(combo)

        if len([
            r for r in selected_rows
            if r["tvd_band"] == band
        ]) >= 4:
            break

# ------------------------------------------------------------
# 3. Create final validation dataframe
# ------------------------------------------------------------

hardware_validation_df = pd.DataFrame(
    selected_rows
).reset_index(drop=True)

print(
    "Validation circuits:",
    len(hardware_validation_df)
)

print(
    hardware_validation_df[
        [
            "data_row_index",
            "reps",
            "entanglement",
            "layout_id",
            "initial_layout",
            "mean_tvd",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "tvd_band"
        ]
    ]
)

Validation circuits: 12
    data_row_index  reps entanglement             layout_id        initial_layout  mean_tvd  transpiled_depth  \
0              184     1         full  [19, 20, 21, 22, 23]  [19, 20, 21, 22, 23]  0.055273               125   
1                8     1       linear  [38, 39, 40, 41, 42]  [38, 39, 40, 41, 42]  0.055664                43   
2              122     1         full       [0, 1, 2, 3, 4]       [0, 1, 2, 3, 4]  0.056250               123   
3              180     1       linear  [19, 20, 21, 22, 23]  [19, 20, 21, 22, 23]  0.057031                45   
4               60     2       linear       [0, 1, 2, 3, 4]       [0, 1, 2, 3, 4]  0.083388                67   
5                6     3       linear       [0, 1, 2, 3, 4]       [0, 1, 2, 3, 4]  0.083523                89   
6               60     3       linear  [38, 39, 40, 41, 42]  [38, 39, 40, 41, 42]  0.093295                89   
7               58     2       linear  [38, 39, 40, 41, 42]  [38, 39, 40

In [56]:
print("\nConfiguration coverage:")
print(
    hardware_validation_df[
        ["reps", "entanglement"]
    ].value_counts()
)

print("\nLayout coverage:")
print(
    hardware_validation_df["layout_id"].value_counts()
)

print("\nTVD bands:")
print(
    hardware_validation_df["tvd_band"].value_counts()
)


Configuration coverage:
reps  entanglement
3     linear          3
1     full            2
      linear          2
2     linear          2
3     full            2
2     full            1
Name: count, dtype: int64

Layout coverage:
layout_id
[19, 20, 21, 22, 23]    4
[38, 39, 40, 41, 42]    4
[0, 1, 2, 3, 4]         4
Name: count, dtype: int64

TVD bands:
tvd_band
low       4
medium    4
high      4
Name: count, dtype: int64


In [57]:
row = hardware_validation_df.iloc[0]

print("Layout:", row["initial_layout"])
print("Physical qubits:", row["physical_qubits_used"])
print("Stored mean CZ error:", row["mean_cz_error"])

Layout: [19, 20, 21, 22, 23]
Physical qubits: [19, 20, 21, 22, 23]
Stored mean CZ error: 0.2681439247659132


In [58]:
# Recreate the same circuit
raw_row = df_with_features.iloc[
    int(row["data_row_index"])
][selected_columns].to_numpy()

scaled_row = scaler.transform(
    raw_row.reshape(1, -1)
)[0]

fm = zz_feature_map(
    feature_dimension=5,
    reps=int(row["reps"]),
    entanglement=row["entanglement"]
)

abstract = fm.assign_parameters(scaled_row)

measured = abstract.copy()
measured.measure_all()

transpiled = transpile(
    measured,
    fake_backend,
    optimization_level=1,
    initial_layout=eval(row["initial_layout"]),
    seed_transpiler=42,
)

print("\nActual physical 2Q operations:")

for inst in transpiled.data:
    if inst.operation.name in ["cz", "cx"]:
        qubits = [
            transpiled.find_bit(q).index
            for q in inst.qubits
        ]
        print(inst.operation.name, qubits)


Actual physical 2Q operations:
cz [19, 20]
cz [19, 20]
cz [20, 21]
cz [20, 21]
cz [20, 21]
cz [19, 20]
cz [19, 20]
cz [21, 20]
cz [21, 20]
cz [20, 19]
cz [20, 19]
cz [20, 19]
cz [21, 20]
cz [21, 20]
cz [21, 20]
cz [21, 22]
cz [21, 22]
cz [22, 21]
cz [22, 21]
cz [22, 21]
cz [20, 21]
cz [20, 21]
cz [21, 20]
cz [21, 20]
cz [21, 20]
cz [19, 20]
cz [19, 20]
cz [22, 23]
cz [22, 23]
cz [22, 23]
cz [22, 23]
cz [22, 23]
cz [21, 22]
cz [21, 22]
cz [21, 22]
cz [21, 22]
cz [21, 22]
cz [20, 21]
cz [20, 21]
cz [20, 21]
cz [19, 20]
cz [19, 20]
cz [21, 20]
cz [21, 20]


In [59]:
pairs = [
    (19, 20),
    (20, 21),
    (21, 22),
    (22, 23),
]

edges = {
    tuple(sorted(edge))
    for edge in fake_backend.coupling_map.get_edges()
}

print("Backend calibration for used CZ pairs:\n")

for q1, q2 in pairs:
    pair = tuple(sorted((q1, q2)))

    print(f"\nPair {pair}")

    if pair not in edges:
        print("  NOT FOUND in coupling map")
        continue

    # Check both directions because backend properties may be directional
    for direction in [(q1, q2), (q2, q1)]:
        try:
            props = fake_backend.target["cz"][
                direction
            ]

            print(
                f"  {direction}: "
                f"error={props.error}, "
                f"duration={props.duration}"
            )

        except Exception as e:
            print(
                f"  {direction}: "
                f"could not retrieve target CZ properties: {e}"
            )

Backend calibration for used CZ pairs:


Pair (19, 20)
  (19, 20): error=1.0, duration=1.08e-07
  (20, 19): error=1.0, duration=1.08e-07

Pair (20, 21)
  (20, 21): error=0.03544601345484144, duration=6.8e-08
  (21, 20): error=0.03544601345484144, duration=6.8e-08

Pair (21, 22)
  (21, 22): error=0.012872397774247424, duration=6.8e-08
  (22, 21): error=0.012872397774247424, duration=6.8e-08

Pair (22, 23)
  (22, 23): error=0.006316093954111535, duration=6.8e-08
  (23, 22): error=0.006316093954111535, duration=6.8e-08


In [60]:
for layout in CANDIDATE_LAYOUTS:
    print("\nLayout:", layout)

    for i in range(len(layout) - 1):
        q1, q2 = layout[i], layout[i+1]

        try:
            props = fake_backend.target["cz"][(q1, q2)]
            print(
                f"  ({q1},{q2}) "
                f"error={props.error:.6f}, "
                f"duration={props.duration:.2e}s"
            )
        except Exception:
            props = fake_backend.target["cz"][(q2, q1)]
            print(
                f"  ({q2},{q1}) "
                f"error={props.error:.6f}, "
                f"duration={props.duration:.2e}s"
            )


Layout: [0, 1, 2, 3, 4]
  (0,1) error=0.004855, duration=6.80e-08s
  (1,2) error=0.004194, duration=6.80e-08s
  (2,3) error=0.003561, duration=6.80e-08s
  (3,4) error=0.004260, duration=6.80e-08s

Layout: [19, 20, 21, 22, 23]
  (19,20) error=1.000000, duration=1.08e-07s
  (20,21) error=0.035446, duration=6.80e-08s
  (21,22) error=0.012872, duration=6.80e-08s
  (22,23) error=0.006316, duration=6.80e-08s

Layout: [38, 39, 40, 41, 42]
  (38,39) error=0.002595, duration=6.80e-08s
  (39,40) error=0.004605, duration=6.80e-08s
  (40,41) error=0.064664, duration=6.80e-08s
  (41,42) error=0.071514, duration=6.80e-08s

Layout: [57, 58, 59, 60, 61]
  (57,58) error=1.000000, duration=6.80e-08s
  (58,59) error=1.000000, duration=6.80e-08s
  (59,60) error=0.003291, duration=6.80e-08s
  (60,61) error=0.002720, duration=6.80e-08s


In [61]:
def layout_edge_errors(layout):
    errors = []

    for i in range(len(layout) - 1):
        q1, q2 = layout[i], layout[i+1]

        try:
            props = fake_backend.target["cz"][(q1, q2)]
        except Exception:
            props = fake_backend.target["cz"][(q2, q1)]

        errors.append(props.error)

    return errors


candidate_quality = []

for layout in valid_layouts:
    errors = layout_edge_errors(layout)

    candidate_quality.append({
        "layout": layout,
        "max_error": max(errors),
        "mean_error": sum(errors) / len(errors),
        "errors": errors
    })

# Remove obviously broken layouts
good_layouts = [
    x for x in candidate_quality
    if x["max_error"] < 0.1
]

# Sort by mean error
good_layouts = sorted(
    good_layouts,
    key=lambda x: x["mean_error"]
)

print("Good candidate layouts:")
for x in good_layouts[:20]:
    print(
        x["layout"],
        "mean=", round(x["mean_error"], 6),
        "max=", round(x["max_error"], 6),
        "edges=", [round(e, 6) for e in x["errors"]]
    )

Good candidate layouts:
[63, 64, 65, 66, 67] mean= 0.002881 max= 0.003303 edges= [0.003089, 0.002328, 0.002804, 0.003303]
[64, 65, 66, 67, 68] mean= 0.002883 max= 0.003303 edges= [0.002328, 0.002804, 0.003303, 0.003097]
[62, 63, 64, 65, 66] mean= 0.003267 max= 0.004847 edges= [0.004847, 0.003089, 0.002328, 0.002804]
[65, 66, 67, 68, 69] mean= 0.003283 max= 0.003929 edges= [0.002804, 0.003303, 0.003097, 0.003929]
[23, 24, 25, 26, 27] mean= 0.00333 max= 0.004405 edges= [0.003167, 0.002954, 0.002793, 0.004405]
[61, 62, 63, 64, 65] mean= 0.003418 max= 0.004847 edges= [0.003406, 0.004847, 0.003089, 0.002328]
[60, 61, 62, 63, 64] mean= 0.003515 max= 0.004847 edges= [0.00272, 0.003406, 0.004847, 0.003089]
[59, 60, 61, 62, 63] mean= 0.003566 max= 0.004847 edges= [0.003291, 0.00272, 0.003406, 0.004847]
[9, 10, 11, 12, 13] mean= 0.003639 max= 0.004138 edges= [0.004072, 0.003466, 0.002879, 0.004138]
[98, 99, 100, 101, 102] mean= 0.003651 max= 0.004187 edges= [0.002537, 0.004051, 0.003828, 0.00418

In [65]:
validation_df = predictor_df.copy()

validation_df["tvd_band"] = pd.qcut(
    validation_df["mean_tvd"],
    q=3,
    labels=["low", "medium", "high"]
)

# Pick 4 rows from each band
low = validation_df[validation_df["tvd_band"] == "low"].sample(
    n=4, random_state=42
)

medium = validation_df[validation_df["tvd_band"] == "medium"].sample(
    n=4, random_state=42
)

high = validation_df[validation_df["tvd_band"] == "high"].sample(
    n=4, random_state=42
)

hardware_validation_df = pd.concat(
    [low, medium, high],
    ignore_index=True
)

print("Validation circuits:", len(hardware_validation_df))

print(
    hardware_validation_df[
        [
            "data_row_index",
            "reps",
            "entanglement",
            "initial_layout",
            "mean_tvd",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "tvd_band"
        ]
    ]
)

Validation circuits: 12
    data_row_index  reps entanglement        initial_layout  mean_tvd  transpiled_depth  transpiled_2q_gates  \
0               65     1         full  [57, 58, 59, 60, 61]  0.065820               125                   44   
1               44     1       linear  [57, 58, 59, 60, 61]  0.070313                45                    8   
2               21     1       linear  [19, 20, 21, 22, 23]  0.069336                45                    8   
3               17     1       linear       [0, 1, 2, 3, 4]  0.072852                43                    8   
4               67     2         full       [0, 1, 2, 3, 4]  0.195166               240                   91   
5               46     3       linear       [0, 1, 2, 3, 4]  0.164342                87                   24   
6               21     3       linear  [57, 58, 59, 60, 61]  0.331858                89                   24   
7               17     3       linear  [38, 39, 40, 41, 42]  0.298367           

In [66]:
hardware_validation_df.to_csv(
    "../experiments/results/hardware_validation_12.csv",
    index=False
)

print("Saved:", hardware_validation_df.shape)

Saved: (12, 62)


In [68]:
import ast

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    raw_row = sampled_raw[data_idx]
    scaled_row = sampled_scaled[data_idx]

    reps = int(row["reps"])
    entanglement = row["entanglement"]
    layout = ast.literal_eval(row["initial_layout"])

    fm = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement
    )

    abstract = fm.assign_parameters(scaled_row)

    measured = abstract.copy()
    measured.measure_all()

    transpiled = transpile(
        measured,
        fake_backend,
        optimization_level=1,
        initial_layout=layout,
        seed_transpiler=42
    )

    recreated_2q = sum(
        1
        for inst in transpiled.data
        if len(inst.qubits) == 2
    )

    print(
        f"{i}: "
        f"stored depth={row['transpiled_depth']}, "
        f"recreated depth={transpiled.depth()} | "
        f"stored 2Q={row['transpiled_2q_gates']}, "
        f"recreated 2Q={recreated_2q}"
    )

0: stored depth=125, recreated depth=126 | stored 2Q=44, recreated 2Q=44
1: stored depth=45, recreated depth=46 | stored 2Q=8, recreated 2Q=8
2: stored depth=45, recreated depth=46 | stored 2Q=8, recreated 2Q=8
3: stored depth=43, recreated depth=44 | stored 2Q=8, recreated 2Q=8
4: stored depth=240, recreated depth=241 | stored 2Q=91, recreated 2Q=91
5: stored depth=87, recreated depth=88 | stored 2Q=24, recreated 2Q=24
6: stored depth=89, recreated depth=90 | stored 2Q=24, recreated 2Q=24
7: stored depth=87, recreated depth=88 | stored 2Q=24, recreated 2Q=24
8: stored depth=67, recreated depth=68 | stored 2Q=16, recreated 2Q=16
9: stored depth=233, recreated depth=234 | stored 2Q=91, recreated 2Q=91
10: stored depth=233, recreated depth=234 | stored 2Q=91, recreated 2Q=91
11: stored depth=353, recreated depth=354 | stored 2Q=138, recreated 2Q=138


In [ ]:
"""
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="",
    instance="",
    set_as_default=True,
    overwrite=True,
)

print("Credentials saved.")
"""

Credentials saved.


In [27]:
!pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()

print("Token loaded:", "IBM_QUANTUM_TOKEN" in os.environ)
print("Instance loaded:", "IBM_QUANTUM_INSTANCE" in os.environ)

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    token=os.environ["IBM_QUANTUM_TOKEN"],
    instance=os.environ["IBM_QUANTUM_INSTANCE"],
)

backend = service.backend("ibm_kingston")
print(backend.name)

qiskit_runtime_service._discover_account:WARNING:2026-09-03 17:22:19,354: Loading account with the given token. A saved account will not be used.


Token loaded: True
Instance loaded: True
ibm_kingston


In [71]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

print("IBM Quantum connection OK")
print(service.backends())

IBM Quantum connection OK
[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


In [72]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

backend = service.backend("ibm_kingston")

print("Backend:", backend.name)
print("Qubits:", backend.num_qubits)
print("Processor:", backend.processor_type)
print("Basis gates:", backend.operation_names)

Backend: ibm_kingston
Qubits: 156
Processor: {'family': 'Heron', 'revision': '2'}
Basis gates: ['sx', 'measure', 'rz', 'reset', 'if_else', 'delay', 'measure_2', 'x', 'cz', 'id']


In [73]:
print("Number of qubits:", backend.num_qubits)

edges = backend.coupling_map.get_edges()

print("Number of coupling edges:", len(edges))
print("First 30 coupling edges:")
for edge in edges[:30]:
    print(edge)

Number of qubits: 156
Number of coupling edges: 352
First 30 coupling edges:
(0, 1)
(1, 0)
(1, 2)
(2, 1)
(2, 3)
(3, 2)
(3, 4)
(3, 16)
(4, 3)
(4, 5)
(5, 4)
(5, 6)
(6, 5)
(6, 7)
(7, 6)
(7, 8)
(7, 17)
(8, 7)
(8, 9)
(9, 8)
(9, 10)
(10, 9)
(10, 11)
(11, 10)
(11, 12)
(11, 18)
(12, 11)
(12, 13)
(13, 12)
(13, 14)


In [74]:
# Look for connected 5-qubit chains
coupling_edges = {
    tuple(sorted(edge))
    for edge in backend.coupling_map.get_edges()
}

kingston_chains = []

for start in range(backend.num_qubits - 4):
    layout = list(range(start, start + 5))

    connected = all(
        tuple(sorted((layout[i], layout[i + 1]))) in coupling_edges
        for i in range(4)
    )

    if connected:
        kingston_chains.append(layout)

print("Connected 5-qubit chains found:", len(kingston_chains))

for layout in kingston_chains[:20]:
    print(layout)

Connected 5-qubit chains found: 96
[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[3, 4, 5, 6, 7]
[4, 5, 6, 7, 8]
[5, 6, 7, 8, 9]
[6, 7, 8, 9, 10]
[7, 8, 9, 10, 11]
[8, 9, 10, 11, 12]
[9, 10, 11, 12, 13]
[10, 11, 12, 13, 14]
[11, 12, 13, 14, 15]
[20, 21, 22, 23, 24]
[21, 22, 23, 24, 25]
[22, 23, 24, 25, 26]
[23, 24, 25, 26, 27]
[24, 25, 26, 27, 28]
[25, 26, 27, 28, 29]
[26, 27, 28, 29, 30]
[27, 28, 29, 30, 31]


In [75]:
from qiskit_ibm_runtime.fake_provider import FakeKingston

fake_kingston = FakeKingston()

print("Fake backend:", fake_kingston.name)
print("Qubits:", fake_kingston.num_qubits)
print("Basis gates:", fake_kingston.operation_names)

Fake backend: fake_kingston
Qubits: 156
Basis gates: ['sx', 'measure', 'rz', 'reset', 'if_else', 'delay', 'measure_2', 'x', 'cz', 'id']


In [76]:
KINGSTON_VALIDATION_LAYOUT = [0, 1, 2, 3, 4]

print("Kingston validation layout:", KINGSTON_VALIDATION_LAYOUT)

Kingston validation layout: [0, 1, 2, 3, 4]


In [77]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

kingston_noise_model = NoiseModel.from_backend(fake_kingston)

fake_kingston_sim = AerSimulator(
    noise_model=kingston_noise_model
)

print("FakeKingston noisy simulator ready.")

FakeKingston noisy simulator ready.


In [78]:
import ast
import numpy as np
import pandas as pd
from qiskit import transpile
from qiskit.quantum_info import Statevector

fake_kingston_results = []

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    raw_row = sampled_raw[data_idx]
    scaled_row = sampled_scaled[data_idx]

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    # --------------------------------------------------------
    # Recreate SAME logical circuit
    # --------------------------------------------------------

    fm = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = fm.assign_parameters(
        scaled_row
    )

    # Exact ideal distribution
    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    # Add measurements
    measured = abstract_circuit.copy()
    measured.measure_all()

    # --------------------------------------------------------
    # Transpile for FakeKingston
    # --------------------------------------------------------

    kingston_circuit = transpile(
        measured,
        fake_kingston,
        optimization_level=1,
        initial_layout=KINGSTON_VALIDATION_LAYOUT,
        seed_transpiler=42,
    )

    # --------------------------------------------------------
    # Run 5 noisy repetitions in one job
    # --------------------------------------------------------

    circuits = [
        kingston_circuit
        for _ in range(5)
    ]

    result = fake_kingston_sim.run(
        circuits,
        shots=1024,
    ).result()

    tvds = []

    for j in range(5):

        counts = result.get_counts(j)

        noisy_probs = counts_to_probabilities(
            counts
        )

        tvd = calculate_tvd_from_probs(
            ideal_probs,
            noisy_probs
        )

        tvds.append(tvd)

    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    fake_kingston_results.append({
        "validation_index": i,
        "data_row_index": data_idx,
        "reps": reps,
        "entanglement": entanglement,

        # Original Torino result
        "torino_mean_tvd": row["mean_tvd"],

        # Kingston simulation
        "kingston_mean_tvd": float(np.mean(tvds)),
        "kingston_std_tvd": float(
            np.std(tvds, ddof=1)
        ),

        "kingston_depth": kingston_circuit.depth(),

        "kingston_2q_gates": sum(
            1
            for inst in kingston_circuit.data
            if len(inst.qubits) == 2
        ),

        "kingston_layout": str(
            KINGSTON_VALIDATION_LAYOUT
        ),
    })

fake_kingston_validation_df = pd.DataFrame(
    fake_kingston_results
)

print(
    fake_kingston_validation_df
)

    validation_index  data_row_index  reps entanglement  torino_mean_tvd  kingston_mean_tvd  kingston_std_tvd  \
0                  0              65     1         full         0.065820           0.076953          0.005584   
1                  1              44     1       linear         0.070313           0.066797          0.004863   
2                  2              21     1       linear         0.069336           0.063867          0.013612   
3                  3              17     1       linear         0.072852           0.067578          0.007640   
4                  4              67     2         full         0.195166           0.127191          0.005798   
5                  5              46     3       linear         0.164342           0.099062          0.010104   
6                  6              21     3       linear         0.331858           0.110679          0.010438   
7                  7              17     3       linear         0.298367           0.119019     

In [79]:
print(
    fake_kingston_validation_df[
        [
            "validation_index",
            "reps",
            "entanglement",
            "torino_mean_tvd",
            "kingston_mean_tvd",
        ]
    ]
)

    validation_index  reps entanglement  torino_mean_tvd  kingston_mean_tvd
0                  0     1         full         0.065820           0.076953
1                  1     1       linear         0.070313           0.066797
2                  2     1       linear         0.069336           0.063867
3                  3     1       linear         0.072852           0.067578
4                  4     2         full         0.195166           0.127191
5                  5     3       linear         0.164342           0.099062
6                  6     3       linear         0.331858           0.110679
7                  7     3       linear         0.298367           0.119019
8                  8     2       linear         0.530607           0.087640
9                  9     2         full         0.361257           0.164500
10                10     2         full         0.371899           0.151002
11                11     3         full         0.362996           0.136236


In [80]:
corr = fake_kingston_validation_df[
    ["torino_mean_tvd", "kingston_mean_tvd"]
].corr().iloc[0, 1]

mae = np.mean(
    np.abs(
        fake_kingston_validation_df["torino_mean_tvd"]
        -
        fake_kingston_validation_df["kingston_mean_tvd"]
    )
)

print(f"Correlation: {corr:.4f}")
print(f"MAE: {mae:.4f}")

Correlation: 0.6356
MAE: 0.1372


In [81]:
fake_kingston_validation_df.to_csv(
    "../experiments/results/fake_kingston_validation_12.csv",
    index=False
)

print("Saved:", fake_kingston_validation_df.shape)

Saved: (12, 10)


In [82]:
KINGSTON_LAYOUT = [0, 1, 2, 3, 4]
real_kingston_circuits = []

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    scaled_row = sampled_scaled[data_idx]

    fm = zz_feature_map(
        feature_dimension=5,
        reps=int(row["reps"]),
        entanglement=row["entanglement"],
    )

    abstract = fm.assign_parameters(scaled_row)

    measured = abstract.copy()
    measured.measure_all()

    isa_circuit = transpile(
        measured,
        backend,
        optimization_level=1,
        initial_layout=[0, 1, 2, 3, 4],
        seed_transpiler=42,
    )

    real_kingston_circuits.append(isa_circuit)

    print(
        f"{i}: "
        f"reps={row['reps']} | "
        f"entanglement={row['entanglement']} | "
        f"depth={isa_circuit.depth()} | "
        f"2Q={sum(len(inst.qubits) == 2 for inst in isa_circuit.data)}"
    )

0: reps=1 | entanglement=full | depth=126 | 2Q=44
1: reps=1 | entanglement=linear | depth=46 | 2Q=8
2: reps=1 | entanglement=linear | depth=46 | 2Q=8
3: reps=1 | entanglement=linear | depth=44 | 2Q=8
4: reps=2 | entanglement=full | depth=241 | 2Q=91
5: reps=3 | entanglement=linear | depth=88 | 2Q=24
6: reps=3 | entanglement=linear | depth=90 | 2Q=24
7: reps=3 | entanglement=linear | depth=88 | 2Q=24
8: reps=2 | entanglement=linear | depth=68 | 2Q=16
9: reps=2 | entanglement=full | depth=234 | 2Q=91
10: reps=2 | entanglement=full | depth=234 | 2Q=91
11: reps=3 | entanglement=full | depth=354 | 2Q=138


In [84]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

job = sampler.run(
    real_kingston_circuits,
    shots=1024
)

print("Job ID:", job.job_id())
print("Submitted to:", backend.name)

Job ID: da0btmv2sl0c73bmtntg
Submitted to: ibm_kingston


In [85]:
result = job.result()

print("Job completed.")
print("Number of circuit results:", len(result))

Job completed.
Number of circuit results: 12


In [86]:
print(result[0])

SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}})


In [87]:
real_kingston_counts = []

for i, pub_result in enumerate(result):

    counts = pub_result.data.meas.get_counts()

    real_kingston_counts.append(counts)

    print(f"\nCircuit {i}")
    print(counts)


Circuit 0
{'10010': 24, '01010': 36, '00001': 32, '00100': 40, '11111': 16, '11011': 32, '10001': 35, '11000': 39, '01011': 34, '00111': 30, '11001': 38, '10110': 28, '01001': 36, '10000': 49, '01101': 27, '10100': 37, '11010': 40, '10101': 24, '00010': 38, '11110': 22, '00011': 40, '00110': 36, '00000': 28, '01100': 33, '11100': 41, '01000': 41, '11101': 29, '01111': 22, '01110': 17, '00101': 33, '10011': 26, '10111': 21}

Circuit 1
{'01000': 26, '10011': 30, '00100': 32, '01010': 27, '01111': 36, '01001': 34, '11100': 34, '01011': 34, '10010': 43, '01101': 25, '01100': 21, '10100': 38, '10000': 39, '00010': 26, '00011': 27, '00000': 46, '00111': 36, '00101': 32, '11111': 38, '00001': 31, '11010': 34, '11000': 52, '11001': 33, '11110': 27, '10001': 31, '10111': 27, '11011': 31, '01110': 23, '11101': 37, '00110': 28, '10110': 22, '10101': 24}

Circuit 2
{'00101': 38, '00000': 37, '00010': 37, '10001': 36, '11000': 21, '10000': 31, '10110': 36, '11001': 28, '10111': 35, '00001': 37, '1

In [88]:
for i, counts in enumerate(real_kingston_counts):
    print(
        f"Circuit {i}: "
        f"{sum(counts.values())} shots, "
        f"{len(counts)} unique outcomes"
    )

Circuit 0: 1024 shots, 32 unique outcomes
Circuit 1: 1024 shots, 32 unique outcomes
Circuit 2: 1024 shots, 32 unique outcomes
Circuit 3: 1024 shots, 32 unique outcomes
Circuit 4: 1024 shots, 32 unique outcomes
Circuit 5: 1024 shots, 32 unique outcomes
Circuit 6: 1024 shots, 31 unique outcomes
Circuit 7: 1024 shots, 30 unique outcomes
Circuit 8: 1024 shots, 32 unique outcomes
Circuit 9: 1024 shots, 32 unique outcomes
Circuit 10: 1024 shots, 32 unique outcomes
Circuit 11: 1024 shots, 32 unique outcomes


In [89]:
from qiskit.quantum_info import Statevector

real_validation_results = []

for i, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    scaled_row = sampled_scaled[data_idx]

    fm = zz_feature_map(
        feature_dimension=5,
        reps=int(row["reps"]),
        entanglement=row["entanglement"],
    )

    abstract_circuit = fm.assign_parameters(
        scaled_row
    )

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    hardware_counts = real_kingston_counts[i]

    hardware_probs = counts_to_probabilities(
        hardware_counts
    )

    hardware_tvd = calculate_tvd_from_probs(
        ideal_probs,
        hardware_probs
    )

    real_validation_results.append({
        "validation_index": i,
        "data_row_index": data_idx,
        "reps": int(row["reps"]),
        "entanglement": row["entanglement"],
        "fake_torino_tvd": float(row["mean_tvd"]),
        "fake_kingston_tvd": float(
            fake_kingston_validation_df.iloc[i]["kingston_mean_tvd"]
        ),
        "real_kingston_tvd": float(hardware_tvd),
    })

real_validation_df = pd.DataFrame(
    real_validation_results
)

print(real_validation_df)

    validation_index  data_row_index  reps entanglement  fake_torino_tvd  fake_kingston_tvd  real_kingston_tvd
0                  0              65     1         full         0.065820           0.076953           0.099609
1                  1              44     1       linear         0.070313           0.066797           0.083984
2                  2              21     1       linear         0.069336           0.063867           0.074219
3                  3              17     1       linear         0.072852           0.067578           0.083008
4                  4              67     2         full         0.195166           0.127191           0.167147
5                  5              46     3       linear         0.164342           0.099062           0.110502
6                  6              21     3       linear         0.331858           0.110679           0.089919
7                  7              17     3       linear         0.298367           0.119019           0.101927
8

In [90]:
print(
    "FakeKingston vs Real Kingston:"
)

print(
    real_validation_df[
        ["fake_kingston_tvd", "real_kingston_tvd"]
    ].corr().iloc[0, 1]
)

print(
    "\nFakeTorino vs Real Kingston:"
)

print(
    real_validation_df[
        ["fake_torino_tvd", "real_kingston_tvd"]
    ].corr().iloc[0, 1]
)

FakeKingston vs Real Kingston:
0.8259812524496929

FakeTorino vs Real Kingston:
0.4146539242075321


In [91]:
fake_kingston_mae = np.mean(
    np.abs(
        real_validation_df["fake_kingston_tvd"]
        -
        real_validation_df["real_kingston_tvd"]
    )
)

fake_torino_mae = np.mean(
    np.abs(
        real_validation_df["fake_torino_tvd"]
        -
        real_validation_df["real_kingston_tvd"]
    )
)

print(
    f"FakeKingston → real Kingston MAE: "
    f"{fake_kingston_mae:.4f}"
)

print(
    f"FakeTorino → real Kingston MAE: "
    f"{fake_torino_mae:.4f}"
)

FakeKingston → real Kingston MAE: 0.0229
FakeTorino → real Kingston MAE: 0.1294


# Recreating Dataset using fake_kingston instead of fake_torino #


In [1]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeKingston
from qiskit.quantum_info import Statevector

fake_backend = FakeKingston()

print("Backend:", fake_backend.name)
print("Number of qubits:", fake_backend.num_qubits)

ideal_simulator = AerSimulator()

noise_model = NoiseModel.from_backend(fake_backend)

noisy_simulator = AerSimulator(
    noise_model=noise_model
)

Backend: fake_kingston
Number of qubits: 156


In [3]:
import numpy as np

In [4]:
def layout_edge_errors(layout, backend):
    errors = []

    for i in range(len(layout) - 1):
        q1 = layout[i]
        q2 = layout[i + 1]

        try:
            props = backend.target["cz"][(q1, q2)]
        except Exception:
            props = backend.target["cz"][(q2, q1)]

        if props is None or props.error is None:
            return None

        errors.append(props.error)

    return errors


coupling_edges = {
    tuple(sorted(edge))
    for edge in fake_backend.coupling_map.get_edges()
}


valid_kingston_layouts = []

for start in range(fake_backend.num_qubits - 4):

    layout = list(range(start, start + 5))

    connected = all(
        tuple(sorted((layout[i], layout[i + 1])))
        in coupling_edges
        for i in range(4)
    )

    if not connected:
        continue

    errors = layout_edge_errors(
        layout,
        fake_backend
    )

    if errors is None:
        continue

    valid_kingston_layouts.append({
        "layout": layout,
        "mean_error": np.mean(errors),
        "max_error": np.max(errors),
        "errors": errors
    })


valid_kingston_layouts = sorted(
    valid_kingston_layouts,
    key=lambda x: x["mean_error"]
)

print(
    "Valid Kingston layouts:",
    len(valid_kingston_layouts)
)

for item in valid_kingston_layouts[:20]:
    print(
        item["layout"],
        "mean=",
        round(item["mean_error"], 6),
        "max=",
        round(item["max_error"], 6),
        "edges=",
        [round(e, 6) for e in item["errors"]]
    )

Valid Kingston layouts: 96
[10, 11, 12, 13, 14] mean= 0.001253 max= 0.00158 edges= [0.001341, 0.000873, 0.00158, 0.001218]
[0, 1, 2, 3, 4] mean= 0.001259 max= 0.001524 edges= [0.00115, 0.001207, 0.001156, 0.001524]
[122, 123, 124, 125, 126] mean= 0.001283 max= 0.00151 edges= [0.001139, 0.001342, 0.00151, 0.00114]
[91, 92, 93, 94, 95] mean= 0.001351 max= 0.0019 edges= [0.0019, 0.001378, 0.000888, 0.001237]
[140, 141, 142, 143, 144] mean= 0.001358 max= 0.001523 edges= [0.001523, 0.001246, 0.001252, 0.001411]
[123, 124, 125, 126, 127] mean= 0.001359 max= 0.00151 edges= [0.001342, 0.00151, 0.00114, 0.001443]
[50, 51, 52, 53, 54] mean= 0.001393 max= 0.00181 edges= [0.001291, 0.00181, 0.001472, 0.001]
[71, 72, 73, 74, 75] mean= 0.0014 max= 0.001639 edges= [0.001599, 0.001639, 0.001202, 0.001159]
[20, 21, 22, 23, 24] mean= 0.001418 max= 0.001729 edges= [0.001638, 0.000899, 0.001406, 0.001729]
[80, 81, 82, 83, 84] mean= 0.00142 max= 0.002212 edges= [0.001296, 0.0011, 0.001073, 0.002212]
[11, 1

In [5]:
# Keep only non-catastrophic layouts
usable = [
    x for x in valid_kingston_layouts
    if x["max_error"] < 0.10
]

print("Usable layouts:", len(usable))

# Pick approximately spread-out calibration quality
indices = np.linspace(
    0,
    len(usable) - 1,
    4,
    dtype=int
)

CANDIDATE_LAYOUTS = [
    usable[i]["layout"]
    for i in indices
]

print("\nSelected Kingston layouts:")
for layout in CANDIDATE_LAYOUTS:
    info = next(
        x for x in usable
        if x["layout"] == layout
    )

    print(
        layout,
        "mean=",
        round(info["mean_error"], 6),
        "max=",
        round(info["max_error"], 6),
        "edges=",
        [round(e, 6) for e in info["errors"]]
    )

Usable layouts: 82

Selected Kingston layouts:
[10, 11, 12, 13, 14] mean= 0.001253 max= 0.00158 edges= [0.001341, 0.000873, 0.00158, 0.001218]
[61, 62, 63, 64, 65] mean= 0.001791 max= 0.002067 edges= [0.001832, 0.002067, 0.001405, 0.00186]
[147, 148, 149, 150, 151] mean= 0.002294 max= 0.002585 edges= [0.002434, 0.002585, 0.0022, 0.001958]
[83, 84, 85, 86, 87] mean= 0.012769 max= 0.035945 edges= [0.002212, 0.035945, 0.008216, 0.004704]


In [7]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit import transpile
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

from qiskit_ibm_runtime.fake_provider import FakeKingston

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Imports successful.")

Imports successful.


In [8]:
DATA_PATH = "../data/raw/blood_cell_anomaly_detection.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (5880, 36)
       cell_id                  cell_type  anomaly_label disease_category  cell_diameter_um  nucleus_area_pct  chromatin_density  cytoplasm_ratio  \
0  CELL_005371  Hypersegmented_Neutrophil              1        Infection             15.18              58.8              0.542            0.301   
1  CELL_005300  Hypersegmented_Neutrophil              1        Infection             16.47              73.6              0.583            0.365   
2  CELL_000200                 Neutrophil              0       Normal_WBC             13.41              55.5              0.448            0.376   
3  CELL_003269                 Normal_RBC              0       Normal_RBC              7.36               0.0              0.000            1.000   
4  CELL_003505                 Normal_RBC              0       Normal_RBC              7.53               0.0              0.000            1.000   

   circularity  eccentricity  granularity_score  lobularity_score  membrane_smo

In [9]:
selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

feature_data = df[selected_columns].copy()

print("Selected features:")
print(selected_columns)

print("\nMissing values:")
print(feature_data.isna().sum())

print("\nShape:")
print(feature_data.shape)

Selected features:
['cell_diameter_um', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity']

Missing values:
cell_diameter_um     0
chromatin_density    0
cytoplasm_ratio      0
circularity          0
eccentricity         0
dtype: int64

Shape:
(5880, 5)


In [10]:
scaler = MinMaxScaler(
    feature_range=(0, 2 * np.pi)
)

scaled_features = scaler.fit_transform(
    feature_data.values
)

print("Scaled shape:", scaled_features.shape)
print("Example scaled row:", scaled_features[0])

Scaled shape: (5880, 5)
Example scaled row: [4.415043   3.40548644 1.6600837  3.23234977 3.35737882]


In [11]:
noise_model = NoiseModel.from_backend(
    fake_backend
)

noisy_simulator = AerSimulator(
    noise_model=noise_model
)

print("FakeKingston noisy simulator ready.")

FakeKingston noisy simulator ready.


In [12]:
coupling_edges = {
    tuple(sorted(edge))
    for edge in fake_backend.coupling_map.get_edges()
}

valid_layouts = []

for start in range(fake_backend.num_qubits - 4):

    layout = list(range(start, start + 5))

    connected = all(
        tuple(sorted((layout[i], layout[i + 1])))
        in coupling_edges
        for i in range(4)
    )

    if connected:
        valid_layouts.append(layout)

print("Connected 5-qubit chains:", len(valid_layouts))

for layout in valid_layouts[:20]:
    print(layout)

Connected 5-qubit chains: 96
[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
[3, 4, 5, 6, 7]
[4, 5, 6, 7, 8]
[5, 6, 7, 8, 9]
[6, 7, 8, 9, 10]
[7, 8, 9, 10, 11]
[8, 9, 10, 11, 12]
[9, 10, 11, 12, 13]
[10, 11, 12, 13, 14]
[11, 12, 13, 14, 15]
[20, 21, 22, 23, 24]
[21, 22, 23, 24, 25]
[22, 23, 24, 25, 26]
[23, 24, 25, 26, 27]
[24, 25, 26, 27, 28]
[25, 26, 27, 28, 29]
[26, 27, 28, 29, 30]
[27, 28, 29, 30, 31]


In [13]:
def get_layout_edge_errors(layout, backend):
    errors = []

    for i in range(len(layout) - 1):

        q1 = layout[i]
        q2 = layout[i + 1]

        props = None

        try:
            props = backend.target["cz"][(q1, q2)]
        except Exception:
            pass

        if props is None:
            try:
                props = backend.target["cz"][(q2, q1)]
            except Exception:
                pass

        if props is None or props.error is None:
            return None

        errors.append(float(props.error))

    return errors


layout_quality = []

for layout in valid_layouts:

    errors = get_layout_edge_errors(
        layout,
        fake_backend
    )

    if errors is None:
        continue

    layout_quality.append({
        "layout": layout,
        "mean_error": np.mean(errors),
        "max_error": np.max(errors),
        "errors": errors,
    })

layout_quality = sorted(
    layout_quality,
    key=lambda x: x["mean_error"]
)

print("Best Kingston layouts:\n")

for item in layout_quality[:20]:

    print(
        item["layout"],
        "| mean=",
        round(item["mean_error"], 6),
        "| max=",
        round(item["max_error"], 6),
        "| edges=",
        [round(x, 6) for x in item["errors"]]
    )

Best Kingston layouts:

[10, 11, 12, 13, 14] | mean= 0.001253 | max= 0.00158 | edges= [0.001341, 0.000873, 0.00158, 0.001218]
[0, 1, 2, 3, 4] | mean= 0.001259 | max= 0.001524 | edges= [0.00115, 0.001207, 0.001156, 0.001524]
[122, 123, 124, 125, 126] | mean= 0.001283 | max= 0.00151 | edges= [0.001139, 0.001342, 0.00151, 0.00114]
[91, 92, 93, 94, 95] | mean= 0.001351 | max= 0.0019 | edges= [0.0019, 0.001378, 0.000888, 0.001237]
[140, 141, 142, 143, 144] | mean= 0.001358 | max= 0.001523 | edges= [0.001523, 0.001246, 0.001252, 0.001411]
[123, 124, 125, 126, 127] | mean= 0.001359 | max= 0.00151 | edges= [0.001342, 0.00151, 0.00114, 0.001443]
[50, 51, 52, 53, 54] | mean= 0.001393 | max= 0.00181 | edges= [0.001291, 0.00181, 0.001472, 0.001]
[71, 72, 73, 74, 75] | mean= 0.0014 | max= 0.001639 | edges= [0.001599, 0.001639, 0.001202, 0.001159]
[20, 21, 22, 23, 24] | mean= 0.001418 | max= 0.001729 | edges= [0.001638, 0.000899, 0.001406, 0.001729]
[80, 81, 82, 83, 84] | mean= 0.00142 | max= 0.0022

In [14]:
usable_layouts = [
    x
    for x in layout_quality
    if x["max_error"] < 0.10
]

print("Usable layouts:", len(usable_layouts))

if len(usable_layouts) < 4:
    raise RuntimeError(
        "Fewer than 4 usable Kingston layouts found."
    )

indices = np.linspace(
    0,
    len(usable_layouts) - 1,
    4,
    dtype=int
)

CANDIDATE_LAYOUTS = [
    usable_layouts[i]["layout"]
    for i in indices
]

print("\nSelected Kingston layouts:\n")

for layout in CANDIDATE_LAYOUTS:

    item = next(
        x for x in usable_layouts
        if x["layout"] == layout
    )

    print(
        layout,
        "| mean=",
        round(item["mean_error"], 6),
        "| max=",
        round(item["max_error"], 6),
        "| edges=",
        [round(x, 6) for x in item["errors"]]
    )

Usable layouts: 82

Selected Kingston layouts:

[10, 11, 12, 13, 14] | mean= 0.001253 | max= 0.00158 | edges= [0.001341, 0.000873, 0.00158, 0.001218]
[61, 62, 63, 64, 65] | mean= 0.001791 | max= 0.002067 | edges= [0.001832, 0.002067, 0.001405, 0.00186]
[147, 148, 149, 150, 151] | mean= 0.002294 | max= 0.002585 | edges= [0.002434, 0.002585, 0.0022, 0.001958]
[83, 84, 85, 86, 87] | mean= 0.012769 | max= 0.035945 | edges= [0.002212, 0.035945, 0.008216, 0.004704]


In [15]:
def counts_to_probabilities(counts):
    total = sum(counts.values())

    return {
        key: value / total
        for key, value in counts.items()
    }


def calculate_tvd_from_probs(
    ideal_probs,
    noisy_probs
):
    keys = set(ideal_probs) | set(noisy_probs)

    return 0.5 * sum(
        abs(
            ideal_probs.get(key, 0.0)
            - noisy_probs.get(key, 0.0)
        )
        for key in keys
    )

In [69]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    backend,
    noisy_simulator,
    shots=1024,
    repeats=5,
    initial_layout=None,
    seed_transpiler=42,
):
    """
    Run one complete Kingston experiment.

    Returns one row containing:
    - raw input features
    - encoded angles
    - abstract circuit features
    - transpiled circuit features
    - Kingston calibration features
    - TVD statistics
    """

    # =========================================================
    # 1. Build logical circuit
    # =========================================================

    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(
        data_row
    )

    # =========================================================
    # 2. Exact ideal distribution
    # =========================================================

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    # =========================================================
    # 3. Add measurements
    # =========================================================

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # =========================================================
    # 4. Transpile for FakeKingston
    # =========================================================

    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # =========================================================
    # 5. Abstract circuit features
    # =========================================================

    abstract_total_gates = len(
        abstract_circuit.data
    )

    abstract_1q = 0
    abstract_2q = 0
    abstract_cx = 0

    qubit_gate_counts = {
        i: 0
        for i in range(abstract_circuit.num_qubits)
    }

    two_qubit_pairs = []

    for inst in abstract_circuit.data:

        nq = len(inst.qubits)

        if nq == 1:
            abstract_1q += 1

        elif nq == 2:
            abstract_2q += 1

            q1 = abstract_circuit.find_bit(
                inst.qubits[0]
            ).index

            q2 = abstract_circuit.find_bit(
                inst.qubits[1]
            ).index

            two_qubit_pairs.append(
                tuple(sorted((q1, q2)))
            )

            if inst.operation.name == "cx":
                abstract_cx += 1

        for qubit in inst.qubits:
            q_idx = abstract_circuit.find_bit(
                qubit
            ).index

            qubit_gate_counts[q_idx] += 1

    unique_2q_pairs = set(two_qubit_pairs)

    pair_counts = pd.Series(
        two_qubit_pairs
    ).value_counts()

    repeated_pairs = int(
        (pair_counts > 1).sum()
    )

    degrees = {
        q: 0
        for q in range(
            abstract_circuit.num_qubits
        )
    }

    for q1, q2 in unique_2q_pairs:
        degrees[q1] += 1
        degrees[q2] += 1

    # =========================================================
    # 6. Transpiled gate features
    # =========================================================

    transpiled_total = len(
        transpiled_circuit.data
    )

    transpiled_1q = 0
    transpiled_2q = 0
    transpiled_cz = 0
    transpiled_cx = 0
    transpiled_sx = 0
    transpiled_rz = 0

    for inst in transpiled_circuit.data:

        nq = len(inst.qubits)

        if nq == 1:
            transpiled_1q += 1

        elif nq == 2:
            transpiled_2q += 1

        name = inst.operation.name

        if name == "cz":
            transpiled_cz += 1

        elif name == "cx":
            transpiled_cx += 1

        elif name == "sx":
            transpiled_sx += 1

        elif name == "rz":
            transpiled_rz += 1

    # =========================================================
    # 7. Determine physical qubits used
    # =========================================================

    physical_qubits_used = sorted({
        transpiled_circuit.find_bit(
            qubit
        ).index
        for inst in transpiled_circuit.data
        for qubit in inst.qubits
    })

    # =========================================================
    # 8. Extract calibration information
    #    for physical CZ operations actually present
    # =========================================================

    cz_errors = []
    cz_durations = []

    used_pairs = []

    for inst in transpiled_circuit.data:

        if inst.operation.name != "cz":
            continue

        if len(inst.qubits) != 2:
            continue

        q1 = transpiled_circuit.find_bit(
            inst.qubits[0]
        ).index

        q2 = transpiled_circuit.find_bit(
            inst.qubits[1]
        ).index

        used_pairs.append(
            (q1, q2)
        )

        props = None

        try:
            props = backend.target["cz"][
                (q1, q2)
            ]
        except Exception:
            pass

        if props is None:
            try:
                props = backend.target["cz"][
                    (q2, q1)
                ]
            except Exception:
                pass

        if props is None:
            continue

        if props.error is not None:
            cz_errors.append(
                float(props.error)
            )

        if props.duration is not None:
            cz_durations.append(
                float(props.duration)
            )

    # =========================================================
    # 9. T1 / T2 for physical qubits used
    # =========================================================

    t1_values = []
    t2_values = []

    for q in physical_qubits_used:

        try:
            q_props = backend.qubit_properties(q)
        except Exception:
            q_props = None

        if q_props is None:
            continue

        if q_props.t1 is not None:
            t1_values.append(
                float(q_props.t1)
            )

        if q_props.t2 is not None:
            t2_values.append(
                float(q_props.t2)
            )

    # =========================================================
    # 10. Repeated noisy execution
    # =========================================================
    # =========================================================
    # 10. Repeated noisy execution in ONE Aer job
    # =========================================================

    repeated_circuits = [
        transpiled_circuit
        for _ in range(repeats)
    ]

    noisy_result = noisy_simulator.run(
        repeated_circuits,
        shots=shots
    ).result()

    tvds = []

    for repeat_idx in range(repeats):

        counts = noisy_result.get_counts(repeat_idx)

        noisy_probs = counts_to_probabilities(
            counts
        )

        tvd = calculate_tvd_from_probs(
            ideal_probs,
            noisy_probs
        )

        tvds.append(float(tvd))
    # =========================================================
    # 11. Build output row
    # =========================================================

    result = {}

    # Input features
    for i, value in enumerate(raw_data_row, start=1):
        result[f"raw_feature_{i}"] = float(value)

    # Encoded angles
    for i, value in enumerate(data_row, start=1):
        result[f"angle_{i}"] = float(value)

    # Abstract features
    result.update({
        "abstract_n_qubits":
            abstract_circuit.num_qubits,

        "abstract_depth":
            abstract_circuit.depth(),

        "abstract_total_gate_count":
            abstract_total_gates,

        "abstract_single_qubit_gate_count":
            abstract_1q,

        "abstract_two_qubit_gate_count":
            abstract_2q,

        "abstract_cx_count":
            abstract_cx,

        "abstract_two_qubit_gate_ratio":
            abstract_2q / max(
                abstract_total_gates,
                1
            ),

        "abstract_average_gates_per_qubit":
            abstract_total_gates
            / abstract_circuit.num_qubits,

        "abstract_max_gates_per_qubit":
            max(
                qubit_gate_counts.values()
            ),

        "abstract_unique_2q_pairs":
            len(unique_2q_pairs),

        "abstract_average_qubit_degree":
            np.mean(
                list(degrees.values())
            ),

        "abstract_max_qubit_degree":
            max(degrees.values()),

        "abstract_repeated_2q_pairs":
            repeated_pairs,

        "reps": reps,

        "entanglement":
            entanglement,
    })

    # Transpiled features
    result.update({
        "transpiled_depth":
            transpiled_circuit.depth(),

        "transpiled_total_gates":
            transpiled_total,

        "transpiled_1q_gates":
            transpiled_1q,

        "transpiled_2q_gates":
            transpiled_2q,

        "transpiled_cz_count":
            transpiled_cz,

        "transpiled_cx_count":
            transpiled_cx,

        "transpiled_sx_count":
            transpiled_sx,

        "transpiled_rz_count":
            transpiled_rz,

        "backend_size":
            backend.num_qubits,
    })

    # CZ calibration
    result.update({
        "mean_cz_error":
            np.mean(cz_errors)
            if cz_errors else np.nan,

        "std_cz_error":
            np.std(cz_errors)
            if cz_errors else np.nan,

        "min_cz_error":
            np.min(cz_errors)
            if cz_errors else np.nan,

        "max_cz_error":
            np.max(cz_errors)
            if cz_errors else np.nan,

        "sum_cz_error":
            np.sum(cz_errors)
            if cz_errors else 0.0,

        "cumulative_cz_error":
            1.0 - np.prod(
                [1.0 - e for e in cz_errors]
            )
            if cz_errors else 0.0,

        "mean_cz_duration":
            np.mean(cz_durations)
            if cz_durations else np.nan,

        "max_cz_duration":
            np.max(cz_durations)
            if cz_durations else np.nan,

        "total_cz_duration":
            np.sum(cz_durations)
            if cz_durations else 0.0,
    })

    # Physical qubits
    result.update({
        "physical_qubit_count":
            len(physical_qubits_used),

        "physical_qubits_used":
            str(physical_qubits_used),
    })

    # T1
    result.update({
        "mean_t1":
            np.mean(t1_values)
            if t1_values else np.nan,

        "min_t1":
            np.min(t1_values)
            if t1_values else np.nan,

        "max_t1":
            np.max(t1_values)
            if t1_values else np.nan,
    })

    # T2
    result.update({
        "mean_t2":
            np.mean(t2_values)
            if t2_values else np.nan,

        "min_t2":
            np.min(t2_values)
            if t2_values else np.nan,

        "max_t2":
            np.max(t2_values)
            if t2_values else np.nan,
    })

    # Target
    result.update({
        "mean_tvd":
            np.mean(tvds),

        "std_tvd":
            np.std(tvds, ddof=1),

        "min_tvd":
            np.min(tvds),

        "max_tvd":
            np.max(tvds),

        "shots":
            shots,

        "repeats":
            repeats,

        "seed_transpiler":
            seed_transpiler,

        "initial_layout":
            str(initial_layout),
    })

    return result

In [17]:
test_raw = feature_data.iloc[0].to_numpy()
test_scaled = scaled_features[0]

test_result = run_complete_experiment(
    raw_data_row=test_raw,
    data_row=test_scaled,
    reps=1,
    entanglement="linear",
    backend=fake_backend,
    noisy_simulator=noisy_simulator,
    shots=1024,
    repeats=5,
    initial_layout=CANDIDATE_LAYOUTS[0],
    seed_transpiler=42,
)

for key, value in test_result.items():
    print(f"{key:35s}: {value}")

raw_feature_1                      : 15.18
raw_feature_2                      : 0.542
raw_feature_3                      : 0.301
raw_feature_4                      : 0.563
raw_feature_5                      : 0.529
angle_1                            : 4.415042995827875
angle_2                            : 3.4054864364913358
angle_3                            : 1.6600836969495538
angle_4                            : 3.232349774693498
angle_5                            : 3.3573788156545468
abstract_n_qubits                  : 5
abstract_depth                     : 14
abstract_total_gate_count          : 22
abstract_single_qubit_gate_count   : 14
abstract_two_qubit_gate_count      : 8
abstract_cx_count                  : 8
abstract_two_qubit_gate_ratio      : 0.36363636363636365
abstract_average_gates_per_qubit   : 4.4
abstract_max_gates_per_qubit       : 7
abstract_unique_2q_pairs           : 4
abstract_average_qubit_degree      : 1.6
abstract_max_qubit_degree          : 2
abstract_repea

In [ ]:
test_raw = feature_data.iloc[0].to_numpy()
test_scaled = scaled_features[0]                          #improved timing

start = time.time()

test_result = run_complete_experiment(
    raw_data_row=test_raw,
    data_row=test_scaled,
    reps=1,
    entanglement="linear",
    backend=fake_backend,
    noisy_simulator=noisy_simulator,
    shots=1024,
    repeats=5,
    initial_layout=CANDIDATE_LAYOUTS[0],
    seed_transpiler=42,
)

elapsed = time.time() - start

print(f"Time: {elapsed:.2f} seconds")
print(f"Mean TVD: {test_result['mean_tvd']:.4f}")
print(f"Depth: {test_result['transpiled_depth']}")
print(f"2Q gates: {test_result['transpiled_2q_gates']}")

Time: 5.60 seconds
Mean TVD: 0.0730
Depth: 46
2Q gates: 8


In [28]:
benchmark_results = []

start = time.time()

for i in range(10):

    result = run_complete_experiment(
        raw_data_row=sampled_raw[i],
        data_row=sampled_scaled[i],
        reps=1,
        entanglement="linear",
        backend=fake_backend,
        noisy_simulator=noisy_simulator,
        shots=1024,
        repeats=5,
        initial_layout=CANDIDATE_LAYOUTS[0],
        seed_transpiler=42,
    )

    benchmark_results.append(result)

elapsed = time.time() - start

print("10 experiments:", round(elapsed, 2), "seconds")
print("Average:", round(elapsed / 10, 2), "sec/experiment")

10 experiments: 23.19 seconds
Average: 2.32 sec/experiment


In [19]:
CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

print("Configurations:", CONFIGS)
print("Number of configurations:", len(CONFIGS))
print("Number of layouts:", len(CANDIDATE_LAYOUTS))

Configurations: [(1, 'linear'), (2, 'linear'), (3, 'linear'), (1, 'full'), (2, 'full'), (3, 'full')]
Number of configurations: 6
Number of layouts: 4


In [20]:
N_ROWS = 200

df_with_features = (
    df
    .dropna(subset=selected_columns)
    .reset_index(drop=True)
)

sampled_df, _ = train_test_split(
    df_with_features,
    train_size=N_ROWS,
    stratify=df_with_features["cell_type"],
    random_state=42,
)

sampled_raw = (
    sampled_df[selected_columns]
    .to_numpy()
)

sampled_scaled = scaler.transform(
    sampled_raw
)

print("Sampled rows:", sampled_raw.shape)

print(
    "Total experiments:",
    len(sampled_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

Sampled rows: (200, 5)
Total experiments: 4800


In [21]:
PILOT_N_ROWS = 5

pilot_sample_df = (
    sampled_df
    .iloc[:PILOT_N_ROWS]
    .reset_index(drop=True)
)

pilot_raw = (
    pilot_sample_df[selected_columns]
    .to_numpy()
)

pilot_scaled = scaler.transform(
    pilot_raw
)

print("Pilot rows:", pilot_raw.shape)
print(
    "Pilot experiments:",
    len(pilot_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

Pilot rows: (5, 5)
Pilot experiments: 120


In [22]:
CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

pilot_results = []

total = (
    len(pilot_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

done = 0
start_time = time.time()

for row_idx in range(len(pilot_raw)):

    for reps, entanglement in CONFIGS:

        for layout in CANDIDATE_LAYOUTS:

            exp_start = time.time()

            result = run_complete_experiment(
                raw_data_row=pilot_raw[row_idx],
                data_row=pilot_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,
                backend=fake_backend,
                noisy_simulator=noisy_simulator,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            result["data_row_index"] = row_idx
            result["layout_id"] = str(layout)

            pilot_results.append(result)

            done += 1

            print(
                f"[{done}/{total}] "
                f"row={row_idx}, "
                f"reps={reps}, "
                f"entanglement={entanglement}, "
                f"layout={layout} | "
                f"{time.time() - exp_start:.1f}s | "
                f"TVD={result['mean_tvd']:.4f}"
            )

pilot_df = pd.DataFrame(pilot_results)

elapsed = time.time() - start_time

print("\nPilot complete")
print("Shape:", pilot_df.shape)
print("Elapsed:", round(elapsed / 60, 2), "minutes")
print(
    "Average:",
    round(elapsed / total, 2),
    "sec/experiment"
)

[1/120] row=0, reps=1, entanglement=linear, layout=[10, 11, 12, 13, 14] | 10.7s | TVD=0.0742
[2/120] row=0, reps=1, entanglement=linear, layout=[61, 62, 63, 64, 65] | 10.4s | TVD=0.0734
[3/120] row=0, reps=1, entanglement=linear, layout=[147, 148, 149, 150, 151] | 10.4s | TVD=0.0643
[4/120] row=0, reps=1, entanglement=linear, layout=[83, 84, 85, 86, 87] | 10.5s | TVD=0.0705
[5/120] row=0, reps=2, entanglement=linear, layout=[10, 11, 12, 13, 14] | 11.1s | TVD=0.0996
[6/120] row=0, reps=2, entanglement=linear, layout=[61, 62, 63, 64, 65] | 10.6s | TVD=0.0801
[7/120] row=0, reps=2, entanglement=linear, layout=[147, 148, 149, 150, 151] | 10.3s | TVD=0.0660
[8/120] row=0, reps=2, entanglement=linear, layout=[83, 84, 85, 86, 87] | 13.5s | TVD=0.1519
[9/120] row=0, reps=3, entanglement=linear, layout=[10, 11, 12, 13, 14] | 27.5s | TVD=0.0908
[10/120] row=0, reps=3, entanglement=linear, layout=[61, 62, 63, 64, 65] | 26.7s | TVD=0.0696
[11/120] row=0, reps=3, entanglement=linear, layout=[147, 1

In [23]:
print("Missing values:")
print(
    pilot_df.isna().sum()[
        lambda x: x > 0
    ]
)

print(
    "\nTVD statistics:"
)

print(
    pilot_df[
        [
            "mean_tvd",
            "std_tvd",
            "min_tvd",
            "max_tvd",
        ]
    ].describe()
)

print(
    "\nKey variation:"
)

print(
    pilot_df[
        [
            "reps",
            "entanglement",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "mean_tvd",
        ]
    ]
    .groupby(
        ["reps", "entanglement"]
    )
    .mean()
)

Missing values:
Series([], dtype: int64)

TVD statistics:
         mean_tvd     std_tvd     min_tvd     max_tvd
count  120.000000  120.000000  120.000000  120.000000
mean     0.121371    0.010645    0.108181    0.134735
std      0.083769    0.004105    0.082116    0.085376
min      0.056836    0.003615    0.041992    0.066406
25%      0.069368    0.007637    0.057373    0.079582
50%      0.079904    0.010078    0.069498    0.097656
75%      0.145506    0.013332    0.129818    0.160770
max      0.437993    0.025050    0.416586    0.451524

Key variation:
                   transpiled_depth  transpiled_2q_gates  mean_cz_error   mean_t1   mean_t2  mean_tvd
reps entanglement                                                                                    
1    full                     124.8                 44.0       0.005768  0.000273  0.000171  0.068760
     linear                    44.8                  8.0       0.004527  0.000273  0.000171  0.069980
2    full                     23

In [24]:
print(
    "\nLayout variation:"
)

print(
    pilot_df[
        [
            "layout_id",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "mean_tvd",
        ]
    ]
    .groupby("layout_id")
    .mean()
)


Layout variation:
                           transpiled_depth  transpiled_2q_gates  mean_cz_error   mean_t1   mean_t2  mean_tvd
layout_id                                                                                                    
[10, 11, 12, 13, 14]             151.766667                 53.5       0.001238  0.000306  0.000220  0.103695
[147, 148, 149, 150, 151]        151.766667                 53.5       0.002323  0.000323  0.000266  0.094001
[61, 62, 63, 64, 65]             151.766667                 53.5       0.001798  0.000237  0.000109  0.092195
[83, 84, 85, 86, 87]             151.766667                 53.5       0.014383  0.000226  0.000089  0.195592


In [29]:
all_results = []

total = (
    len(sampled_raw)
    * len(CONFIGS)
    * len(CANDIDATE_LAYOUTS)
)

done = 0
start_time = time.time()

print("=" * 70)
print("STARTING FULL FAKEKINGSTON DATASET GENERATION")
print("=" * 70)
print("Rows:", len(sampled_raw))
print("Configurations:", len(CONFIGS))
print("Layouts:", len(CANDIDATE_LAYOUTS))
print("Total experiments:", total)
print("Shots per repeat:", 1024)
print("Repeats per experiment:", 5)
print("=" * 70)

for row_idx in range(len(sampled_raw)):

    for reps, entanglement in CONFIGS:

        for layout in CANDIDATE_LAYOUTS:

            result = run_complete_experiment(
                raw_data_row=sampled_raw[row_idx],
                data_row=sampled_scaled[row_idx],
                reps=reps,
                entanglement=entanglement,
                backend=fake_backend,
                noisy_simulator=noisy_simulator,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            result["data_row_index"] = row_idx
            result["layout_id"] = str(layout)

            all_results.append(result)

            done += 1

    if (row_idx + 1) % 10 == 0:

        elapsed = time.time() - start_time
        avg_per_experiment = elapsed / done
        remaining = total - done
        eta_minutes = (
            remaining * avg_per_experiment / 60
        )

        print(
            f"Completed {row_idx + 1}/{len(sampled_raw)} rows "
            f"({done}/{total} experiments) | "
            f"Elapsed: {elapsed / 60:.1f} min | "
            f"ETA: {eta_minutes:.1f} min"
        )

predictor_df_kingston = pd.DataFrame(all_results)

print("\n" + "=" * 70)
print("DATASET GENERATION COMPLETE")
print("=" * 70)
print("Final shape:", predictor_df_kingston.shape)

STARTING FULL FAKEKINGSTON DATASET GENERATION
Rows: 200
Configurations: 6
Layouts: 4
Total experiments: 4800
Shots per repeat: 1024
Repeats per experiment: 5
Completed 10/200 rows (240/4800 experiments) | Elapsed: 23.9 min | ETA: 453.6 min
Completed 20/200 rows (480/4800 experiments) | Elapsed: 51.3 min | ETA: 461.6 min
Completed 30/200 rows (720/4800 experiments) | Elapsed: 88.5 min | ETA: 501.7 min
Completed 40/200 rows (960/4800 experiments) | Elapsed: 107.4 min | ETA: 429.5 min
Completed 50/200 rows (1200/4800 experiments) | Elapsed: 117.9 min | ETA: 353.8 min
Completed 60/200 rows (1440/4800 experiments) | Elapsed: 128.5 min | ETA: 299.8 min
Completed 70/200 rows (1680/4800 experiments) | Elapsed: 158.1 min | ETA: 293.6 min
Completed 80/200 rows (1920/4800 experiments) | Elapsed: 192.1 min | ETA: 288.2 min
Completed 90/200 rows (2160/4800 experiments) | Elapsed: 223.6 min | ETA: 273.3 min
Completed 100/200 rows (2400/4800 experiments) | Elapsed: 256.1 min | ETA: 256.1 min
Complete

In [2]:
print("Shape:", predictor_df_kingston.shape)

print("\nMissing values:")
missing = predictor_df_kingston.isna().sum()
print(missing[missing > 0])

print("\nDuplicate rows:")
print(predictor_df_kingston.duplicated().sum())

NameError: name 'predictor_df_kingston' is not defined

In [32]:
print("\nConfiguration counts:")
print(
    predictor_df_kingston
    .groupby(["reps", "entanglement"])
    .size()
)

print("\nLayout counts:")
print(
    predictor_df_kingston["layout_id"]
    .value_counts()
)


Configuration counts:
reps  entanglement
1     full            800
      linear          800
2     full            800
      linear          800
3     full            800
      linear          800
dtype: int64

Layout counts:
layout_id
[10, 11, 12, 13, 14]         1200
[61, 62, 63, 64, 65]         1200
[147, 148, 149, 150, 151]    1200
[83, 84, 85, 86, 87]         1200
Name: count, dtype: int64


In [33]:
print("\nTVD statistics:")
print(
    predictor_df_kingston[
        ["mean_tvd", "std_tvd", "min_tvd", "max_tvd"]
    ].describe()
)


TVD statistics:
          mean_tvd      std_tvd      min_tvd      max_tvd
count  4800.000000  4800.000000  4800.000000  4800.000000
mean      0.119787     0.010745     0.106695     0.133257
std       0.083998     0.004411     0.082229     0.085987
min       0.045234     0.001070     0.032765     0.054010
25%       0.068843     0.007607     0.057684     0.080078
50%       0.078224     0.010111     0.068138     0.093594
75%       0.139571     0.013381     0.124750     0.155534
max       0.577757     0.031405     0.559974     0.596905


In [34]:
print("\nConfiguration variation:")
print(
    predictor_df_kingston[
        [
            "reps",
            "entanglement",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "mean_tvd",
        ]
    ]
    .groupby(["reps", "entanglement"])
    .mean()
)


Configuration variation:
                   transpiled_depth  transpiled_2q_gates  mean_cz_error   mean_t1   mean_t2  mean_tvd
reps entanglement                                                                                    
1    full                   125.195                43.96       0.005769  0.000273  0.000171  0.069220
     linear                  45.260                 7.99       0.004530  0.000273  0.000171  0.069255
2    full                   238.340                90.92       0.005048  0.000273  0.000171  0.189130
     linear                  67.210                15.98       0.004530  0.000273  0.000171  0.092328
3    full                   350.590               137.88       0.005217  0.000273  0.000171  0.201749
     linear                  89.160                23.97       0.004530  0.000273  0.000171  0.097042


In [35]:
print("\nLayout variation:")
print(
    predictor_df_kingston[
        [
            "layout_id",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "mean_tvd",
        ]
    ]
    .groupby("layout_id")
    .mean()
)


Layout variation:
                           transpiled_depth  transpiled_2q_gates  mean_cz_error   mean_t1   mean_t2  mean_tvd
layout_id                                                                                                    
[10, 11, 12, 13, 14]             152.625833                53.45       0.001238  0.000306  0.000220  0.098974
[147, 148, 149, 150, 151]        152.625833                53.45       0.002324  0.000323  0.000266  0.093519
[61, 62, 63, 64, 65]             152.625833                53.45       0.001797  0.000237  0.000109  0.092052
[83, 84, 85, 86, 87]             152.625833                53.45       0.014391  0.000226  0.000089  0.194604


In [36]:
output_path = (
    "../experiments/results/"
    "quantum_noise_predictor_dataset_kingston.csv"
)

predictor_df_kingston.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: ../experiments/results/quantum_noise_predictor_dataset_kingston.csv


In [37]:
metadata = {
    "backend": "FakeKingston",
    "n_rows": 200,
    "n_experiments": 4800,
    "n_features": 61,
    "shots": 1024,
    "repeats": 5,
    "seed_transpiler": 42,
    "configs": CONFIGS,
    "layouts": CANDIDATE_LAYOUTS,
    "target": "mean_tvd",
}

print(metadata)

{'backend': 'FakeKingston', 'n_rows': 200, 'n_experiments': 4800, 'n_features': 61, 'shots': 1024, 'repeats': 5, 'seed_transpiler': 42, 'configs': [(1, 'linear'), (2, 'linear'), (3, 'linear'), (1, 'full'), (2, 'full'), (3, 'full')], 'layouts': [[10, 11, 12, 13, 14], [61, 62, 63, 64, 65], [147, 148, 149, 150, 151], [83, 84, 85, 86, 87]], 'target': 'mean_tvd'}


In [38]:
import json

with open(
    "../experiments/results/kingston_dataset_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=2)

In [41]:
# ============================================================
# SELECT 12 EXISTING KINGSTON DATASET CIRCUITS
# 4 low + 4 medium + 4 high TVD
# ============================================================

validation_df = predictor_df_kingston.copy()

validation_df["tvd_band"] = pd.qcut(
    validation_df["mean_tvd"],
    q=3,
    labels=["low", "medium", "high"]
)

validation_parts = []

for band in ["low", "medium", "high"]:

    band_df = validation_df[
        validation_df["tvd_band"] == band
    ]

    selected = band_df.sample(
        n=4,
        random_state=42
    )

    validation_parts.append(selected)

hardware_validation_df = (
    pd.concat(validation_parts)
    .reset_index(drop=True)
)

print(
    "Validation circuits:",
    len(hardware_validation_df)
)

print(
    hardware_validation_df[
        [
            "data_row_index",
            "reps",
            "entanglement",
            "initial_layout",
            "mean_tvd",
            "transpiled_depth",
            "transpiled_2q_gates",
            "mean_cz_error",
            "mean_t1",
            "mean_t2",
            "tvd_band",
        ]
    ].to_string(index=False)
)

Validation circuits: 12
 data_row_index  reps entanglement            initial_layout  mean_tvd  transpiled_depth  transpiled_2q_gates  mean_cz_error  mean_t1  mean_t2 tvd_band
             64     3       linear [147, 148, 149, 150, 151]  0.064096                90                   24       0.002294 0.000323 0.000266      low
             43     2       linear      [61, 62, 63, 64, 65]  0.065621                68                   16       0.001791 0.000237 0.000109      low
             21     1         full      [61, 62, 63, 64, 65]  0.068164               126                   44       0.001834 0.000237 0.000109      low
             17     1       linear      [61, 62, 63, 64, 65]  0.067383                44                    8       0.001791 0.000237 0.000109      low
            100     2         full [147, 148, 149, 150, 151]  0.110417               234                   91       0.002322 0.000323 0.000266   medium
             15     2         full      [10, 11, 12, 13, 14]  0.

In [42]:
print("\nConfiguration coverage:")
print(
    hardware_validation_df
    .groupby(["reps", "entanglement"])
    .size()
)

print("\nLayout coverage:")
print(
    hardware_validation_df["layout_id"]
    .value_counts()
)


Configuration coverage:
reps  entanglement
1     full            2
      linear          1
2     full            4
      linear          1
3     full            1
      linear          3
dtype: int64

Layout coverage:
layout_id
[61, 62, 63, 64, 65]         4
[83, 84, 85, 86, 87]         4
[147, 148, 149, 150, 151]    2
[10, 11, 12, 13, 14]         2
Name: count, dtype: int64


In [43]:
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

service = QiskitRuntimeService()

backend = service.backend("ibm_kingston")

print("Backend:", backend.name)
print("Qubits:", backend.num_qubits)

Backend: ibm_kingston
Qubits: 156


In [44]:
real_circuits = []
ideal_distributions = []

for i, row in hardware_validation_df.iterrows():

    data_row_index = int(row["data_row_index"])

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    layout = eval(row["initial_layout"])

    # Same scaled input used during dataset generation
    data_row = sampled_scaled[data_row_index]

    # --------------------------------------------------------
    # Build exact logical circuit
    # --------------------------------------------------------

    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(
        data_row
    )

    # --------------------------------------------------------
    # Exact ideal distribution
    # --------------------------------------------------------

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    ideal_distributions.append(
        ideal_probs
    )

    # --------------------------------------------------------
    # Add measurements
    # --------------------------------------------------------

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # --------------------------------------------------------
    # Transpile exactly as dataset generation
    # --------------------------------------------------------

    transpiled = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=layout,
        seed_transpiler=42,
    )

    real_circuits.append(transpiled)

    print(
        f"{i}: "
        f"reps={reps}, "
        f"entanglement={entanglement}, "
        f"depth={transpiled.depth()}, "
        f"2Q={sum(len(inst.qubits) == 2 for inst in transpiled.data)}, "
        f"layout={layout}"
    )

0: reps=3, entanglement=linear, depth=90, 2Q=24, layout=[147, 148, 149, 150, 151]
1: reps=2, entanglement=linear, depth=68, 2Q=16, layout=[61, 62, 63, 64, 65]
2: reps=1, entanglement=full, depth=126, 2Q=44, layout=[61, 62, 63, 64, 65]
3: reps=1, entanglement=linear, depth=44, 2Q=8, layout=[61, 62, 63, 64, 65]
4: reps=2, entanglement=full, depth=234, 2Q=91, layout=[147, 148, 149, 150, 151]
5: reps=2, entanglement=full, depth=241, 2Q=91, layout=[10, 11, 12, 13, 14]
6: reps=2, entanglement=full, depth=241, 2Q=91, layout=[61, 62, 63, 64, 65]
7: reps=1, entanglement=full, depth=126, 2Q=44, layout=[83, 84, 85, 86, 87]
8: reps=3, entanglement=linear, depth=90, 2Q=24, layout=[83, 84, 85, 86, 87]
9: reps=3, entanglement=linear, depth=90, 2Q=24, layout=[83, 84, 85, 86, 87]
10: reps=2, entanglement=full, depth=234, 2Q=91, layout=[83, 84, 85, 86, 87]
11: reps=3, entanglement=full, depth=354, 2Q=138, layout=[10, 11, 12, 13, 14]


In [45]:
for i, row in hardware_validation_df.iterrows():

    recreated_depth = real_circuits[i].depth()

    recreated_2q = sum(
        len(inst.qubits) == 2
        for inst in real_circuits[i].data
    )

    print(
        f"{i}: "
        f"stored depth={row['transpiled_depth']}, "
        f"recreated depth={recreated_depth} | "
        f"stored 2Q={row['transpiled_2q_gates']}, "
        f"recreated 2Q={recreated_2q}"
    )

0: stored depth=90, recreated depth=90 | stored 2Q=24, recreated 2Q=24
1: stored depth=68, recreated depth=68 | stored 2Q=16, recreated 2Q=16
2: stored depth=126, recreated depth=126 | stored 2Q=44, recreated 2Q=44
3: stored depth=44, recreated depth=44 | stored 2Q=8, recreated 2Q=8
4: stored depth=234, recreated depth=234 | stored 2Q=91, recreated 2Q=91
5: stored depth=241, recreated depth=241 | stored 2Q=91, recreated 2Q=91
6: stored depth=241, recreated depth=241 | stored 2Q=91, recreated 2Q=91
7: stored depth=126, recreated depth=126 | stored 2Q=44, recreated 2Q=44
8: stored depth=90, recreated depth=90 | stored 2Q=24, recreated 2Q=24
9: stored depth=90, recreated depth=90 | stored 2Q=24, recreated 2Q=24
10: stored depth=234, recreated depth=234 | stored 2Q=91, recreated 2Q=91
11: stored depth=354, recreated depth=354 | stored 2Q=138, recreated 2Q=138


In [46]:
for i, row in hardware_validation_df.iterrows():

    recreated_depth = real_circuits[i].depth()

    recreated_2q = sum(
        len(inst.qubits) == 2
        for inst in real_circuits[i].data
    )

    print(
        f"{i}: "
        f"stored depth={row['transpiled_depth']}, "
        f"recreated depth={recreated_depth} | "
        f"stored 2Q={row['transpiled_2q_gates']}, "
        f"recreated 2Q={recreated_2q}"
    )

0: stored depth=90, recreated depth=90 | stored 2Q=24, recreated 2Q=24
1: stored depth=68, recreated depth=68 | stored 2Q=16, recreated 2Q=16
2: stored depth=126, recreated depth=126 | stored 2Q=44, recreated 2Q=44
3: stored depth=44, recreated depth=44 | stored 2Q=8, recreated 2Q=8
4: stored depth=234, recreated depth=234 | stored 2Q=91, recreated 2Q=91
5: stored depth=241, recreated depth=241 | stored 2Q=91, recreated 2Q=91
6: stored depth=241, recreated depth=241 | stored 2Q=91, recreated 2Q=91
7: stored depth=126, recreated depth=126 | stored 2Q=44, recreated 2Q=44
8: stored depth=90, recreated depth=90 | stored 2Q=24, recreated 2Q=24
9: stored depth=90, recreated depth=90 | stored 2Q=24, recreated 2Q=24
10: stored depth=234, recreated depth=234 | stored 2Q=91, recreated 2Q=91
11: stored depth=354, recreated depth=354 | stored 2Q=138, recreated 2Q=138


In [47]:
sampler = Sampler(mode=backend)

print("Submitting", len(real_circuits), "circuits...")

job = sampler.run(
    real_circuits,
    shots=1024
)

print("Job ID:", job.job_id())

Submitting 12 circuits...
Job ID: da5neijotlns739c4q3g


In [48]:
print("Waiting for hardware result...")

result = job.result()

print("Hardware execution complete.")

Waiting for hardware result...
Hardware execution complete.


In [49]:
real_counts = []

for i in range(len(real_circuits)):

    counts = result[i].data.meas.get_counts()

    real_counts.append(counts)

    print(
        f"Circuit {i}: "
        f"{sum(counts.values())} shots, "
        f"{len(counts)} unique outcomes"
    )

Circuit 0: 1024 shots, 27 unique outcomes
Circuit 1: 1024 shots, 32 unique outcomes
Circuit 2: 1024 shots, 32 unique outcomes
Circuit 3: 1024 shots, 32 unique outcomes
Circuit 4: 1024 shots, 32 unique outcomes
Circuit 5: 1024 shots, 32 unique outcomes
Circuit 6: 1024 shots, 32 unique outcomes
Circuit 7: 1024 shots, 32 unique outcomes
Circuit 8: 1024 shots, 32 unique outcomes
Circuit 9: 1024 shots, 32 unique outcomes
Circuit 10: 1024 shots, 32 unique outcomes
Circuit 11: 1024 shots, 32 unique outcomes


In [50]:
real_tvd = []

for i in range(len(real_circuits)):

    noisy_probs = counts_to_probabilities(
        real_counts[i]
    )

    tvd = calculate_tvd_from_probs(
        ideal_distributions[i],
        noisy_probs
    )

    real_tvd.append(tvd)

    print(
        f"Circuit {i}: "
        f"Real Kingston TVD = {tvd:.6f}"
    )

Circuit 0: Real Kingston TVD = 0.119568
Circuit 1: Real Kingston TVD = 0.123850
Circuit 2: Real Kingston TVD = 0.107422
Circuit 3: Real Kingston TVD = 0.075195
Circuit 4: Real Kingston TVD = 0.235060
Circuit 5: Real Kingston TVD = 0.122893
Circuit 6: Real Kingston TVD = 0.235272
Circuit 7: Real Kingston TVD = 0.243164
Circuit 8: Real Kingston TVD = 0.349127
Circuit 9: Real Kingston TVD = 0.226329
Circuit 10: Real Kingston TVD = 0.412387
Circuit 11: Real Kingston TVD = 0.194677


In [51]:
comparison_df = hardware_validation_df[
    [
        "data_row_index",
        "reps",
        "entanglement",
        "initial_layout",
        "mean_tvd",
        "tvd_band",
    ]
].copy()

comparison_df = comparison_df.rename(
    columns={
        "mean_tvd": "fake_kingston_tvd"
    }
)

comparison_df["real_kingston_tvd"] = real_tvd

print(comparison_df)

    data_row_index  reps entanglement             initial_layout  fake_kingston_tvd tvd_band  real_kingston_tvd
0               64     3       linear  [147, 148, 149, 150, 151]           0.064096      low           0.119568
1               43     2       linear       [61, 62, 63, 64, 65]           0.065621      low           0.123850
2               21     1         full       [61, 62, 63, 64, 65]           0.068164      low           0.107422
3               17     1       linear       [61, 62, 63, 64, 65]           0.067383      low           0.075195
4              100     2         full  [147, 148, 149, 150, 151]           0.110417   medium           0.235060
5               15     2         full       [10, 11, 12, 13, 14]           0.109211   medium           0.122893
6               44     2         full       [61, 62, 63, 64, 65]           0.124103   medium           0.235272
7               87     1         full       [83, 84, 85, 86, 87]           0.072070   medium           0

In [52]:
from sklearn.metrics import mean_absolute_error

correlation = comparison_df[
    "fake_kingston_tvd"
].corr(
    comparison_df["real_kingston_tvd"]
)

mae = mean_absolute_error(
    comparison_df["fake_kingston_tvd"],
    comparison_df["real_kingston_tvd"]
)

print(
    f"FakeKingston → Real Kingston\n"
    f"Correlation: {correlation:.4f}\n"
    f"MAE: {mae:.4f}"
)

FakeKingston → Real Kingston
Correlation: 0.8306
MAE: 0.0819


In [53]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(
        comparison_df["fake_kingston_tvd"],
        comparison_df["real_kingston_tvd"]
    )
)

print(f"RMSE: {rmse:.4f}")

RMSE: 0.0991


In [54]:
comparison_df.to_csv(
    "../experiments/results/"
    "kingston_hardware_validation.csv",
    index=False
)

print("Validation results saved.")

Validation results saved.


In [55]:
comparison_df["abs_error"] = (
    comparison_df["fake_kingston_tvd"]
    - comparison_df["real_kingston_tvd"]
).abs()

print(
    comparison_df
    .sort_values("abs_error", ascending=False)
    [
        [
            "data_row_index",
            "reps",
            "entanglement",
            "initial_layout",
            "fake_kingston_tvd",
            "real_kingston_tvd",
            "abs_error",
            "tvd_band",
        ]
    ]
)

    data_row_index  reps entanglement             initial_layout  fake_kingston_tvd  real_kingston_tvd  abs_error tvd_band
8               64     3       linear       [83, 84, 85, 86, 87]           0.157442           0.349127   0.191685     high
7               87     1         full       [83, 84, 85, 86, 87]           0.072070           0.243164   0.171094   medium
4              100     2         full  [147, 148, 149, 150, 151]           0.110417           0.235060   0.124643   medium
6               44     2         full       [61, 62, 63, 64, 65]           0.124103           0.235272   0.111168   medium
10              20     2         full       [83, 84, 85, 86, 87]           0.317672           0.412387   0.094715     high
9               44     3       linear       [83, 84, 85, 86, 87]           0.161501           0.226329   0.064828     high
1               43     2       linear       [61, 62, 63, 64, 65]           0.065621           0.123850   0.058230      low
0               

In [56]:
errors = comparison_df["abs_error"].sort_values(
    ascending=False
).reset_index(drop=True)

for k in [0, 1, 2, 3, 4]:
    filtered = comparison_df[
        ~comparison_df["abs_error"].isin(errors.head(k))
    ]

    mae_k = filtered["abs_error"].mean()

    rmse_k = np.sqrt(
        np.mean(
            (
                filtered["fake_kingston_tvd"]
                - filtered["real_kingston_tvd"]
            ) ** 2
        )
    )

    print(
        f"Excluding top {k} errors: "
        f"MAE={mae_k:.4f}, RMSE={rmse_k:.4f}, "
        f"n={len(filtered)}"
    )

Excluding top 0 errors: MAE=0.0819, RMSE=0.0991, n=12
Excluding top 1 errors: MAE=0.0719, RMSE=0.0859, n=11
Excluding top 2 errors: MAE=0.0620, RMSE=0.0720, n=10
Excluding top 3 errors: MAE=0.0551, RMSE=0.0635, n=9
Excluding top 4 errors: MAE=0.0480, RMSE=0.0547, n=8


In [ ]:
comparison_df["abs_error"] = (
    comparison_df["fake_kingston_tvd"]
    - comparison_df["real_kingston_tvd"]
).abs()

print(
    comparison_df
    .groupby("initial_layout")
    .agg(
        mae=("abs_error", "mean"),
        mean_fake=("fake_kingston_tvd", "mean"),+
        mean_real=("real_kingston_tvd", "mean"),
        n=("abs_error", "size"),
    )
)

                                mae  mean_fake  mean_real  n
initial_layout                                              
[10, 11, 12, 13, 14]       0.031999   0.126787   0.158785  2
[147, 148, 149, 150, 151]  0.090058   0.087257   0.177314  2
[61, 62, 63, 64, 65]       0.054117   0.081318   0.135435  4
[83, 84, 85, 86, 87]       0.130580   0.177172   0.307752  4


In [ ]:
layout_results = []

for layout, group in comparison_df.groupby("initial_layout"):

    correlation = group["fake_kingston_tvd"].corr(
        group["real_kingston_tvd"]
    )

    mae = group["abs_error"].mean()

    layout_results.append({
        "layout": layout,
        "correlation": correlation,
        "mae": mae,
        "mean_fake": group["fake_kingston_tvd"].mean(),
        "mean_real": group["real_kingston_tvd"].mean(),
        "n": len(group),
    })

layout_results_df = pd.DataFrame(layout_results)

print(layout_results_df.to_string(index=False))

                   layout  correlation      mae  mean_fake  mean_real  n
     [10, 11, 12, 13, 14]     1.000000 0.031999   0.126787   0.158785  2
[147, 148, 149, 150, 151]     1.000000 0.090058   0.087257   0.177314  2
     [61, 62, 63, 64, 65]     0.950472 0.054117   0.081318   0.135435  4
     [83, 84, 85, 86, 87]     0.808399 0.130580   0.177172   0.307752  4


: 

In [35]:
import pandas as pd
import numpy as np
import ast
import json
import time

kingston_path = (
    "../experiments/results/"
    "quantum_noise_predictor_dataset_kingston.csv"
)

predictor_df_kingston = pd.read_csv(kingston_path)

print(predictor_df_kingston.shape)

(4800, 61)


In [36]:
from sklearn.model_selection import train_test_split

DATA_PATH = "../data/raw/blood_cell_anomaly_detection.csv"

df = pd.read_csv(DATA_PATH)

selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

df_with_features = (
    df.dropna(subset=selected_columns)
      .reset_index(drop=True)
)

sampled_df, _ = train_test_split(
    df_with_features,
    train_size=200,
    stratify=df_with_features["cell_type"],
    random_state=42
)

sampled_raw = sampled_df[selected_columns].values
sampled_scaled = scaler.transform(sampled_raw)

print(sampled_raw.shape)
print(sampled_scaled.shape)

(200, 5)
(200, 5)


c:\Users\jaspr\miniconda3\envs\qnoise\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [37]:
old_validation = pd.read_csv(
    "../experiments/results/"
    "kingston_hardware_validation.csv"
)

old_validation["initial_layout"] = (
    old_validation["initial_layout"].astype(str)
)

validation_df = predictor_df_kingston.copy()

validation_df["initial_layout"] = (
    validation_df["initial_layout"].astype(str)
)

used = set(
    zip(
        old_validation["data_row_index"].astype(int),
        old_validation["reps"].astype(int),
        old_validation["entanglement"],
        old_validation["initial_layout"]
    )
)

validation_df["circuit_key"] = list(zip(
    validation_df["data_row_index"].astype(int),
    validation_df["reps"].astype(int),
    validation_df["entanglement"],
    validation_df["initial_layout"]
))

available_df = validation_df[
    ~validation_df["circuit_key"].isin(used)
].copy()

print("Previously validated:", len(used))
print("Available:", len(available_df))

Previously validated: 12
Available: 4788


In [8]:
print(old_validation.columns.tolist())

['data_row_index', 'reps', 'entanglement', 'initial_layout', 'fake_kingston_tvd', 'tvd_band', 'real_kingston_tvd']


In [38]:
# ============================================================
# SELECT 60 ADDITIONAL EXISTING KINGSTON CIRCUITS
# No new simulation is generated.
# The 12 already run on real Kingston are excluded.
# ============================================================

import pandas as pd

old_validation = pd.read_csv(
    "../experiments/results/"
    "kingston_hardware_validation.csv"
)

# Make sure layout is represented consistently in both dataframes
old_validation["initial_layout"] = (
    old_validation["initial_layout"].astype(str)
)

validation_df["initial_layout"] = (
    validation_df["initial_layout"].astype(str)
)

# ------------------------------------------------------------
# Identify the EXACT 12 circuits already run
# ------------------------------------------------------------
used = set(
    zip(
        old_validation["data_row_index"].astype(int),
        old_validation["reps"].astype(int),
        old_validation["entanglement"],
        old_validation["initial_layout"]
    )
)

print("Previously validated circuits:", len(used))

# ------------------------------------------------------------
# Build exact circuit key for all 4800 existing circuits
# ------------------------------------------------------------
validation_df["circuit_key"] = list(zip(
    validation_df["data_row_index"].astype(int),
    validation_df["reps"].astype(int),
    validation_df["entanglement"],
    validation_df["initial_layout"]
))

# Remove the 12 already executed on real hardware
available_df = validation_df[
    ~validation_df["circuit_key"].isin(used)
].copy()

print(
    "Available circuits after exclusion:",
    len(available_df)
)

# ============================================================
# CREATE TVD BANDS
# ============================================================

available_df["tvd_band"] = pd.qcut(
    available_df["mean_tvd"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

# ============================================================
# SELECT 60 CIRCUITS
#
# We want:
#   - all 4 layouts represented
#   - all 6 circuit configurations represented
#   - low / medium / high TVD represented
# ============================================================

N_NEW = 60

group_cols = [
    "initial_layout",
    "reps",
    "entanglement",
    "tvd_band"
]

groups = list(
    available_df.groupby(
        group_cols,
        observed=True
    )
)

n_groups = len(groups)

print("Number of strata:", n_groups)

# Initial equal allocation
base = N_NEW // n_groups
remainder = N_NEW % n_groups

selected_parts = []

for idx, (_, group) in enumerate(groups):

    n_take = base + (1 if idx < remainder else 0)

    # Some strata may contain fewer rows
    n_take = min(n_take, len(group))

    if n_take > 0:

        selected_parts.append(
            group.sample(
                n=n_take,
                random_state=100 + idx
            )
        )

new_validation_df = (
    pd.concat(selected_parts)
    .drop_duplicates(
        subset=["circuit_key"]
    )
    .reset_index(drop=True)
)

# In case the exact stratification produced >60 rows
if len(new_validation_df) > N_NEW:

    new_validation_df = (
        new_validation_df
        .sample(
            n=N_NEW,
            random_state=42
        )
        .reset_index(drop=True)
    )

print(
    "\nNew Kingston validation circuits:",
    len(new_validation_df)
)

# ============================================================
# CHECK DISTRIBUTION
# ============================================================

print("\nTVD bands:")
print(
    new_validation_df["tvd_band"]
    .value_counts()
    .sort_index()
)

print("\nLayouts:")
print(
    new_validation_df["initial_layout"]
    .value_counts()
)

print("\nConfigurations:")
print(
    new_validation_df
    .groupby(
        ["reps", "entanglement"]
    )
    .size()
)

# ============================================================
# SHOW SELECTED CIRCUITS
# ============================================================

cols_to_show = [
    "data_row_index",
    "reps",
    "entanglement",
    "initial_layout",
    "mean_tvd",
    "tvd_band"
]

print("\nSelected circuits:")
print(
    new_validation_df[
        cols_to_show
    ].sort_values(
        ["initial_layout", "reps", "entanglement"]
    ).to_string(index=False)
)

Previously validated circuits: 12
Available circuits after exclusion: 4788
Number of strata: 48

New Kingston validation circuits: 60

TVD bands:
tvd_band
low       18
medium    27
high      15
Name: count, dtype: int64

Layouts:
initial_layout
[10, 11, 12, 13, 14]         26
[147, 148, 149, 150, 151]    12
[61, 62, 63, 64, 65]         12
[83, 84, 85, 86, 87]         10
Name: count, dtype: int64

Configurations:
reps  entanglement
1     full            10
      linear          10
2     full             9
      linear          12
3     full             9
      linear          10
dtype: int64

Selected circuits:
 data_row_index  reps entanglement            initial_layout  mean_tvd tvd_band
             16     1         full      [10, 11, 12, 13, 14]  0.069922      low
            169     1         full      [10, 11, 12, 13, 14]  0.063086      low
             18     1         full      [10, 11, 12, 13, 14]  0.072266   medium
            140     1         full      [10, 11, 12, 13, 14]  

In [39]:
hardware_validation_df = new_validation_df.copy()

In [40]:
print(
    new_validation_df[
        new_validation_df["initial_layout"]
        == str([83, 84, 85, 86, 87])
    ][
        [
            "data_row_index",
            "reps",
            "entanglement",
            "mean_tvd"
        ]
    ]
)

    data_row_index  reps entanglement  mean_tvd
50             104     1         full  0.070312
51             110     1         full  0.072852
52             196     1       linear  0.068555
53             164     1       linear  0.072461
54             187     2         full  0.436767
55             184     2       linear  0.111768
56             180     2       linear  0.142317
57             128     3         full  0.358952
58              93     3       linear  0.120652
59             193     3       linear  0.175068


In [42]:
from qiskit.quantum_info import Statevector
from qiskit import transpile

real_kingston_circuits = []
ideal_distributions = []

for _, row in hardware_validation_df.iterrows():

    data_idx = int(row["data_row_index"])

    scaled_row = sampled_scaled[data_idx]

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    layout = ast.literal_eval(
        row["initial_layout"]
    )

    fm = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement
    )

    abstract = fm.assign_parameters(
        scaled_row
    )

    ideal_sv = Statevector.from_instruction(
        abstract
    )

    ideal_distributions.append(
        ideal_sv.probabilities_dict()
    )

    measured = abstract.copy()
    measured.measure_all()

    isa_circuit = transpile(
        measured,
        backend,
        optimization_level=1,
        initial_layout=layout,
        seed_transpiler=42
    )

    real_kingston_circuits.append(
        isa_circuit
    )

print(
    "Prepared:",
    len(real_kingston_circuits)
)

Prepared: 60


In [43]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform"
)

backend = service.backend("ibm_kingston")

print(backend.name)

ibm_kingston


In [44]:
for i, row in hardware_validation_df.iterrows():

    circuit = real_kingston_circuits[i]

    stored_depth = int(row["transpiled_depth"])
    stored_2q = int(row["transpiled_2q_gates"])

    actual_2q = sum(
        len(inst.qubits) == 2
        for inst in circuit.data
    )

    print(
        i,
        "depth:", circuit.depth(), stored_depth,
        "| 2Q:", actual_2q, stored_2q
    )

    assert circuit.depth() == stored_depth
    assert actual_2q == stored_2q

print("ALL CIRCUITS MATCH.")

0 depth: 126 126 | 2Q: 44 44
1 depth: 103 103 | 2Q: 36 36
2 depth: 126 126 | 2Q: 44 44
3 depth: 124 124 | 2Q: 44 44
4 depth: 46 46 | 2Q: 8 8
5 depth: 46 46 | 2Q: 8 8
6 depth: 46 46 | 2Q: 8 8
7 depth: 46 46 | 2Q: 8 8
8 depth: 234 234 | 2Q: 91 91
9 depth: 234 234 | 2Q: 91 91
10 depth: 241 241 | 2Q: 91 91
11 depth: 241 241 | 2Q: 91 91
12 depth: 68 68 | 2Q: 16 16
13 depth: 68 68 | 2Q: 16 16
14 depth: 68 68 | 2Q: 16 16
15 depth: 68 68 | 2Q: 16 16
16 depth: 66 66 | 2Q: 16 16
17 depth: 58 58 | 2Q: 12 12
18 depth: 345 345 | 2Q: 138 138
19 depth: 354 354 | 2Q: 138 138
20 depth: 345 345 | 2Q: 138 138
21 depth: 345 345 | 2Q: 138 138
22 depth: 90 90 | 2Q: 24 24
23 depth: 88 88 | 2Q: 24 24
24 depth: 90 90 | 2Q: 24 24
25 depth: 90 90 | 2Q: 24 24
26 depth: 126 126 | 2Q: 44 44
27 depth: 126 126 | 2Q: 44 44
28 depth: 44 44 | 2Q: 8 8
29 depth: 46 46 | 2Q: 8 8
30 depth: 241 241 | 2Q: 91 91
31 depth: 241 241 | 2Q: 91 91
32 depth: 66 66 | 2Q: 16 16
33 depth: 68 68 | 2Q: 16 16
34 depth: 345 345 | 2Q: 138 13

In [56]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService(
    channel="ibm_quantum_platform"
)

backend = service.backend(
    "ibm_kingston"
)

sampler = Sampler(
    mode=backend
)

job = sampler.run(
    real_kingston_circuits,
    shots=1024
)

print("Job ID:", job.job_id())

result = job.result()

print("DONE")

Job ID: dad8kkjdd5gc73d70isg


RequestsApiError: 'HTTPSConnectionPool(host=\'quantum.cloud.ibm.com\', port=443): Max retries exceeded with url: /api/v1/jobs/dad8kkjdd5gc73d70isg?exclude_params=true (Caused by NameResolutionError("HTTPSConnection(host=\'quantum.cloud.ibm.com\', port=443): Failed to resolve \'quantum.cloud.ibm.com\' ([Errno 11001] getaddrinfo failed)"))'

In [82]:

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform"
)

job = service.job(
    "dad8kkjdd5gc73d70isg"
)

print("Status:", job.status())

Status: DONE


In [83]:
result = job.result()

print("Result retrieved successfully.")

Result retrieved successfully.


In [84]:
print(result)
print("Number of PUB results:", len(result))
pub_result = result[0]

print(pub_result)
print(pub_result.data)
print(pub_result.data.meas)

PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=5>)), metadata={'circuit_metadata': {}}), Sampler

In [85]:
print(type(result))
print(len(result))
print(type(result[0]))

<class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>
60
<class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


In [81]:
def counts_to_probs(counts):
    total = sum(counts.values())
    return {
        k: v / total
        for k, v in counts.items()
    }


def tvd(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(
        abs(p.get(k, 0) - q.get(k, 0))
        for k in keys
    )

In [86]:
# Convert PrimitiveResult into a normal Python list
pub_results = list(result)

print("Number of PUB results:", len(pub_results))
print("Number of validation circuits:", len(hardware_validation_df))

if len(pub_results) != len(hardware_validation_df):
    raise ValueError(
        f"Mismatch: {len(pub_results)} results vs "
        f"{len(hardware_validation_df)} validation circuits"
    )

records = []

for pos, (_, row) in enumerate(hardware_validation_df.iterrows()):

    # Get the corresponding SamplerPubResult by POSITION
    pub_result = pub_results[pos]

    # Extract real hardware counts
    counts = pub_result.data.meas.get_counts()

    # Convert counts to probabilities
    real_probs = counts_to_probs(counts)

    # Calculate TVD against exact ideal distribution
    real_tvd = tvd(
        ideal_distributions[pos],
        real_probs
    )

    records.append({
        "data_row_index": int(row["data_row_index"]),
        "reps": int(row["reps"]),
        "entanglement": row["entanglement"],
        "initial_layout": row["initial_layout"],
        "fake_kingston_tvd": row["mean_tvd"],
        "real_kingston_tvd": real_tvd
    })

new_results = pd.DataFrame(records)

print(new_results.head())
print("\nShape:", new_results.shape)

Number of PUB results: 60
Number of validation circuits: 60
   data_row_index  reps entanglement             initial_layout  \
0              64     1         full       [10, 11, 12, 13, 14]   
1             143     1       linear  [147, 148, 149, 150, 151]   
2              21     1         full       [61, 62, 63, 64, 65]   
3              17     1       linear       [61, 62, 63, 64, 65]   
4              99     1       linear       [83, 84, 85, 86, 87]   

   fake_kingston_tvd  real_kingston_tvd  
0           0.063672           0.107422  
1           0.065430           0.095703  
2           0.068164           0.146484  
3           0.067383           0.091797  
4           0.067773           0.097656  

Shape: (60, 6)


In [88]:
print(new_results[[
    "data_row_index",
    "reps",
    "entanglement",
    "initial_layout",
    "fake_kingston_tvd",
    "real_kingston_tvd"
]].to_string(index=False))

 data_row_index  reps entanglement            initial_layout  fake_kingston_tvd  real_kingston_tvd
             64     1         full      [10, 11, 12, 13, 14]           0.063672           0.107422
            143     1       linear [147, 148, 149, 150, 151]           0.065430           0.095703
             21     1         full      [61, 62, 63, 64, 65]           0.068164           0.146484
             17     1       linear      [61, 62, 63, 64, 65]           0.067383           0.091797
             99     1       linear      [83, 84, 85, 86, 87]           0.067773           0.097656
            195     3       linear [147, 148, 149, 150, 151]           0.067330           0.076172
            141     1         full      [83, 84, 85, 86, 87]           0.062305           0.073242
             30     2       linear      [10, 11, 12, 13, 14]           0.069952           0.079102
            138     1       linear      [83, 84, 85, 86, 87]           0.065820           0.155694
          

In [89]:
print("Fake TVD range :", fake.min(), fake.max())
print("Real TVD range :", real.min(), real.max())

print("\nFirst 10 paired values:")
for i in range(10):
    print(
        i,
        "fake =", fake[i],
        "real =", real[i]
    )

Fake TVD range : 0.059765625 0.4850778876371404
Real TVD range : 0.05465493185087716 0.46597408156195425

First 10 paired values:
0 fake = 0.063671875 real = 0.10742187500000008
1 fake = 0.0654296875 real = 0.09570312500000003
2 fake = 0.0681640625 real = 0.1464843749999998
3 fake = 0.0673828125 real = 0.09179687500000008
4 fake = 0.0677734375 real = 0.09765625000000008
5 fake = 0.0673297153660059 real = 0.07617187499999997
6 fake = 0.0623046875 real = 0.07324218750000001
7 fake = 0.0699522377851725 real = 0.07910156249999999
8 fake = 0.0658203125 real = 0.1556943829857625
9 fake = 0.059765625 real = 0.19104372977782053


In [91]:
for (reps, entanglement), g in new_results.groupby(
    ["reps", "entanglement"]
):

    fake = g["fake_kingston_tvd"]
    real = g["real_kingston_tvd"]

    print("\nConfig:", reps, entanglement)
    print("N       :", len(g))
    print("Pearson :", fake.corr(real, method="pearson"))
    print("Spearman:", fake.corr(real, method="spearman"))
    print("MAE     :", np.mean(np.abs(fake - real)))
    print("RMSE    :", np.sqrt(np.mean((fake - real) ** 2)))


Config: 1 full
N       : 9
Pearson : 0.2463327931547971
Spearman: 0.03333333333333333
MAE     : 0.06509607121752245
RMSE    : 0.08570821249283389

Config: 1 linear
N       : 13
Pearson : -0.06964699825116454
Spearman: -0.005494505494505494
MAE     : 0.07775387581313005
RMSE    : 0.09246759407016436

Config: 2 full
N       : 11
Pearson : -0.00375099428074664
Spearman: 0.10000000000000002
MAE     : 0.08986440321301431
RMSE    : 0.11702393280639044

Config: 2 linear
N       : 9
Pearson : 0.6588934328032786
Spearman: 0.7
MAE     : 0.017164785898138052
RMSE    : 0.02772434077691038

Config: 3 full
N       : 10
Pearson : 0.1304237319904158
Spearman: 0.2606060606060606
MAE     : 0.1202266623526986
RMSE    : 0.1635402149890825

Config: 3 linear
N       : 8
Pearson : 0.15104768334895335
Spearman: -0.04761904761904763
MAE     : 0.11637439722335596
RMSE    : 0.1314345655617938


In [90]:
for layout, g in new_results.groupby("initial_layout"):

    fake = g["fake_kingston_tvd"]
    real = g["real_kingston_tvd"]

    print("\nLayout:", layout)
    print("N       :", len(g))
    print("Pearson :", fake.corr(real, method="pearson"))
    print("Spearman:", fake.corr(real, method="spearman"))
    print("MAE     :", np.mean(np.abs(fake - real)))
    print("RMSE    :", np.sqrt(np.mean((fake - real) ** 2)))


Layout: [10, 11, 12, 13, 14]
N       : 13
Pearson : -0.030126005626523827
Spearman: 0.11538461538461539
MAE     : 0.0688383468432197
RMSE    : 0.09638943481141818

Layout: [147, 148, 149, 150, 151]
N       : 8
Pearson : 0.5350864474354562
Spearman: 0.4761904761904762
MAE     : 0.0943552986336264
RMSE    : 0.13596285151217818

Layout: [61, 62, 63, 64, 65]
N       : 16
Pearson : 0.03597375924895079
Spearman: -0.0411764705882353
MAE     : 0.06666560456238037
RMSE    : 0.08386673049197264

Layout: [83, 84, 85, 86, 87]
N       : 23
Pearson : 0.4276488221917043
Spearman: 0.3735177865612649
MAE     : 0.09376207674955102
RMSE    : 0.12498207078919651


In [87]:
fake = new_results["fake_kingston_tvd"].to_numpy()
real = new_results["real_kingston_tvd"].to_numpy()

mae = np.mean(np.abs(fake - real))
rmse = np.sqrt(np.mean((fake - real) ** 2))
pearson = np.corrcoef(fake, real)[0, 1]

spearman = (
    pd.Series(fake)
    .rank()
    .corr(
        pd.Series(real).rank()
    )
)

print("N        :", len(new_results))
print("MAE      :", mae)
print("RMSE     :", rmse)
print("Pearson  :", pearson)
print("Spearman :", spearman)

N        : 60
MAE      : 0.08121530560447711
RMSE     : 0.1110914898450082
Pearson  : 0.2512024016404896
Spearman : 0.20011114198388444


In [ ]:
for layout, g in new_results.groupby("initial_layout"):

    fake_l = g["fake_kingston_tvd"].to_numpy()
    real_l = g["real_kingston_tvd"].to_numpy()

    print("\nLayout:", layout)
    print("N:", len(g))
    print(
        "MAE:",
        np.mean(np.abs(fake_l - real_l))
    )
    print(
        "RMSE:",
        np.sqrt(np.mean((fake_l - real_l) ** 2))
    )

    if len(g) > 1:
        print(
            "Pearson:",
            np.corrcoef(fake_l, real_l)[0, 1]
        )

In [ ]:
new_results.to_csv(
    "../experiments/results/"
    "kingston_hardware_validation_extended_v2.csv",
    index=False
)

print("Saved.")

In [93]:
# ============================================================
# MATCHED-LAYOUT VALIDATION: FakeKingston → Real Kingston
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 1. Identify the feature-angle columns in the Kingston dataset
# ------------------------------------------------------------

angle_cols = [f"angle_{i}" for i in range(1, 6)]

print("Angle columns found:", angle_cols)

# Use the actual Kingston dataset containing the 4800 experiments
# Change ONLY this line if your dataframe has a different name.
dataset = kingston_df


# ------------------------------------------------------------
# 2. Build matched-layout validation circuits
# ------------------------------------------------------------

real_kingston_circuits = []
ideal_distributions = []

for _, row in hardware_validation_df.iterrows():

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    # --------------------------------------------------------
    # Recover EXACT encoded angles used for this dataset row
    # --------------------------------------------------------

    data_row = np.array(
        [row[col] for col in angle_cols],
        dtype=float
    )

    # --------------------------------------------------------
    # Same logical circuit as dataset
    # --------------------------------------------------------

    circuit = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement
    )

    circuit = circuit.assign_parameters(data_row)

    # --------------------------------------------------------
    # Exact ideal distribution
    # --------------------------------------------------------

    statevector = Statevector.from_instruction(circuit)

    ideal_distributions.append(
        statevector.probabilities_dict()
    )

    # --------------------------------------------------------
    # Add measurements
    # --------------------------------------------------------

    measured_circuit = circuit.copy()
    measured_circuit.measure_all()

    # --------------------------------------------------------
    # EXACT SAME PHYSICAL LAYOUT AS DATASET
    # --------------------------------------------------------

    layout = row["initial_layout"]

    if isinstance(layout, str):
        layout = eval(layout)

    layout = list(layout)

    # --------------------------------------------------------
    # Transpile on REAL Kingston using SAME layout
    # --------------------------------------------------------

    real_circuit = transpile(
        measured_circuit,
        backend=backend,
        initial_layout=layout,
        optimization_level=1,
        seed_transpiler=42
    )

    real_kingston_circuits.append(real_circuit)


print("\nBuilt matched-layout circuits:", len(real_kingston_circuits))


# ------------------------------------------------------------
# 3. Submit to REAL KINGSTON
# ------------------------------------------------------------

sampler = Sampler(mode=backend)

job = sampler.run(
    real_kingston_circuits,
    shots=1024
)

print("Job ID:", job.job_id())

result = job.result()

print("REAL KINGSTON JOB DONE")


# ------------------------------------------------------------
# 4. Calculate REAL Kingston TVD
# ------------------------------------------------------------

pub_results = list(result)

records = []

for pos, (_, row) in enumerate(
    hardware_validation_df.iterrows()
):

    counts = pub_results[pos].data.meas.get_counts()

    real_probs = counts_to_probs(counts)

    real_tvd = tvd(
        ideal_distributions[pos],
        real_probs
    )

    records.append({
        "validation_index": pos,
        "data_row_index": int(row["data_row_index"]),
        "reps": int(row["reps"]),
        "entanglement": row["entanglement"],
        "initial_layout": row["initial_layout"],
        "fake_kingston_tvd": float(row["mean_tvd"]),
        "real_kingston_tvd": float(real_tvd)
    })


matched_validation_df = pd.DataFrame(records)


# ------------------------------------------------------------
# 5. Overall validation
# ------------------------------------------------------------

fake = matched_validation_df["fake_kingston_tvd"]
real = matched_validation_df["real_kingston_tvd"]

pearson_r, pearson_p = pearsonr(fake, real)
spearman_r, spearman_p = spearmanr(fake, real)

mae = mean_absolute_error(real, fake)

rmse = np.sqrt(
    mean_squared_error(real, fake)
)

print("\n" + "=" * 65)
print("MATCHED-LAYOUT FAKEKINGSTON → REAL KINGSTON VALIDATION")
print("=" * 65)

print(f"N:           {len(matched_validation_df)}")
print(f"Pearson r:   {pearson_r:.4f}")
print(f"Pearson p:   {pearson_p:.4f}")
print(f"Spearman ρ:  {spearman_r:.4f}")
print(f"Spearman p:  {spearman_p:.4f}")
print(f"MAE:         {mae:.4f}")
print(f"RMSE:        {rmse:.4f}")

print("=" * 65)


# ------------------------------------------------------------
# 6. Layout-wise validation
# ------------------------------------------------------------

print("\nLAYOUT-WISE VALIDATION")
print("=" * 70)

for layout, group in matched_validation_df.groupby(
    "initial_layout"
):

    if len(group) < 3:
        continue

    r, p = pearsonr(
        group["fake_kingston_tvd"],
        group["real_kingston_tvd"]
    )

    rho, rho_p = spearmanr(
        group["fake_kingston_tvd"],
        group["real_kingston_tvd"]
    )

    group_mae = mean_absolute_error(
        group["real_kingston_tvd"],
        group["fake_kingston_tvd"]
    )

    group_rmse = np.sqrt(
        mean_squared_error(
            group["real_kingston_tvd"],
            group["fake_kingston_tvd"]
        )
    )

    print(f"\nLayout: {layout}")
    print(f"N:        {len(group)}")
    print(f"Pearson:  {r:.4f}  (p={p:.4f})")
    print(f"Spearman: {rho:.4f}  (p={rho_p:.4f})")
    print(f"MAE:      {group_mae:.4f}")
    print(f"RMSE:     {group_rmse:.4f}")


# ------------------------------------------------------------
# 7. Configuration-wise validation
# ------------------------------------------------------------

print("\nCONFIGURATION-WISE VALIDATION")
print("=" * 70)

for (reps, entanglement), group in matched_validation_df.groupby(
    ["reps", "entanglement"]
):

    if len(group) < 3:
        continue

    r, p = pearsonr(
        group["fake_kingston_tvd"],
        group["real_kingston_tvd"]
    )

    rho, rho_p = spearmanr(
        group["fake_kingston_tvd"],
        group["real_kingston_tvd"]
    )

    group_mae = mean_absolute_error(
        group["real_kingston_tvd"],
        group["fake_kingston_tvd"]
    )

    group_rmse = np.sqrt(
        mean_squared_error(
            group["real_kingston_tvd"],
            group["fake_kingston_tvd"]
        )
    )

    print(
        f"\nConfig: reps={reps}, "
        f"entanglement={entanglement}"
    )
    print(f"N:        {len(group)}")
    print(f"Pearson:  {r:.4f}  (p={p:.4f})")
    print(f"Spearman: {rho:.4f}  (p={rho_p:.4f})")
    print(f"MAE:      {group_mae:.4f}")
    print(f"RMSE:     {group_rmse:.4f}")


# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

matched_validation_df.to_csv(
    "matched_layout_real_kingston_validation.csv",
    index=False
)

print(
    "\nSaved: matched_layout_real_kingston_validation.csv"
)

Angle columns found: ['angle_1', 'angle_2', 'angle_3', 'angle_4', 'angle_5']

Built matched-layout circuits: 60
Job ID: dae2ljjdd5gc73d81es0
REAL KINGSTON JOB DONE

MATCHED-LAYOUT FAKEKINGSTON → REAL KINGSTON VALIDATION
N:           60
Pearson r:   0.8805
Pearson p:   0.0000
Spearman ρ:  0.7857
Spearman p:  0.0000
MAE:         0.0603
RMSE:        0.0791

LAYOUT-WISE VALIDATION

Layout: [10, 11, 12, 13, 14]
N:        13
Pearson:  0.8974  (p=0.0000)
Spearman: 0.7912  (p=0.0013)
MAE:      0.0390
RMSE:     0.0472

Layout: [147, 148, 149, 150, 151]
N:        8
Pearson:  0.8736  (p=0.0046)
Spearman: 0.7857  (p=0.0208)
MAE:      0.0711
RMSE:     0.0799

Layout: [61, 62, 63, 64, 65]
N:        16
Pearson:  0.8963  (p=0.0000)
Spearman: 0.8618  (p=0.0000)
MAE:      0.0239
RMSE:     0.0316

Layout: [83, 84, 85, 86, 87]
N:        23
Pearson:  0.8937  (p=0.0000)
Spearman: 0.9051  (p=0.0000)
MAE:      0.0939
RMSE:     0.1103

CONFIGURATION-WISE VALIDATION

Config: reps=1, entanglement=full
N:        

In [4]:
import os
import ast
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from qiskit import transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import zz_feature_map

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

from qiskit_ibm_runtime.fake_provider import FakeMarrakesh

In [58]:
def counts_to_probabilities(counts):
    total = sum(counts.values())
    return {
        state: count / total
        for state, count in counts.items()
    }


def calculate_tvd_from_probs(p, q):
    keys = set(p.keys()) | set(q.keys())

    return 0.5 * sum(
        abs(p.get(k, 0.0) - q.get(k, 0.0))
        for k in keys
    )

In [59]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    backend,
    noisy_simulator,
    shots=1024,
    repeats=5,
    initial_layout=None,
    seed_transpiler=42,
):

    # =========================================================
    # 1. Build logical circuit
    # =========================================================

    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(data_row)

    # =========================================================
    # 2. Exact ideal distribution
    # =========================================================

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    # =========================================================
    # 3. Add measurements
    # =========================================================

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # =========================================================
    # 4. Transpile for FakeMarrakesh
    # =========================================================

    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # =========================================================
    # 5. Abstract circuit features
    # =========================================================

    abstract_total_gates = len(
        abstract_circuit.data
    )

    abstract_1q = 0
    abstract_2q = 0
    abstract_cx = 0

    qubit_gate_counts = {
        i: 0
        for i in range(abstract_circuit.num_qubits)
    }

    two_qubit_pairs = []

    for inst in abstract_circuit.data:

        nq = len(inst.qubits)

        if nq == 1:
            abstract_1q += 1

        elif nq == 2:
            abstract_2q += 1

            q1 = abstract_circuit.find_bit(
                inst.qubits[0]
            ).index

            q2 = abstract_circuit.find_bit(
                inst.qubits[1]
            ).index

            two_qubit_pairs.append(
                tuple(sorted((q1, q2)))
            )

            if inst.operation.name == "cx":
                abstract_cx += 1

        for qubit in inst.qubits:
            q_idx = abstract_circuit.find_bit(
                qubit
            ).index

            qubit_gate_counts[q_idx] += 1

    unique_2q_pairs = set(two_qubit_pairs)

    if two_qubit_pairs:
        pair_counts = pd.Series(
            two_qubit_pairs
        ).value_counts()

        repeated_pairs = int(
            (pair_counts > 1).sum()
        )
    else:
        repeated_pairs = 0

    degrees = {
        q: 0
        for q in range(
            abstract_circuit.num_qubits
        )
    }

    for q1, q2 in unique_2q_pairs:
        degrees[q1] += 1
        degrees[q2] += 1

    # =========================================================
    # 6. Transpiled gate features
    # =========================================================

    transpiled_total = len(
        transpiled_circuit.data
    )

    transpiled_1q = 0
    transpiled_2q = 0
    transpiled_cz = 0
    transpiled_cx = 0
    transpiled_sx = 0
    transpiled_rz = 0

    for inst in transpiled_circuit.data:

        nq = len(inst.qubits)

        if nq == 1:
            transpiled_1q += 1

        elif nq == 2:
            transpiled_2q += 1

        name = inst.operation.name

        if name == "cz":
            transpiled_cz += 1

        elif name == "cx":
            transpiled_cx += 1

        elif name == "sx":
            transpiled_sx += 1

        elif name == "rz":
            transpiled_rz += 1

    # =========================================================
    # 7. Determine physical qubits used
    # =========================================================

    physical_qubits_used = sorted({
        transpiled_circuit.find_bit(
            qubit
        ).index
        for inst in transpiled_circuit.data
        for qubit in inst.qubits
    })

    # =========================================================
    # 8. Extract CZ calibration information
    # =========================================================

    cz_errors = []
    cz_durations = []

    used_pairs = []

    for inst in transpiled_circuit.data:

        if inst.operation.name != "cz":
            continue

        if len(inst.qubits) != 2:
            continue

        q1 = transpiled_circuit.find_bit(
            inst.qubits[0]
        ).index

        q2 = transpiled_circuit.find_bit(
            inst.qubits[1]
        ).index

        used_pairs.append(
            (q1, q2)
        )

        props = None

        try:
            props = backend.target["cz"][
                (q1, q2)
            ]
        except Exception:
            pass

        if props is None:
            try:
                props = backend.target["cz"][
                    (q2, q1)
                ]
            except Exception:
                pass

        if props is None:
            continue

        if props.error is not None:
            cz_errors.append(
                float(props.error)
            )

        if props.duration is not None:
            cz_durations.append(
                float(props.duration)
            )

    # =========================================================
    # 9. T1 / T2 for physical qubits used
    # =========================================================

    t1_values = []
    t2_values = []

    for q in physical_qubits_used:

        try:
            q_props = backend.qubit_properties(q)
        except Exception:
            q_props = None

        if q_props is None:
            continue

        if q_props.t1 is not None:
            t1_values.append(
                float(q_props.t1)
            )

        if q_props.t2 is not None:
            t2_values.append(
                float(q_props.t2)
            )

    # =========================================================
    # 10. Repeated noisy execution in ONE Aer job
    # =========================================================

    repeated_circuits = [
        transpiled_circuit
        for _ in range(repeats)
    ]

    noisy_result = noisy_simulator.run(
        repeated_circuits,
        shots=shots
    ).result()

    tvds = []

    for repeat_idx in range(repeats):

        counts = noisy_result.get_counts(
            repeat_idx
        )

        noisy_probs = counts_to_probabilities(
            counts
        )

        current_tvd = calculate_tvd_from_probs(
            ideal_probs,
            noisy_probs
        )

        tvds.append(
            float(current_tvd)
        )

    # =========================================================
    # 11. Build output row
    # =========================================================

    result = {}

    # Raw input features
    for i, value in enumerate(
        raw_data_row,
        start=1
    ):
        result[
            f"raw_feature_{i}"
        ] = float(value)

    # Encoded angles
    for i, value in enumerate(
        data_row,
        start=1
    ):
        result[
            f"angle_{i}"
        ] = float(value)

    # =========================================================
    # Abstract features
    # =========================================================

    result.update({

        "abstract_n_qubits":
            abstract_circuit.num_qubits,

        "abstract_depth":
            abstract_circuit.depth(),

        "abstract_total_gate_count":
            abstract_total_gates,

        "abstract_single_qubit_gate_count":
            abstract_1q,

        "abstract_two_qubit_gate_count":
            abstract_2q,

        "abstract_cx_count":
            abstract_cx,

        "abstract_two_qubit_gate_ratio":
            abstract_2q /
            max(abstract_total_gates, 1),

        "abstract_average_gates_per_qubit":
            abstract_total_gates /
            abstract_circuit.num_qubits,

        "abstract_max_gates_per_qubit":
            max(
                qubit_gate_counts.values()
            ),

        "abstract_unique_2q_pairs":
            len(unique_2q_pairs),

        "abstract_average_qubit_degree":
            np.mean(
                list(degrees.values())
            ),

        "abstract_max_qubit_degree":
            max(degrees.values()),

        "abstract_repeated_2q_pairs":
            repeated_pairs,

        "reps":
            reps,

        "entanglement":
            entanglement,
    })

    # =========================================================
    # Transpiled features
    # =========================================================

    result.update({

        "transpiled_depth":
            transpiled_circuit.depth(),

        "transpiled_total_gates":
            transpiled_total,

        "transpiled_1q_gates":
            transpiled_1q,

        "transpiled_2q_gates":
            transpiled_2q,

        "transpiled_cz_count":
            transpiled_cz,

        "transpiled_cx_count":
            transpiled_cx,

        "transpiled_sx_count":
            transpiled_sx,

        "transpiled_rz_count":
            transpiled_rz,

        "backend_size":
            backend.num_qubits,
    })

    # =========================================================
    # CZ calibration
    # =========================================================

    result.update({

        "mean_cz_error":
            np.mean(cz_errors)
            if cz_errors else np.nan,

        "std_cz_error":
            np.std(cz_errors)
            if cz_errors else np.nan,

        "min_cz_error":
            np.min(cz_errors)
            if cz_errors else np.nan,

        "max_cz_error":
            np.max(cz_errors)
            if cz_errors else np.nan,

        "sum_cz_error":
            np.sum(cz_errors)
            if cz_errors else 0.0,

        "cumulative_cz_error":
            1.0 - np.prod(
                [1.0 - e for e in cz_errors]
            )
            if cz_errors else 0.0,

        "mean_cz_duration":
            np.mean(cz_durations)
            if cz_durations else np.nan,

        "max_cz_duration":
            np.max(cz_durations)
            if cz_durations else np.nan,

        "total_cz_duration":
            np.sum(cz_durations)
            if cz_durations else 0.0,
    })

    # =========================================================
    # Physical qubits
    # =========================================================

    result.update({

        "physical_qubit_count":
            len(physical_qubits_used),

        "physical_qubits_used":
            str(physical_qubits_used),
    })

    # =========================================================
    # T1
    # =========================================================

    result.update({

        "mean_t1":
            np.mean(t1_values)
            if t1_values else np.nan,

        "min_t1":
            np.min(t1_values)
            if t1_values else np.nan,

        "max_t1":
            np.max(t1_values)
            if t1_values else np.nan,
    })

    # =========================================================
    # T2
    # =========================================================

    result.update({

        "mean_t2":
            np.mean(t2_values)
            if t2_values else np.nan,

        "min_t2":
            np.min(t2_values)
            if t2_values else np.nan,

        "max_t2":
            np.max(t2_values)
            if t2_values else np.nan,
    })

    # =========================================================
    # Target
    # =========================================================

    result.update({

        "mean_tvd":
            np.mean(tvds),

        "std_tvd":
            np.std(tvds, ddof=1),

        "min_tvd":
            np.min(tvds),

        "max_tvd":
            np.max(tvds),

        "shots":
            shots,

        "repeats":
            repeats,

        "seed_transpiler":
            seed_transpiler,

        "initial_layout":
            str(initial_layout),
    })

    return result

In [61]:
SOURCE_PATH = "../data/raw/blood_cell_anomaly_detection.csv"

df = pd.read_csv(SOURCE_PATH)

selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

df_with_features = df[
    selected_columns + ["cell_type"]
].dropna().reset_index(drop=True)

print("Cleaned dataset shape:", df_with_features.shape)
print(df_with_features.head())

Cleaned dataset shape: (5880, 6)
   cell_diameter_um  chromatin_density  cytoplasm_ratio  circularity  \
0             15.18              0.542            0.301        0.563   
1             16.47              0.583            0.365        0.859   
2             13.41              0.448            0.376        0.781   
3              7.36              0.000            1.000        0.880   
4              7.53              0.000            1.000        1.000   

   eccentricity                  cell_type  
0         0.529  Hypersegmented_Neutrophil  
1         0.443  Hypersegmented_Neutrophil  
2         0.407                 Neutrophil  
3         0.167                 Normal_RBC  
4         0.158                 Normal_RBC  


In [63]:
# ---------------------------------------------------------
# Recreate the exact Kingston 200-row sample
# ---------------------------------------------------------

kingston_df, remaining_df = train_test_split(
    df_with_features,
    train_size=200,
    stratify=df_with_features["cell_type"],
    random_state=42
)

# ---------------------------------------------------------
# Take Marrakesh from the remaining rows
# ---------------------------------------------------------

marrakesh_df, _ = train_test_split(
    remaining_df,
    train_size=200,
    stratify=remaining_df["cell_type"],
    random_state=42
)

marrakesh_df = marrakesh_df.reset_index(drop=True)

print("Kingston rows :", len(kingston_df))
print("Marrakesh rows:", len(marrakesh_df))

Kingston rows : 200
Marrakesh rows: 200


In [64]:
scaler = MinMaxScaler(
    feature_range=(0, 2 * np.pi)
)

# Fit on the FULL cleaned dataset
scaler.fit(
    df_with_features[selected_columns]
)

marrakesh_raw = (
    marrakesh_df[selected_columns]
    .values
)

marrakesh_scaled = scaler.transform(
    marrakesh_df[selected_columns]
)

print("Raw shape   :", marrakesh_raw.shape)
print("Scaled shape:", marrakesh_scaled.shape)
print(
    "Scaled range:",
    marrakesh_scaled.min(),
    "to",
    marrakesh_scaled.max()
)

Raw shape   : (200, 5)
Scaled shape: (200, 5)
Scaled range: 0.0 to 6.283185307179586


In [65]:
fake_marrakesh = FakeMarrakesh()

marrakesh_noise_model = NoiseModel.from_backend(
    fake_marrakesh
)

marrakesh_noisy_simulator = AerSimulator(
    noise_model=marrakesh_noise_model
)

print("Backend:", fake_marrakesh.name)
print(
    "Number of qubits:",
    fake_marrakesh.num_qubits
)

Backend: fake_marrakesh
Number of qubits: 156


In [68]:
CONFIGS = [
    (1, "linear"),
    (1, "full"),
    (2, "linear"),
    (2, "full"),
    (3, "linear"),
    (3, "full"),
]

# ============================================================
# SELECT 4 MARRAKESH LAYOUTS BY CALIBRATION QUALITY
# ============================================================

coupling_edges = {
    tuple(sorted(edge))
    for edge in fake_marrakesh.coupling_map.get_edges()
}

valid_marrakesh_layouts = []

# Find connected 5-qubit chains
for start in range(fake_marrakesh.num_qubits - 4):

    layout = list(range(start, start + 5))

    connected = all(
        tuple(sorted((layout[i], layout[i + 1])))
        in coupling_edges
        for i in range(4)
    )

    if not connected:
        continue

    errors = []

    for i in range(4):

        q1 = layout[i]
        q2 = layout[i + 1]

        props = None

        try:
            props = fake_marrakesh.target["cz"][(q1, q2)]
        except Exception:
            pass

        if props is None:
            try:
                props = fake_marrakesh.target["cz"][(q2, q1)]
            except Exception:
                pass

        if props is None or props.error is None:
            errors = None
            break

        errors.append(float(props.error))

    if errors is None:
        continue

    valid_marrakesh_layouts.append({
        "layout": layout,
        "mean_error": np.mean(errors),
        "max_error": np.max(errors),
        "errors": errors
    })


# Sort by mean CZ error
valid_marrakesh_layouts = sorted(
    valid_marrakesh_layouts,
    key=lambda x: x["mean_error"]
)

print(
    "Valid Marrakesh layouts:",
    len(valid_marrakesh_layouts)
)

# ------------------------------------------------------------
# Remove layouts with catastrophic CZ error
# Same criterion as Kingston
# ------------------------------------------------------------

usable_marrakesh = [
    x
    for x in valid_marrakesh_layouts
    if x["max_error"] < 0.10
]

print(
    "Usable Marrakesh layouts:",
    len(usable_marrakesh)
)

if len(usable_marrakesh) < 4:
    raise RuntimeError(
        "Fewer than 4 usable Marrakesh layouts found."
    )


# ------------------------------------------------------------
# Select 4 layouts spread across calibration quality
# ------------------------------------------------------------

indices = np.linspace(
    0,
    len(usable_marrakesh) - 1,
    4,
    dtype=int
)

MARRAKESH_LAYOUTS = [
    usable_marrakesh[i]["layout"]
    for i in indices
]


# ------------------------------------------------------------
# Print selected layouts
# ------------------------------------------------------------

print("\nSelected Marrakesh layouts:\n")

for layout in MARRAKESH_LAYOUTS:

    item = next(
        x
        for x in usable_marrakesh
        if x["layout"] == layout
    )

    print(
        layout,
        "| mean=",
        round(item["mean_error"], 6),
        "| max=",
        round(item["max_error"], 6),
        "| edges=",
        [round(x, 6) for x in item["errors"]]
    )

print("Configurations:", len(CONFIGS))
print("Layouts:", len(MARRAKESH_LAYOUTS))

for layout in MARRAKESH_LAYOUTS:
    print(layout)

Valid Marrakesh layouts: 96
Usable Marrakesh layouts: 75

Selected Marrakesh layouts:

[11, 12, 13, 14, 15] | mean= 0.001187 | max= 0.00124 | edges= [0.001231, 0.001239, 0.001036, 0.00124]
[107, 108, 109, 110, 111] | mean= 0.002963 | max= 0.003232 | edges= [0.002779, 0.003201, 0.00264, 0.003232]
[125, 126, 127, 128, 129] | mean= 0.004254 | max= 0.007534 | edges= [0.003759, 0.007534, 0.00241, 0.003313]
[69, 70, 71, 72, 73] | mean= 0.023286 | max= 0.084688 | edges= [0.002658, 0.003041, 0.002757, 0.084688]
Configurations: 6
Layouts: 4
[11, 12, 13, 14, 15]
[107, 108, 109, 110, 111]
[125, 126, 127, 128, 129]
[69, 70, 71, 72, 73]


In [2]:
def run_complete_experiment(
    raw_data_row,
    data_row,
    reps,
    entanglement,
    backend,
    noisy_simulator,
    shots=1024,
    repeats=5,
    initial_layout=None,
    seed_transpiler=42,
):
    """
    Run one complete quantum noise experiment.

    Returns one row containing:
    - raw input features
    - encoded angles
    - abstract circuit features
    - transpiled circuit features
    - calibration features
    - physical qubit features
    - T1/T2 features
    - TVD statistics
    """

    # =========================================================
    # 1. Build logical circuit
    # =========================================================

    feature_map = zz_feature_map(
        feature_dimension=len(data_row),
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(
        data_row
    )

    # =========================================================
    # 2. Exact ideal distribution
    # =========================================================

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    # =========================================================
    # 3. Add measurements
    # =========================================================

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # =========================================================
    # 4. Transpile for selected backend/layout
    # =========================================================

    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=initial_layout,
        seed_transpiler=seed_transpiler,
    )

    # =========================================================
    # 5. Abstract circuit features
    # =========================================================

    abstract_total_gates = len(
        abstract_circuit.data
    )

    abstract_1q = 0
    abstract_2q = 0
    abstract_cx = 0

    qubit_gate_counts = {
        i: 0
        for i in range(abstract_circuit.num_qubits)
    }

    two_qubit_pairs = []

    for inst in abstract_circuit.data:

        nq = len(inst.qubits)

        if nq == 1:
            abstract_1q += 1

        elif nq == 2:
            abstract_2q += 1

            q1 = abstract_circuit.find_bit(
                inst.qubits[0]
            ).index

            q2 = abstract_circuit.find_bit(
                inst.qubits[1]
            ).index

            two_qubit_pairs.append(
                tuple(sorted((q1, q2)))
            )

            if inst.operation.name == "cx":
                abstract_cx += 1

        for qubit in inst.qubits:

            q_idx = abstract_circuit.find_bit(
                qubit
            ).index

            qubit_gate_counts[q_idx] += 1

    unique_2q_pairs = set(two_qubit_pairs)

    pair_counts = pd.Series(
        two_qubit_pairs
    ).value_counts()

    repeated_pairs = int(
        (pair_counts > 1).sum()
    )

    degrees = {
        q: 0
        for q in range(
            abstract_circuit.num_qubits
        )
    }

    for q1, q2 in unique_2q_pairs:
        degrees[q1] += 1
        degrees[q2] += 1

    # =========================================================
    # 6. Transpiled gate features
    # =========================================================

    transpiled_total = len(
        transpiled_circuit.data
    )

    transpiled_1q = 0
    transpiled_2q = 0
    transpiled_cz = 0
    transpiled_cx = 0
    transpiled_sx = 0
    transpiled_rz = 0

    for inst in transpiled_circuit.data:

        nq = len(inst.qubits)

        if nq == 1:
            transpiled_1q += 1

        elif nq == 2:
            transpiled_2q += 1

        name = inst.operation.name

        if name == "cz":
            transpiled_cz += 1

        elif name == "cx":
            transpiled_cx += 1

        elif name == "sx":
            transpiled_sx += 1

        elif name == "rz":
            transpiled_rz += 1

    # =========================================================
    # 7. Determine physical qubits used
    # =========================================================

    physical_qubits_used = sorted({
        transpiled_circuit.find_bit(
            qubit
        ).index
        for inst in transpiled_circuit.data
        for qubit in inst.qubits
    })

    # =========================================================
    # 8. Extract CZ calibration information
    #    for physical CZ operations actually present
    # =========================================================

    cz_errors = []
    cz_durations = []

    used_pairs = []

    for inst in transpiled_circuit.data:

        if inst.operation.name != "cz":
            continue

        if len(inst.qubits) != 2:
            continue

        q1 = transpiled_circuit.find_bit(
            inst.qubits[0]
        ).index

        q2 = transpiled_circuit.find_bit(
            inst.qubits[1]
        ).index

        used_pairs.append(
            (q1, q2)
        )

        props = None

        try:
            props = backend.target["cz"][
                (q1, q2)
            ]
        except Exception:
            pass

        if props is None:
            try:
                props = backend.target["cz"][
                    (q2, q1)
                ]
            except Exception:
                pass

        if props is None:
            continue

        if props.error is not None:
            cz_errors.append(
                float(props.error)
            )

        if props.duration is not None:
            cz_durations.append(
                float(props.duration)
            )

    # =========================================================
    # 9. T1 / T2 for physical qubits used
    # =========================================================

    t1_values = []
    t2_values = []

    for q in physical_qubits_used:

        try:
            q_props = backend.qubit_properties(q)
        except Exception:
            q_props = None

        if q_props is None:
            continue

        if q_props.t1 is not None:
            t1_values.append(
                float(q_props.t1)
            )

        if q_props.t2 is not None:
            t2_values.append(
                float(q_props.t2)
            )

    # =========================================================
    # 10. Repeated noisy execution in ONE Aer job
    # =========================================================

    repeated_circuits = [
        transpiled_circuit
        for _ in range(repeats)
    ]

    noisy_result = noisy_simulator.run(
        repeated_circuits,
        shots=shots
    ).result()

    tvds = []

    for repeat_idx in range(repeats):

        counts = noisy_result.get_counts(
            repeat_idx
        )

        noisy_probs = counts_to_probabilities(
            counts
        )

        tvd = calculate_tvd_from_probs(
            ideal_probs,
            noisy_probs
        )

        tvds.append(float(tvd))

    # =========================================================
    # 11. Build output row
    # =========================================================

    result = {}

    # ---------------------------------------------------------
    # Input features
    # ---------------------------------------------------------

    for i, value in enumerate(
        raw_data_row,
        start=1
    ):
        result[
            f"raw_feature_{i}"
        ] = float(value)

    # ---------------------------------------------------------
    # Encoded angles
    # ---------------------------------------------------------

    for i, value in enumerate(
        data_row,
        start=1
    ):
        result[
            f"angle_{i}"
        ] = float(value)

    # ---------------------------------------------------------
    # Abstract circuit features
    # ---------------------------------------------------------

    result.update({

        "abstract_n_qubits":
            abstract_circuit.num_qubits,

        "abstract_depth":
            abstract_circuit.depth(),

        "abstract_total_gate_count":
            abstract_total_gates,

        "abstract_single_qubit_gate_count":
            abstract_1q,

        "abstract_two_qubit_gate_count":
            abstract_2q,

        "abstract_cx_count":
            abstract_cx,

        "abstract_two_qubit_gate_ratio":
            abstract_2q /
            max(abstract_total_gates, 1),

        "abstract_average_gates_per_qubit":
            abstract_total_gates /
            abstract_circuit.num_qubits,

        "abstract_max_gates_per_qubit":
            max(
                qubit_gate_counts.values()
            ),

        "abstract_unique_2q_pairs":
            len(unique_2q_pairs),

        "abstract_average_qubit_degree":
            np.mean(
                list(degrees.values())
            ),

        "abstract_max_qubit_degree":
            max(degrees.values()),

        "abstract_repeated_2q_pairs":
            repeated_pairs,

        "reps":
            reps,

        "entanglement":
            entanglement,
    })

    # ---------------------------------------------------------
    # Transpiled features
    # ---------------------------------------------------------

    result.update({

        "transpiled_depth":
            transpiled_circuit.depth(),

        "transpiled_total_gates":
            transpiled_total,

        "transpiled_1q_gates":
            transpiled_1q,

        "transpiled_2q_gates":
            transpiled_2q,

        "transpiled_cz_count":
            transpiled_cz,

        "transpiled_cx_count":
            transpiled_cx,

        "transpiled_sx_count":
            transpiled_sx,

        "transpiled_rz_count":
            transpiled_rz,

        "backend_size":
            backend.num_qubits,
    })

    # ---------------------------------------------------------
    # CZ calibration
    # ---------------------------------------------------------

    result.update({

        "mean_cz_error":
            np.mean(cz_errors)
            if cz_errors else np.nan,

        "std_cz_error":
            np.std(cz_errors)
            if cz_errors else np.nan,

        "min_cz_error":
            np.min(cz_errors)
            if cz_errors else np.nan,

        "max_cz_error":
            np.max(cz_errors)
            if cz_errors else np.nan,

        "sum_cz_error":
            np.sum(cz_errors)
            if cz_errors else 0.0,

        "cumulative_cz_error":
            1.0 -
            np.prod(
                [1.0 - e for e in cz_errors]
            )
            if cz_errors else 0.0,

        "mean_cz_duration":
            np.mean(cz_durations)
            if cz_durations else np.nan,

        "max_cz_duration":
            np.max(cz_durations)
            if cz_durations else np.nan,

        "total_cz_duration":
            np.sum(cz_durations)
            if cz_durations else 0.0,
    })

    # ---------------------------------------------------------
    # Physical qubits
    # ---------------------------------------------------------

    result.update({

        "physical_qubit_count":
            len(physical_qubits_used),

        "physical_qubits_used":
            str(physical_qubits_used),
    })

    # ---------------------------------------------------------
    # T1
    # ---------------------------------------------------------

    result.update({

        "mean_t1":
            np.mean(t1_values)
            if t1_values else np.nan,

        "min_t1":
            np.min(t1_values)
            if t1_values else np.nan,

        "max_t1":
            np.max(t1_values)
            if t1_values else np.nan,
    })

    # ---------------------------------------------------------
    # T2
    # ---------------------------------------------------------

    result.update({

        "mean_t2":
            np.mean(t2_values)
            if t2_values else np.nan,

        "min_t2":
            np.min(t2_values)
            if t2_values else np.nan,

        "max_t2":
            np.max(t2_values)
            if t2_values else np.nan,
    })

    # ---------------------------------------------------------
    # Target + experiment settings
    # ---------------------------------------------------------

    result.update({

        "mean_tvd":
            np.mean(tvds),

        "std_tvd":
            np.std(
                tvds,
                ddof=1
            ),

        "min_tvd":
            np.min(tvds),

        "max_tvd":
            np.max(tvds),

        "shots":
            shots,

        "repeats":
            repeats,

        "seed_transpiler":
            seed_transpiler,

        "initial_layout":
            str(initial_layout),
    })

    return result

In [71]:
print("Layouts:", len(MARRAKESH_LAYOUTS))

for layout in MARRAKESH_LAYOUTS:
    print(layout)

Layouts: 4
[11, 12, 13, 14, 15]
[107, 108, 109, 110, 111]
[125, 126, 127, 128, 129]
[69, 70, 71, 72, 73]


In [73]:
# ============================================================
# GENERATE MARRAKESH DATASET WITH PROGRESS + ETA
# ============================================================

import time

marrakesh_results = []

total_experiments = (
    len(marrakesh_scaled)
    * len(CONFIGS)
    * len(MARRAKESH_LAYOUTS)
)

print("Starting Marrakesh experiment generation...")
print(f"Total experiments: {total_experiments}")
print(f"Experiments per source row: {len(CONFIGS) * len(MARRAKESH_LAYOUTS)}")
print()

start_time = time.time()

for data_idx in range(len(marrakesh_scaled)):

    raw_row = marrakesh_raw[data_idx]
    scaled_row = marrakesh_scaled[data_idx]

    for reps, entanglement in CONFIGS:

        for layout_id, layout in enumerate(MARRAKESH_LAYOUTS):

            result = run_complete_experiment(
                raw_data_row=raw_row,
                data_row=scaled_row,
                reps=reps,
                entanglement=entanglement,
                backend=fake_marrakesh,
                noisy_simulator=marrakesh_noisy_simulator,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            result["data_row_index"] = data_idx
            result["layout_id"] = layout_id

            marrakesh_results.append(result)

    # --------------------------------------------------------
    # Progress + ETA after every 10 source rows
    # --------------------------------------------------------

    if (data_idx + 1) % 10 == 0:

        completed = len(marrakesh_results)

        elapsed = time.time() - start_time

        rate = completed / elapsed

        remaining = total_experiments - completed

        eta_seconds = remaining / rate

        eta_minutes = eta_seconds / 60

        elapsed_minutes = elapsed / 60

        progress = (
            completed / total_experiments
        ) * 100

        print(
            f"[{progress:6.2f}%] "
            f"{completed}/{total_experiments} experiments | "
            f"Elapsed: {elapsed_minutes:.1f} min | "
            f"Rate: {rate:.2f} exp/s | "
            f"ETA: {eta_minutes:.1f} min"
        )


# ============================================================
# FINAL TIMING
# ============================================================

total_time = time.time() - start_time

print()
print("=" * 60)
print("MARRAKESH GENERATION COMPLETE")
print("=" * 60)

print(
    f"Experiments completed: {len(marrakesh_results)}"
)

print(
    f"Total time: {total_time / 60:.2f} minutes"
)

print(
    f"Average time per experiment: "
    f"{total_time / len(marrakesh_results):.2f} seconds"
)

print("=" * 60)

Starting Marrakesh experiment generation...
Total experiments: 4800
Experiments per source row: 24

[  5.00%] 240/4800 experiments | Elapsed: 23.6 min | Rate: 0.17 exp/s | ETA: 447.5 min
[ 10.00%] 480/4800 experiments | Elapsed: 46.1 min | Rate: 0.17 exp/s | ETA: 415.2 min
[ 15.00%] 720/4800 experiments | Elapsed: 63.3 min | Rate: 0.19 exp/s | ETA: 358.6 min
[ 20.00%] 960/4800 experiments | Elapsed: 80.7 min | Rate: 0.20 exp/s | ETA: 322.7 min
[ 25.00%] 1200/4800 experiments | Elapsed: 99.2 min | Rate: 0.20 exp/s | ETA: 297.7 min
[ 30.00%] 1440/4800 experiments | Elapsed: 118.2 min | Rate: 0.20 exp/s | ETA: 275.9 min
[ 35.00%] 1680/4800 experiments | Elapsed: 136.3 min | Rate: 0.21 exp/s | ETA: 253.1 min
[ 40.00%] 1920/4800 experiments | Elapsed: 154.8 min | Rate: 0.21 exp/s | ETA: 232.3 min
[ 45.00%] 2160/4800 experiments | Elapsed: 173.6 min | Rate: 0.21 exp/s | ETA: 212.2 min
[ 50.00%] 2400/4800 experiments | Elapsed: 191.9 min | Rate: 0.21 exp/s | ETA: 191.9 min
[ 55.00%] 2640/4800

In [74]:
# ============================================================
# CREATE MARRAKESH DATAFRAME
# ============================================================

marrakesh_df = pd.DataFrame(
    marrakesh_results
)

print("\nMarrakesh dataset generated.")
print("Shape:", marrakesh_df.shape)


Marrakesh dataset generated.
Shape: (4800, 61)


In [75]:
# ============================================================
# VALIDATION
# ============================================================

print("\nLayout distribution:")
print(
    marrakesh_df[
        "layout_id"
    ].value_counts().sort_index()
)

print("\nConfiguration distribution:")
print(
    marrakesh_df[
        ["reps", "entanglement"]
    ].value_counts().sort_index()
)

print("\nTVD summary:")
print(
    marrakesh_df[
        "mean_tvd"
    ].describe()
)

print("\nMissing values:")
print(
    marrakesh_df.isna().sum()[
        marrakesh_df.isna().sum() > 0
    ]
)


Layout distribution:
layout_id
0    1200
1    1200
2    1200
3    1200
Name: count, dtype: int64

Configuration distribution:
reps  entanglement
1     full            800
      linear          800
2     full            800
      linear          800
3     full            800
      linear          800
Name: count, dtype: int64

TVD summary:
count    4800.000000
mean        0.130589
std         0.098684
min         0.041936
25%         0.067930
50%         0.076172
75%         0.170236
max         0.692062
Name: mean_tvd, dtype: float64

Missing values:
Series([], dtype: int64)


In [76]:
# ============================================================
# SAVE MARRAKESH DATASET
# ============================================================

MARRAKESH_OUTPUT = (
    "quantum_noise_predictor_dataset_marrakesh_61.csv"
)

marrakesh_df.to_csv(
    MARRAKESH_OUTPUT,
    index=False
)

print(
    "Saved:",
    MARRAKESH_OUTPUT
)

print(
    "Final shape:",
    marrakesh_df.shape
)

Saved: quantum_noise_predictor_dataset_marrakesh_61.csv
Final shape: (4800, 61)


In [77]:
# ============================================================
# SAVE MARRAKESH DATASET METADATA
# ============================================================

import json

marrakesh_metadata = {
    "backend": "FakeMarrakesh",

    "source_rows": len(marrakesh_scaled),

    "configurations": [
        {
            "reps": reps,
            "entanglement": entanglement
        }
        for reps, entanglement in CONFIGS
    ],

    "layouts": MARRAKESH_LAYOUTS,

    "total_experiments": len(marrakesh_df),

    "shots": 1024,

    "repeats": 5,

    "seed_transpiler": 42,

    "source_row_overlap_with_kingston": 0,

    "independent_dataset": True,

    "description": (
        "Independent FakeMarrakesh dataset using "
        "200 source rows disjoint from Kingston. "
        "Four connected 5-qubit layouts were selected "
        "from FakeMarrakesh based on CZ calibration quality."
    )
}


# ============================================================
# SAVE METADATA JSON
# ============================================================

MARRAKESH_METADATA_OUTPUT = (
    "marrakesh_dataset_metadata.json"
)

with open(
    MARRAKESH_METADATA_OUTPUT,
    "w"
) as f:
    json.dump(
        marrakesh_metadata,
        f,
        indent=2
    )


print(
    "Saved metadata:",
    MARRAKESH_METADATA_OUTPUT
)

print(
    json.dumps(
        marrakesh_metadata,
        indent=2
    )
)

Saved metadata: marrakesh_dataset_metadata.json
{
  "backend": "FakeMarrakesh",
  "source_rows": 200,
  "configurations": [
    {
      "reps": 1,
      "entanglement": "linear"
    },
    {
      "reps": 1,
      "entanglement": "full"
    },
    {
      "reps": 2,
      "entanglement": "linear"
    },
    {
      "reps": 2,
      "entanglement": "full"
    },
    {
      "reps": 3,
      "entanglement": "linear"
    },
    {
      "reps": 3,
      "entanglement": "full"
    }
  ],
  "layouts": [
    [
      11,
      12,
      13,
      14,
      15
    ],
    [
      107,
      108,
      109,
      110,
      111
    ],
    [
      125,
      126,
      127,
      128,
      129
    ],
    [
      69,
      70,
      71,
      72,
      73
    ]
  ],
  "total_experiments": 4800,
  "shots": 1024,
  "repeats": 5,
  "seed_transpiler": 42,
  "source_row_overlap_with_kingston": 0,
  "independent_dataset": true,
  "description": "Independent FakeMarrakesh dataset using 200 source rows di

In [94]:
backend = service.backend("ibm_marrakesh")

In [95]:
# ============================================================
# MATCHED-LAYOUT VALIDATION: FakeMarrakesh → REAL MARRAKESH
# 60 CIRCUITS
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 1. Select 60 validation circuits from Marrakesh dataset
# ------------------------------------------------------------

# Use your existing Marrakesh dataframe.
# If your dataframe has a different name, change this line.
marrakesh_validation_df = marrakesh_df.sample(
    n=60,
    random_state=42
).copy()

marrakesh_validation_df = marrakesh_validation_df.reset_index(drop=True)

print("Validation circuits:", len(marrakesh_validation_df))


# ------------------------------------------------------------
# 2. Build SAME logical circuits
#    and use SAME physical layout
# ------------------------------------------------------------

real_marrakesh_circuits = []
ideal_distributions = []

for _, row in marrakesh_validation_df.iterrows():

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    # --------------------------------------------------------
    # Recover EXACT encoded angles from dataset
    # --------------------------------------------------------

    data_row = np.array(
        [row[f"angle_{i}"] for i in range(1, 6)],
        dtype=float
    )

    # --------------------------------------------------------
    # Same logical circuit used to create dataset
    # --------------------------------------------------------

    circuit = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement
    )

    circuit = circuit.assign_parameters(data_row)

    # --------------------------------------------------------
    # Exact ideal distribution
    # --------------------------------------------------------

    statevector = Statevector.from_instruction(circuit)

    ideal_distributions.append(
        statevector.probabilities_dict()
    )

    # --------------------------------------------------------
    # Add measurement
    # --------------------------------------------------------

    measured_circuit = circuit.copy()
    measured_circuit.measure_all()

    # --------------------------------------------------------
    # EXACT SAME PHYSICAL LAYOUT AS DATASET
    # --------------------------------------------------------

    layout = row["initial_layout"]

    if isinstance(layout, str):
        layout = eval(layout)

    layout = list(layout)

    # --------------------------------------------------------
    # Transpile on REAL Marrakesh
    # using SAME physical layout
    # --------------------------------------------------------

    real_circuit = transpile(
        measured_circuit,
        backend=backend,
        initial_layout=layout,
        optimization_level=1,
        seed_transpiler=42
    )

    real_marrakesh_circuits.append(real_circuit)


print("Built matched-layout circuits:",
      len(real_marrakesh_circuits))


# ------------------------------------------------------------
# 3. RUN ON REAL MARRAKESH
# ------------------------------------------------------------

sampler = Sampler(mode=backend)

job = sampler.run(
    real_marrakesh_circuits,
    shots=1024
)

print("Job ID:", job.job_id())

result = job.result()

print("REAL MARRAKESH JOB DONE")


# ------------------------------------------------------------
# 4. Calculate REAL Marrakesh TVD
# ------------------------------------------------------------

pub_results = list(result)

records = []

for pos, (_, row) in enumerate(
    marrakesh_validation_df.iterrows()
):

    counts = pub_results[pos].data.meas.get_counts()

    real_probs = counts_to_probs(counts)

    real_tvd = tvd(
        ideal_distributions[pos],
        real_probs
    )

    records.append({
        "validation_index": pos,

        "data_row_index": int(
            row["data_row_index"]
        ),

        "reps": int(row["reps"]),

        "entanglement":
            row["entanglement"],

        "initial_layout":
            row["initial_layout"],

        # Fake Marrakesh TVD from dataset
        "fake_marrakesh_tvd":
            float(row["mean_tvd"]),

        # Real hardware TVD
        "real_marrakesh_tvd":
            float(real_tvd)
    })


matched_marrakesh_validation_df = pd.DataFrame(records)


# ------------------------------------------------------------
# 5. OVERALL VALIDATION
# ------------------------------------------------------------

fake = matched_marrakesh_validation_df[
    "fake_marrakesh_tvd"
]

real = matched_marrakesh_validation_df[
    "real_marrakesh_tvd"
]

pearson_r, pearson_p = pearsonr(fake, real)

spearman_r, spearman_p = spearmanr(fake, real)

mae = mean_absolute_error(real, fake)

rmse = np.sqrt(
    mean_squared_error(real, fake)
)

print("\n" + "=" * 65)
print("MATCHED-LAYOUT FAKE MARRAKESH → REAL MARRAKESH")
print("=" * 65)

print(
    f"N:           "
    f"{len(matched_marrakesh_validation_df)}"
)

print(f"Pearson r:   {pearson_r:.4f}")
print(f"Pearson p:   {pearson_p:.4f}")

print(f"Spearman ρ:  {spearman_r:.4f}")
print(f"Spearman p:  {spearman_p:.4f}")

print(f"MAE:         {mae:.4f}")
print(f"RMSE:        {rmse:.4f}")

print("=" * 65)


# ------------------------------------------------------------
# 6. LAYOUT-WISE VALIDATION
# ------------------------------------------------------------

print("\nLAYOUT-WISE VALIDATION")
print("=" * 70)

for layout, group in (
    matched_marrakesh_validation_df
    .groupby("initial_layout")
):

    if len(group) < 3:
        continue

    r, p = pearsonr(
        group["fake_marrakesh_tvd"],
        group["real_marrakesh_tvd"]
    )

    rho, rho_p = spearmanr(
        group["fake_marrakesh_tvd"],
        group["real_marrakesh_tvd"]
    )

    group_mae = mean_absolute_error(
        group["real_marrakesh_tvd"],
        group["fake_marrakesh_tvd"]
    )

    group_rmse = np.sqrt(
        mean_squared_error(
            group["real_marrakesh_tvd"],
            group["fake_marrakesh_tvd"]
        )
    )

    print(f"\nLayout: {layout}")
    print(f"N:        {len(group)}")
    print(f"Pearson:  {r:.4f}  (p={p:.4f})")
    print(f"Spearman: {rho:.4f}  (p={rho_p:.4f})")
    print(f"MAE:      {group_mae:.4f}")
    print(f"RMSE:     {group_rmse:.4f}")


# ------------------------------------------------------------
# 7. CONFIGURATION-WISE VALIDATION
# ------------------------------------------------------------

print("\nCONFIGURATION-WISE VALIDATION")
print("=" * 70)

for (reps, entanglement), group in (
    matched_marrakesh_validation_df
    .groupby(["reps", "entanglement"])
):

    if len(group) < 3:
        continue

    r, p = pearsonr(
        group["fake_marrakesh_tvd"],
        group["real_marrakesh_tvd"]
    )

    rho, rho_p = spearmanr(
        group["fake_marrakesh_tvd"],
        group["real_marrakesh_tvd"]
    )

    group_mae = mean_absolute_error(
        group["real_marrakesh_tvd"],
        group["fake_marrakesh_tvd"]
    )

    group_rmse = np.sqrt(
        mean_squared_error(
            group["real_marrakesh_tvd"],
            group["fake_marrakesh_tvd"]
        )
    )

    print(
        f"\nConfig: reps={reps}, "
        f"entanglement={entanglement}"
    )

    print(f"N:        {len(group)}")
    print(f"Pearson:  {r:.4f}  (p={p:.4f})")
    print(f"Spearman: {rho:.4f}  (p={rho_p:.4f})")
    print(f"MAE:      {group_mae:.4f}")
    print(f"RMSE:     {group_rmse:.4f}")


# ------------------------------------------------------------
# 8. SAVE VALIDATION RESULTS
# ------------------------------------------------------------

matched_marrakesh_validation_df.to_csv(
    "matched_layout_real_marrakesh_validation.csv",
    index=False
)

print(
    "\nSaved: "
    "matched_layout_real_marrakesh_validation.csv"
)

Validation circuits: 60
Built matched-layout circuits: 60
Job ID: daeevjbdd5gc73d8h300
REAL MARRAKESH JOB DONE

MATCHED-LAYOUT FAKE MARRAKESH → REAL MARRAKESH
N:           60
Pearson r:   0.5773
Pearson p:   0.0000
Spearman ρ:  0.7151
Spearman p:  0.0000
MAE:         0.0578
RMSE:        0.0776

LAYOUT-WISE VALIDATION

Layout: [107, 108, 109, 110, 111]
N:        16
Pearson:  0.8520  (p=0.0000)
Spearman: 0.7706  (p=0.0005)
MAE:      0.0548
RMSE:     0.0691

Layout: [11, 12, 13, 14, 15]
N:        16
Pearson:  0.9003  (p=0.0000)
Spearman: 0.8118  (p=0.0001)
MAE:      0.0331
RMSE:     0.0445

Layout: [125, 126, 127, 128, 129]
N:        12
Pearson:  0.9582  (p=0.0000)
Spearman: 0.7063  (p=0.0102)
MAE:      0.0304
RMSE:     0.0337

Layout: [69, 70, 71, 72, 73]
N:        16
Pearson:  0.7560  (p=0.0007)
Spearman: 0.6000  (p=0.0140)
MAE:      0.1062
RMSE:     0.1225

CONFIGURATION-WISE VALIDATION

Config: reps=1, entanglement=full
N:        10
Pearson:  -0.4993  (p=0.1417)
Spearman: -0.4667  (p=

In [3]:
# ============================================================
# MARRAKESH MATCHED-LAYOUT REAL-HARDWARE VALIDATION
# Reload saved dataset — no regeneration required
# ============================================================

import ast
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

from qiskit import transpile
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2


# ============================================================
# 1. LOAD SAVED MARRAKESH DATASET
# ============================================================

DATASET_NAME = "quantum_noise_predictor_dataset_marrakesh_61.csv"

matches = list(Path.cwd().rglob(DATASET_NAME))

if not matches:
    raise FileNotFoundError(
        f"Could not find {DATASET_NAME}. "
        "Make sure the generated Marrakesh CSV is in the notebook environment."
    )

MARRAKESH_DATASET_PATH = matches[0]

print("Loading:", MARRAKESH_DATASET_PATH)

marrakesh_df = pd.read_csv(MARRAKESH_DATASET_PATH)

print("Dataset shape:", marrakesh_df.shape)


# ============================================================
# 2. BASIC CHECKS
# ============================================================

required_columns = [
    "angle_1",
    "angle_2",
    "angle_3",
    "angle_4",
    "angle_5",
    "reps",
    "entanglement",
    "initial_layout",
    "mean_tvd",
    "data_row_index",
]

missing = [
    c for c in required_columns
    if c not in marrakesh_df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(marrakesh_df) != 4800:
    raise ValueError(
        f"Expected 4800 Marrakesh rows, found {len(marrakesh_df)}"
    )

print("Required columns: OK")
print("Dataset size: 4800 rows")


# ============================================================
# 3. SELECT EXACT SAME 60 VALIDATION CIRCUITS
# ============================================================

marrakesh_validation_df = (
    marrakesh_df
    .sample(n=60, random_state=42)
    .copy()
    .reset_index(drop=True)
)

print("\nValidation circuits:", len(marrakesh_validation_df))


# ============================================================
# 4. CONNECT TO REAL IBM MARRAKESH
# ============================================================

service = QiskitRuntimeService()

backend = service.backend("ibm_marrakesh")

print("Real backend:", backend.name)
print("Backend qubits:", backend.num_qubits)


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def counts_to_probs(counts):
    total = sum(counts.values())
    return {
        str(k).replace(" ", ""): v / total
        for k, v in counts.items()
    }


def tvd(p, q):
    keys = set(p) | set(q)

    return 0.5 * sum(
        abs(p.get(k, 0.0) - q.get(k, 0.0))
        for k in keys
    )


# ============================================================
# 6. REBUILD EXACT LOGICAL CIRCUITS
#    + EXACT STORED PHYSICAL LAYOUT
# ============================================================

real_marrakesh_circuits = []
ideal_distributions = []

validation_records = []

for pos, (_, row) in enumerate(
    marrakesh_validation_df.iterrows()
):

    reps = int(row["reps"])
    entanglement = row["entanglement"]

    # --------------------------------------------------------
    # Recover EXACT encoded angles stored during simulation
    # --------------------------------------------------------

    data_row = np.array(
        [
            row[f"angle_{i}"]
            for i in range(1, 6)
        ],
        dtype=float
    )

    # --------------------------------------------------------
    # Rebuild same logical circuit
    # --------------------------------------------------------

    feature_map = zz_feature_map(
        feature_dimension=5,
        reps=reps,
        entanglement=entanglement,
    )

    abstract_circuit = feature_map.assign_parameters(
        data_row
    )

    # --------------------------------------------------------
    # Exact ideal distribution
    # --------------------------------------------------------

    ideal_sv = Statevector.from_instruction(
        abstract_circuit
    )

    ideal_probs = ideal_sv.probabilities_dict()

    ideal_distributions.append(ideal_probs)

    # --------------------------------------------------------
    # Add measurement
    # --------------------------------------------------------

    measured_circuit = abstract_circuit.copy()
    measured_circuit.measure_all()

    # --------------------------------------------------------
    # Recover EXACT layout used in FakeMarrakesh simulation
    # --------------------------------------------------------

    layout = row["initial_layout"]

    if isinstance(layout, str):
        layout = ast.literal_eval(layout)

    layout = list(layout)

    # --------------------------------------------------------
    # Transpile using EXACT same settings
    # --------------------------------------------------------

    transpiled_circuit = transpile(
        measured_circuit,
        backend,
        optimization_level=1,
        initial_layout=layout,
        seed_transpiler=42,
    )

    real_marrakesh_circuits.append(
        transpiled_circuit
    )

    validation_records.append({
        "validation_index": pos,
        "data_row_index": int(row["data_row_index"]),
        "reps": reps,
        "entanglement": entanglement,
        "initial_layout": str(layout),
        "fake_marrakesh_tvd": float(row["mean_tvd"]),
    })


print("Logical circuits rebuilt:", len(real_marrakesh_circuits))
print("Ideal distributions computed:", len(ideal_distributions))


# ============================================================
# 7. REAL HARDWARE EXECUTION
# ============================================================

sampler = SamplerV2(mode=backend)

job = sampler.run(
    real_marrakesh_circuits,
    shots=1024,
)

print("\nSubmitted real Marrakesh job:")
print("Job ID:", job.job_id())

result = job.result()

pub_results = list(result)

print("Completed PUB results:", len(pub_results))


# ============================================================
# 8. COMPUTE REAL-HARDWARE TVD
# ============================================================

for pos, pub_result in enumerate(pub_results):

    counts = pub_result.data.meas.get_counts()

    real_probs = counts_to_probs(counts)

    real_tvd = tvd(
        ideal_distributions[pos],
        real_probs,
    )

    validation_records[pos][
        "real_marrakesh_tvd"
    ] = real_tvd


validation_results_df = pd.DataFrame(
    validation_records
)


# ============================================================
# 9. OVERALL METRICS
# ============================================================

fake = validation_results_df[
    "fake_marrakesh_tvd"
].values

real = validation_results_df[
    "real_marrakesh_tvd"
].values

pearson_r, pearson_p = pearsonr(fake, real)

spearman_r, spearman_p = spearmanr(fake, real)

mae = mean_absolute_error(
    real,
    fake
)

rmse = np.sqrt(
    mean_squared_error(
        real,
        fake
    )
)


print("\n" + "=" * 70)
print("MARRAKESH MATCHED-LAYOUT VALIDATION")
print("=" * 70)

print(f"N = {len(real)}")

print(
    f"Pearson r    = {pearson_r:.4f}"
)

print(
    f"Pearson p    = {pearson_p:.4g}"
)

print(
    f"Spearman rho = {spearman_r:.4f}"
)

print(
    f"Spearman p   = {spearman_p:.4g}"
)

print(
    f"MAE          = {mae:.4f}"
)

print(
    f"RMSE         = {rmse:.4f}"
)


# ============================================================
# 10. LAYOUT-WISE METRICS
# ============================================================

layout_results = []

for layout, group in validation_results_df.groupby(
    "initial_layout"
):

    if len(group) >= 2:

        f = group["fake_marrakesh_tvd"].values
        r = group["real_marrakesh_tvd"].values

        pr, pp = pearsonr(f, r)

        sr, sp = spearmanr(f, r)

        layout_results.append({
            "initial_layout": layout,
            "N": len(group),
            "Pearson_r": pr,
            "Pearson_p": pp,
            "Spearman_rho": sr,
            "Spearman_p": sp,
            "MAE": mean_absolute_error(r, f),
            "RMSE": np.sqrt(
                mean_squared_error(r, f)
            ),
        })

layout_results_df = pd.DataFrame(
    layout_results
)

print("\n" + "=" * 70)
print("LAYOUT-WISE RESULTS")
print("=" * 70)

display(layout_results_df)


# ============================================================
# 11. CONFIGURATION-WISE METRICS
# ============================================================

config_results = []

for (reps, entanglement), group in (
    validation_results_df
    .groupby(["reps", "entanglement"])
):

    if len(group) >= 2:

        f = group["fake_marrakesh_tvd"].values
        r = group["real_marrakesh_tvd"].values

        pr, pp = pearsonr(f, r)

        sr, sp = spearmanr(f, r)

        config_results.append({
            "reps": reps,
            "entanglement": entanglement,
            "N": len(group),
            "Pearson_r": pr,
            "Pearson_p": pp,
            "Spearman_rho": sr,
            "Spearman_p": sp,
            "MAE": mean_absolute_error(r, f),
            "RMSE": np.sqrt(
                mean_squared_error(r, f)
            ),
        })

config_results_df = pd.DataFrame(
    config_results
)

print("\n" + "=" * 70)
print("CONFIGURATION-WISE RESULTS")
print("=" * 70)

display(config_results_df)


# ============================================================
# 12. SAVE RESULTS
# ============================================================

validation_results_df.to_csv(
    "matched_layout_real_marrakesh_validation_rerun.csv",
    index=False
)

layout_results_df.to_csv(
    "matched_layout_real_marrakesh_layout_results_rerun.csv",
    index=False
)

config_results_df.to_csv(
    "matched_layout_real_marrakesh_config_results_rerun.csv",
    index=False
)

print("\nSaved:")
print(" - matched_layout_real_marrakesh_validation_rerun.csv")
print(" - matched_layout_real_marrakesh_layout_results_rerun.csv")
print(" - matched_layout_real_marrakesh_config_results_rerun.csv")

Loading: c:\Users\jaspr\Downloads\quantum_noise_predictor\notebooks\quantum_noise_predictor_dataset_marrakesh_61.csv
Dataset shape: (4800, 61)
Required columns: OK
Dataset size: 4800 rows

Validation circuits: 60
Real backend: ibm_marrakesh
Backend qubits: 156
Logical circuits rebuilt: 60
Ideal distributions computed: 60

Submitted real Marrakesh job:
Job ID: daf7a0l1ierc738n0820
Completed PUB results: 60

MARRAKESH MATCHED-LAYOUT VALIDATION
N = 60
Pearson r    = 0.6215
Pearson p    = 1.164e-07
Spearman rho = 0.6937
Spearman p   = 8.037e-10
MAE          = 0.0573
RMSE         = 0.0753

LAYOUT-WISE RESULTS


,initial_layout,N,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,MAE,RMSE
0,"[107, 108, 109, 110, 111]",16,0.962161,2.694146e-09,0.708824,0.002112,0.029430,0.035681
1,"[11, 12, 13, 14, 15]",16,0.896542,2.577607e-06,0.611765,0.011788,0.061881,0.078146
2,"[125, 126, 127, 128, 129]",12,0.978720,3.316054e-08,0.706294,0.010245,0.031167,0.039188
3,"[69, 70, 71, 72, 73]",16,0.649579,6.461616e-03,0.494118,0.051723,0.100192,0.112835



CONFIGURATION-WISE RESULTS


,reps,entanglement,N,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,MAE,RMSE
0,1,full,10,0.153798,0.671411,0.139394,0.700932,0.040508,0.045784
1,1,linear,11,-0.273327,0.416076,-0.327273,0.325895,0.013370,0.014704
2,2,full,11,0.629248,0.038052,0.681818,0.020843,0.094411,0.100134
3,2,linear,12,0.227648,0.476722,0.454545,0.137658,0.058365,0.083065
4,3,full,5,0.295827,0.628910,-0.100000,0.872889,0.071121,0.094074
5,3,linear,11,0.762366,0.006372,0.845455,0.001045,0.071945,0.085029



Saved:
 - matched_layout_real_marrakesh_validation_rerun.csv
 - matched_layout_real_marrakesh_layout_results_rerun.csv
 - matched_layout_real_marrakesh_config_results_rerun.csv


In [6]:
# ============================================================
# RESTORE DATA + FEATURES + SCALER
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Load dataset
DATA_PATH = "../data/raw/blood_cell_anomaly_detection.csv"

df = pd.read_csv(DATA_PATH)

# Exact 5 features used in the project
selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

# Clean dataframe
df_with_features = (
    df
    .dropna(subset=selected_columns)
    .reset_index(drop=True)
)

# Fit scaler exactly as before
scaler = MinMaxScaler(
    feature_range=(0, 2 * np.pi)
)

scaler.fit(
    df_with_features[selected_columns].values
)

print("Dataset shape:", df.shape)
print("Clean dataframe:", df_with_features.shape)
print("Selected features:", selected_columns)
print("Scaler range: [0, 2π]")

Dataset shape: (5880, 36)
Clean dataframe: (5880, 36)
Selected features: ['cell_diameter_um', 'chromatin_density', 'cytoplasm_ratio', 'circularity', 'eccentricity']
Scaler range: [0, 2π]


In [12]:
# ============================================================
# FEZ: RECREATE SOURCE DATAFRAME
# ============================================================

N_ROWS = 200

df_with_features = (
    df
    .dropna(subset=selected_columns)
    .reset_index(drop=True)
)

print("Clean dataframe shape:", df_with_features.shape)

Clean dataframe shape: (5880, 36)


In [6]:
def counts_to_probabilities(counts):
    total_shots = sum(counts.values())

    return {
        bitstring: count / total_shots
        for bitstring, count in counts.items()
    }


def calculate_tvd_from_probs(p, q):
    all_keys = set(p.keys()) | set(q.keys())

    return 0.5 * sum(
        abs(p.get(key, 0.0) - q.get(key, 0.0))
        for key in all_keys
    )

In [17]:
fez_df = (
    df_with_features
    .sample(
        n=N_ROWS,
        random_state=123
    )
    .copy()
    .reset_index(drop=True)
)

fez_raw = fez_df[selected_columns].values

fez_scaled = scaler.transform(
    fez_df[selected_columns]
)

print("Fez raw shape:", fez_raw.shape)
print("Fez scaled shape:", fez_scaled.shape)

Fez raw shape: (200, 5)
Fez scaled shape: (200, 5)


c:\Users\jaspr\miniconda3\envs\qnoise\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


In [ ]:
# ============================================================
# FEZ DATASET GENERATION
# 200 source rows × 6 configs × 4 Fez layouts
# 5 noisy repeats × 1024 shots
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel


# ============================================================
# 1. CREATE FAKE FEZ BACKEND
# ============================================================

fake_fez = FakeFez()

fez_noise_model = NoiseModel.from_backend(
    fake_fez
)

fez_noisy_simulator = AerSimulator(
    noise_model=fez_noise_model
)

print("Fake backend:", fake_fez.name)
print("Number of qubits:", fake_fez.num_qubits)


# ============================================================
# 2. RECREATE KINGSTON + MARRAKESH ROW SELECTION
#    AND SELECT FEZ FROM THE REMAINING DATA
#
#    This guarantees:
#       Kingston ∩ Marrakesh = 0
#       Kingston ∩ Fez       = 0
#       Marrakesh ∩ Fez      = 0
# ============================================================

N_ROWS = 200

# Exact Kingston selection used in the project
kingston_df, remaining_df = train_test_split(
    df_with_features,
    train_size=N_ROWS,
    stratify=df_with_features["cell_type"],
    random_state=42
)

# Exact Marrakesh selection used in the project:
# sampled from the rows NOT used by Kingston
marrakesh_df, remaining_df = train_test_split(
    remaining_df,
    train_size=N_ROWS,
    stratify=remaining_df["cell_type"],
    random_state=42
)

# Fez is sampled only from the rows still unused
fez_df, _ = train_test_split(
    remaining_df,
    train_size=N_ROWS,
    stratify=remaining_df["cell_type"],
    random_state=123
)

# Keep original source-row indices temporarily for overlap checks
kingston_source_indices = set(kingston_df.index)
marrakesh_source_indices = set(marrakesh_df.index)
fez_source_indices = set(fez_df.index)

# Reset only AFTER checking disjointness
fez_df = fez_df.reset_index(drop=True)


# ============================================================
# 3. VERIFY ZERO SOURCE-ROW OVERLAP
# ============================================================

km_overlap = (
    kingston_source_indices
    & marrakesh_source_indices
)

kf_overlap = (
    kingston_source_indices
    & fez_source_indices
)

mf_overlap = (
    marrakesh_source_indices
    & fez_source_indices
)

print("\nSource-row overlap check:")
print(
    "Kingston ∩ Marrakesh:",
    len(km_overlap)
)
print(
    "Kingston ∩ Fez      :",
    len(kf_overlap)
)
print(
    "Marrakesh ∩ Fez     :",
    len(mf_overlap)
)

assert len(km_overlap) == 0
assert len(kf_overlap) == 0
assert len(mf_overlap) == 0

print("All three source datasets are disjoint.")


# ============================================================
# 4. PREPARE FEZ RAW + SCALED FEATURES
# ============================================================

fez_raw = (
    fez_df[selected_columns]
    .values
)

# .values avoids the sklearn feature-name warning
fez_scaled = scaler.transform(
    fez_df[selected_columns].values
)

print("\nFez source data:")
print(
    "Raw shape   :",
    fez_raw.shape
)
print(
    "Scaled shape:",
    fez_scaled.shape
)
print(
    "Scaled range:",
    fez_scaled.min(),
    "to",
    fez_scaled.max()
)


# ============================================================
# 5. CONFIGURATIONS
# ============================================================

CONFIGS = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

print(
    "\nConfigurations:",
    len(CONFIGS)
)


# ============================================================
# 6. HELPER: GET CZ ERROR
# ============================================================

def get_cz_error(backend, q1, q2):

    props = None

    try:
        props = backend.target["cz"][
            (q1, q2)
        ]
    except Exception:
        pass

    if props is None:

        try:
            props = backend.target["cz"][
                (q2, q1)
            ]
        except Exception:
            pass

    if props is None:
        return None

    if props.error is None:
        return None

    return float(props.error)


# ============================================================
# 7. FIND CONNECTED 5-QUBIT CHAINS
# ============================================================

coupling_edges = {
    tuple(sorted(edge))
    for edge in fake_fez.coupling_map.get_edges()
}

valid_fez_layouts = []

for start in range(
    fake_fez.num_qubits - 4
):

    layout = list(
        range(start, start + 5)
    )

    # --------------------------------------------------------
    # Require a connected 5-qubit chain
    # --------------------------------------------------------

    connected = all(
        tuple(
            sorted(
                (
                    layout[i],
                    layout[i + 1]
                )
            )
        ) in coupling_edges
        for i in range(4)
    )

    if not connected:
        continue

    # --------------------------------------------------------
    # Collect CZ errors for all four adjacent edges
    # --------------------------------------------------------

    errors = []

    for i in range(4):

        q1 = layout[i]
        q2 = layout[i + 1]

        error = get_cz_error(
            fake_fez,
            q1,
            q2
        )

        if error is None:
            errors = None
            break

        errors.append(error)

    if errors is None:
        continue

    valid_fez_layouts.append({
        "layout": layout,
        "mean_error": np.mean(errors),
        "max_error": np.max(errors),
        "errors": errors
    })


# ============================================================
# 8. SORT + REMOVE POOR/CATASTROPHIC LAYOUTS
# ============================================================

valid_fez_layouts = sorted(
    valid_fez_layouts,
    key=lambda x: x["mean_error"]
)

usable_fez = [
    x
    for x in valid_fez_layouts
    if x["max_error"] < 0.10
]

print(
    "\nValid Fez layouts:",
    len(valid_fez_layouts)
)

print(
    "Usable Fez layouts:",
    len(usable_fez)
)

if len(usable_fez) < 4:

    raise RuntimeError(
        "Fewer than 4 usable Fez layouts found."
    )


# ============================================================
# 9. SELECT FOUR LAYOUTS SPREAD ACROSS
#    CALIBRATION QUALITY
# ============================================================

indices = np.linspace(
    0,
    len(usable_fez) - 1,
    4,
    dtype=int
)

FEZ_LAYOUTS = [
    usable_fez[i]["layout"]
    for i in indices
]


print(
    "\nSelected Fez layouts:\n"
)

for layout in FEZ_LAYOUTS:

    item = next(
        x
        for x in usable_fez
        if x["layout"] == layout
    )

    print(
        layout,
        "| mean =",
        round(
            item["mean_error"],
            6
        ),
        "| max =",
        round(
            item["max_error"],
            6
        ),
        "| edges =",
        [
            round(x, 6)
            for x in item["errors"]
        ]
    )


# ============================================================
# 10. CHECK EXPERIMENT COUNT
# ============================================================

total_experiments = (
    len(fez_scaled)
    * len(CONFIGS)
    * len(FEZ_LAYOUTS)
)

print(
    "\nExperiment design:"
)

print(
    "Source rows :",
    len(fez_scaled)
)

print(
    "Configs     :",
    len(CONFIGS)
)

print(
    "Layouts     :",
    len(FEZ_LAYOUTS)
)

print(
    "Total       :",
    total_experiments
)

print(
    "Repeats     :",
    5
)

print(
    "Shots       :",
    1024
)

assert len(fez_scaled) == 200
assert len(CONFIGS) == 6
assert len(FEZ_LAYOUTS) == 4
assert total_experiments == 4800


# ============================================================
# 11. GENERATE FEZ DATASET
# ============================================================

fez_results = []

start_time = time.time()

for data_idx in range(
    len(fez_scaled)
):

    raw_row = fez_raw[data_idx]

    scaled_row = fez_scaled[data_idx]

    for reps, entanglement in CONFIGS:

        for layout_id, layout in enumerate(
            FEZ_LAYOUTS
        ):

            result = run_complete_experiment(
                raw_data_row=raw_row,
                data_row=scaled_row,
                reps=reps,
                entanglement=entanglement,
                backend=fake_fez,
                noisy_simulator=fez_noisy_simulator,
                shots=1024,
                repeats=5,
                initial_layout=layout,
                seed_transpiler=42,
            )

            # Source-row identity within Fez
            result["data_row_index"] = data_idx

            # Fez-specific physical layout identifier
            result["layout_id"] = layout_id

            fez_results.append(
                result
            )

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (data_idx + 1) % 5 == 0:

        completed = len(fez_results)

        elapsed = (
            time.time()
            - start_time
        )

        rate = (
            completed / elapsed
        )

        remaining = (
            total_experiments
            - completed
        )

        eta_minutes = (
            remaining
            / rate
            / 60
        )

        print(
            f"[{completed / total_experiments * 100:6.2f}%] "
            f"{completed}/{total_experiments} experiments | "
            f"Elapsed: {elapsed / 60:.1f} min | "
            f"ETA: {eta_minutes:.1f} min"
        )


# ============================================================
# 12. CREATE FEZ DATAFRAME
# ============================================================

fez_predictor_df = pd.DataFrame(
    fez_results
)

print(
    "\n" + "=" * 60
)

print(
    "FEZ DATASET GENERATED"
)

print(
    "=" * 60
)

print(
    "Shape:",
    fez_predictor_df.shape
)


# ============================================================
# 13. VERIFY 61-COLUMN SCHEMA
# ============================================================

print(
    "\nNumber of columns:",
    len(fez_predictor_df.columns)
)

assert (
    len(fez_predictor_df.columns)
    == 61
)

print(
    "61-column schema check PASSED."
)


# ============================================================
# 14. VALIDATION CHECKS
# ============================================================

print(
    "\nLayout distribution:"
)

print(
    fez_predictor_df[
        "layout_id"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nConfiguration distribution:"
)

print(
    fez_predictor_df[
        [
            "reps",
            "entanglement"
        ]
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nTVD summary:"
)

print(
    fez_predictor_df[
        "mean_tvd"
    ].describe()
)


# ============================================================
# 15. CHECK EXPECTED 4,800 ROWS
# ============================================================

assert (
    len(fez_predictor_df)
    == 4800
)

assert (
    fez_predictor_df[
        "layout_id"
    ]
    .value_counts()
    .sort_index()
    .tolist()
    == [1200, 1200, 1200, 1200]
)

assert (
    fez_predictor_df[
        [
            "reps",
            "entanglement"
        ]
    ]
    .value_counts()
    .sort_index()
    .tolist()
    == [800, 800, 800, 800, 800, 800]
)

print(
    "\nAll dataset checks PASSED."
)


# ============================================================
# 16. SAVE FEZ DATASET
# ============================================================

FEZ_OUTPUT = (
    "quantum_noise_predictor_dataset_fez_61.csv"
)

fez_predictor_df.to_csv(
    FEZ_OUTPUT,
    index=False
)

print(
    "\nSaved:",
    FEZ_OUTPUT
)

Fake backend: fake_fez
Number of qubits: 156

Source-row overlap check:
Kingston ∩ Marrakesh: 0
Kingston ∩ Fez      : 0
Marrakesh ∩ Fez     : 0
All three source datasets are disjoint.

Fez source data:
Raw shape   : (200, 5)
Scaled shape: (200, 5)
Scaled range: 0.0 to 6.283185307179586

Configurations: 6

Valid Fez layouts: 96
Usable Fez layouts: 79

Selected Fez layouts:

[86, 87, 88, 89, 90] | mean = 0.002609 | max = 0.002969 | edges = [0.002604, 0.002278, 0.002584, 0.002969]
[125, 126, 127, 128, 129] | mean = 0.003664 | max = 0.005058 | edges = [0.002931, 0.003113, 0.003556, 0.005058]
[23, 24, 25, 26, 27] | mean = 0.005088 | max = 0.010274 | edges = [0.003024, 0.003582, 0.003471, 0.010274]
[69, 70, 71, 72, 73] | mean = 0.019313 | max = 0.039803 | edges = [0.005484, 0.004062, 0.039803, 0.027903]

Experiment design:
Source rows : 200
Configs     : 6
Layouts     : 4
Total       : 4800
Repeats     : 5
Shots       : 1024


In [7]:
# ============================================================
# MATCHED 14,400-EXPERIMENT DATASET
#
# 200 NEW source rows
# × 3 backends
# × 6 configurations
# × 4 hardware-specific layouts
# = 14,400 circuit instances
#
# 5 noisy repetitions × 1024 shots
#
# CRASH-SAFE:
# - JSONL checkpoint after EVERY experiment
# - CSV checkpoint after EVERY source-row/backend block
# - Automatically resumes and skips completed experiments
# ============================================================

import os
import json
import time
import ast
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

from qiskit_ibm_runtime.fake_provider import (
    FakeKingston,
    FakeMarrakesh,
    FakeFez,
)

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel


# ============================================================
# 1. PATHS
# ============================================================

RESULT_DIR = Path(
    "../experiments/results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_JSONL = (
    RESULT_DIR /
    "quantum_noise_predictor_dataset_matched_14400_checkpoint.jsonl"
)

CHECKPOINT_CSV = (
    RESULT_DIR /
    "quantum_noise_predictor_dataset_matched_14400_checkpoint.csv"
)

FINAL_FULL_CSV = (
    RESULT_DIR /
    "quantum_noise_predictor_dataset_matched_14400_full.csv"
)

FINAL_61_CSV = (
    RESULT_DIR /
    "quantum_noise_predictor_dataset_matched_14400_61.csv"
)

METADATA_JSON = (
    RESULT_DIR /
    "quantum_noise_predictor_dataset_matched_14400_metadata.json"
)


# ============================================================
# 2. BASIC SETTINGS
# ============================================================

N_NEW_ROWS = 200
SHOTS = 1024
REPEATS = 5
SEED_TRANSPILER = 42

MATCHED_RANDOM_STATE = 2026

CONFIGS_MATCHED = [
    (1, "linear"),
    (2, "linear"),
    (3, "linear"),
    (1, "full"),
    (2, "full"),
    (3, "full"),
]

print("=" * 75)
print("MATCHED 14,400-EXPERIMENT DATASET")
print("=" * 75)

print("Source rows:", N_NEW_ROWS)
print("Backends:", 3)
print("Configurations:", len(CONFIGS_MATCHED))
print("Layouts/backend:", 4)
print("Shots:", SHOTS)
print("Repeats:", REPEATS)

print(
    "Expected experiments:",
    N_NEW_ROWS
    * 3
    * len(CONFIGS_MATCHED)
    * 4
)


# ============================================================
# 3. RESTORE CLEAN SOURCE DATAFRAME
# ============================================================

SOURCE_PATH = (
    "../data/raw/"
    "blood_cell_anomaly_detection.csv"
)

df_source = pd.read_csv(
    SOURCE_PATH
)

selected_columns = [
    "cell_diameter_um",
    "chromatin_density",
    "cytoplasm_ratio",
    "circularity",
    "eccentricity",
]

df_clean = (
    df_source
    .dropna(
        subset=selected_columns
    )
    .reset_index(drop=True)
)

print("\nClean source dataframe:")
print(df_clean.shape)


# ============================================================
# 4. FIT SCALER EXACTLY AS PROJECT
# ============================================================

from sklearn.preprocessing import MinMaxScaler

matched_scaler = MinMaxScaler(
    feature_range=(0, 2 * np.pi)
)

matched_scaler.fit(
    df_clean[selected_columns].values
)

print("Scaler: [0, 2π]")


# ============================================================
# 5. RECONSTRUCT PREVIOUS KINGSTON 200
# ============================================================

previous_kingston, remaining_after_kingston = (
    train_test_split(
        df_clean,
        train_size=200,
        stratify=df_clean["cell_type"],
        random_state=42,
    )
)

kingston_previous_indices = set(
    previous_kingston.index
)


# ============================================================
# 6. RECONSTRUCT PREVIOUS MARRAKESH 200
# ============================================================

previous_marrakesh, remaining_after_marrakesh = (
    train_test_split(
        remaining_after_kingston,
        train_size=200,
        stratify=remaining_after_kingston["cell_type"],
        random_state=42,
    )
)

marrakesh_previous_indices = set(
    previous_marrakesh.index
)


# ============================================================
# 7. RECONSTRUCT PREVIOUS FEZ 200
# ============================================================

previous_fez, remaining_after_fez = (
    train_test_split(
        remaining_after_marrakesh,
        train_size=200,
        stratify=remaining_after_marrakesh["cell_type"],
        random_state=123,
    )
)

fez_previous_indices = set(
    previous_fez.index
)


# ============================================================
# 8. VERIFY PREVIOUS 600 ARE DISJOINT
# ============================================================

assert (
    len(
        kingston_previous_indices
        & marrakesh_previous_indices
    )
    == 0
)

assert (
    len(
        kingston_previous_indices
        & fez_previous_indices
    )
    == 0
)

assert (
    len(
        marrakesh_previous_indices
        & fez_previous_indices
    )
    == 0
)

previous_600_indices = (
    kingston_previous_indices
    | marrakesh_previous_indices
    | fez_previous_indices
)

assert len(previous_600_indices) == 600

print("\nPrevious source rows:")
print("Kingston :", len(kingston_previous_indices))
print("Marrakesh:", len(marrakesh_previous_indices))
print("Fez      :", len(fez_previous_indices))
print("Combined :", len(previous_600_indices))


# ============================================================
# 9. REMOVE PREVIOUS 600
# ============================================================

available_for_matched = (
    df_clean
    .drop(
        index=previous_600_indices
    )
    .copy()
)

print(
    "\nRows available for NEW matched dataset:",
    len(available_for_matched)
)


# ============================================================
# 10. SELECT NEW COMMON 200 ROWS
# ============================================================

matched_source_df, _ = train_test_split(
    available_for_matched,
    train_size=N_NEW_ROWS,
    stratify=available_for_matched["cell_type"],
    random_state=MATCHED_RANDOM_STATE,
)

# Preserve ORIGINAL cleaned-data index
matched_source_indices = (
    matched_source_df.index
    .tolist()
)

# ------------------------------------------------------------
# Critical overlap check
# ------------------------------------------------------------

matched_overlap = (
    set(matched_source_indices)
    & previous_600_indices
)

assert len(matched_overlap) == 0

print(
    "\nNEW matched source rows:",
    len(matched_source_df)
)

print(
    "Overlap with previous 600:",
    len(matched_overlap)
)

assert len(matched_source_df) == 200


# ============================================================
# 11. RESET POSITIONAL INDEX
# ============================================================

matched_source_df = (
    matched_source_df
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "source_row_index"
        }
    )
)

assert (
    matched_source_df[
        "source_row_index"
    ].nunique()
    == 200
)

print(
    "Unique original source indices:",
    matched_source_df[
        "source_row_index"
    ].nunique()
)


# ============================================================
# 12. RAW + SCALED MATCHED INPUTS
# ============================================================

matched_raw = (
    matched_source_df[
        selected_columns
    ]
    .values
)

matched_scaled = matched_scaler.transform(
    matched_raw
)

print(
    "\nMatched raw shape:",
    matched_raw.shape
)

print(
    "Matched scaled shape:",
    matched_scaled.shape
)


# ============================================================
# 13. CREATE THREE FAKE BACKENDS
# ============================================================

print("\nCreating fake backends...")

fake_kingston_matched = FakeKingston()

fake_marrakesh_matched = FakeMarrakesh()

fake_fez_matched = FakeFez()


# ============================================================
# 14. CREATE NOISE SIMULATORS
# ============================================================

kingston_noise_model_matched = (
    NoiseModel.from_backend(
        fake_kingston_matched
    )
)

marrakesh_noise_model_matched = (
    NoiseModel.from_backend(
        fake_marrakesh_matched
    )
)

fez_noise_model_matched = (
    NoiseModel.from_backend(
        fake_fez_matched
    )
)


kingston_sim_matched = AerSimulator(
    noise_model=kingston_noise_model_matched
)

marrakesh_sim_matched = AerSimulator(
    noise_model=marrakesh_noise_model_matched
)

fez_sim_matched = AerSimulator(
    noise_model=fez_noise_model_matched
)


# ============================================================
# 15. HARDWARE LAYOUT SELECTION FUNCTION
#
# Same methodology used in the existing notebook:
# - connected 5-qubit chain
# - all four CZ edges must have calibration
# - discard max CZ error >= 0.10
# - sort by mean CZ error
# - select 4 layouts spread across quality
# ============================================================

def select_four_layouts(
    backend,
    backend_label
):

    coupling_edges = {
        tuple(sorted(edge))
        for edge in backend.coupling_map.get_edges()
    }

    valid_layouts = []

    for start in range(
        backend.num_qubits - 4
    ):

        layout = list(
            range(
                start,
                start + 5
            )
        )

        connected = all(
            tuple(
                sorted(
                    (
                        layout[i],
                        layout[i + 1]
                    )
                )
            )
            in coupling_edges
            for i in range(4)
        )

        if not connected:
            continue

        errors = []

        for i in range(4):

            q1 = layout[i]
            q2 = layout[i + 1]

            props = None

            try:
                props = backend.target["cz"][
                    (q1, q2)
                ]
            except Exception:
                pass

            if props is None:

                try:
                    props = backend.target["cz"][
                        (q2, q1)
                    ]
                except Exception:
                    pass

            if (
                props is None
                or props.error is None
            ):
                errors = None
                break

            errors.append(
                float(props.error)
            )

        if errors is None:
            continue

        valid_layouts.append({
            "layout": layout,
            "mean_error": np.mean(errors),
            "max_error": np.max(errors),
            "errors": errors,
        })


    valid_layouts = sorted(
        valid_layouts,
        key=lambda x: x["mean_error"]
    )


    usable_layouts = [
        x
        for x in valid_layouts
        if x["max_error"] < 0.10
    ]


    if len(usable_layouts) < 4:

        raise RuntimeError(
            f"{backend_label}: fewer than "
            f"4 usable layouts."
        )


    indices = np.linspace(
        0,
        len(usable_layouts) - 1,
        4,
        dtype=int
    )


    selected = [
        usable_layouts[i]["layout"]
        for i in indices
    ]


    print(
        f"\n{backend_label} selected layouts:"
    )

    for layout in selected:

        item = next(
            x
            for x in usable_layouts
            if x["layout"] == layout
        )

        print(
            layout,
            "| mean =",
            round(
                item["mean_error"],
                6
            ),
            "| max =",
            round(
                item["max_error"],
                6
            ),
            "| edges =",
            [
                round(x, 6)
                for x in item["errors"]
            ]
        )

    return selected


# ============================================================
# 16. SELECT HARDWARE-SPECIFIC LAYOUTS
# ============================================================

KINGSTON_MATCHED_LAYOUTS = (
    select_four_layouts(
        fake_kingston_matched,
        "FakeKingston"
    )
)

MARRAKESH_MATCHED_LAYOUTS = (
    select_four_layouts(
        fake_marrakesh_matched,
        "FakeMarrakesh"
    )
)

FEZ_MATCHED_LAYOUTS = (
    select_four_layouts(
        fake_fez_matched,
        "FakeFez"
    )
)


# ============================================================
# 17. BACKEND CONFIGURATION
# ============================================================

BACKENDS = [
    {
        "name": "FakeKingston",
        "backend": fake_kingston_matched,
        "simulator": kingston_sim_matched,
        "layouts": KINGSTON_MATCHED_LAYOUTS,
    },
    {
        "name": "FakeMarrakesh",
        "backend": fake_marrakesh_matched,
        "simulator": marrakesh_sim_matched,
        "layouts": MARRAKESH_MATCHED_LAYOUTS,
    },
    {
        "name": "FakeFez",
        "backend": fake_fez_matched,
        "simulator": fez_sim_matched,
        "layouts": FEZ_MATCHED_LAYOUTS,
    },
]


# ============================================================
# 18. EXPECTED EXPERIMENT COUNT
# ============================================================

EXPECTED_TOTAL = (
    N_NEW_ROWS
    * len(BACKENDS)
    * len(CONFIGS_MATCHED)
    * 4
)

assert EXPECTED_TOTAL == 14400

print(
    "\nExpected total experiments:",
    EXPECTED_TOTAL
)


# ============================================================
# 19. LOAD EXISTING CHECKPOINT
#
# JSONL is the PRIMARY crash-safe checkpoint.
# One completed experiment = one line.
# ============================================================

checkpoint_records = []

if CHECKPOINT_JSONL.exists():

    print(
        "\nExisting checkpoint found:"
    )

    print(
        CHECKPOINT_JSONL
    )

    with open(
        CHECKPOINT_JSONL,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                checkpoint_records.append(
                    json.loads(line)
                )
            except json.JSONDecodeError:
                # Ignore an incomplete final line
                # caused by an interrupted write.
                print(
                    "Warning: ignored "
                    "incomplete checkpoint line."
                )

else:

    print(
        "\nNo previous checkpoint."
    )


print(
    "Recovered checkpoint rows:",
    len(checkpoint_records)
)


# ============================================================
# 20. BUILD COMPLETED-EXPERIMENT KEY SET
# ============================================================

completed_keys = set()

for record in checkpoint_records:

    key = (
        record["backend"],
        int(record["source_row_position"]),
        int(record["reps"]),
        record["entanglement"],
        int(record["layout_id"]),
    )

    completed_keys.add(key)


print(
    "Recovered completed experiments:",
    len(completed_keys)
)


# ============================================================
# 21. SANITY CHECK CHECKPOINT
# ============================================================

if len(completed_keys) > EXPECTED_TOTAL:

    raise RuntimeError(
        "Checkpoint contains more experiments "
        "than expected."
    )


# ============================================================
# 22. HELPER: APPEND ONE EXPERIMENT SAFELY
# ============================================================

def append_checkpoint(record):

    with open(
        CHECKPOINT_JSONL,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                default=str
            )
            + "\n"
        )

        f.flush()


# ============================================================
# 23. RUN / RESUME
# ============================================================

start_time = time.time()

newly_completed = 0

last_progress_time = time.time()


for backend_info in BACKENDS:

    backend_name = backend_info["name"]

    backend = backend_info["backend"]

    noisy_simulator = (
        backend_info["simulator"]
    )

    layouts = backend_info["layouts"]


    print("\n")
    print("=" * 75)
    print(
        f"BACKEND: {backend_name}"
    )
    print("=" * 75)


    for source_position in range(
        N_NEW_ROWS
    ):

        raw_row = matched_raw[
            source_position
        ]

        scaled_row = matched_scaled[
            source_position
        ]

        original_source_index = int(
            matched_source_df.iloc[
                source_position
            ]["source_row_index"]
        )


        row_start = time.time()

        row_completed = 0


        for reps, entanglement in (
            CONFIGS_MATCHED
        ):

            for layout_id, layout in enumerate(
                layouts
            ):

                key = (
                    backend_name,
                    source_position,
                    int(reps),
                    entanglement,
                    int(layout_id),
                )


                # ------------------------------------------------
                # RESUME LOGIC
                # ------------------------------------------------

                if key in completed_keys:
                    continue


                # ------------------------------------------------
                # RUN ONE EXPERIMENT
                # ------------------------------------------------

                result = run_complete_experiment(
                    raw_data_row=raw_row,
                    data_row=scaled_row,
                    reps=reps,
                    entanglement=entanglement,
                    backend=backend,
                    noisy_simulator=noisy_simulator,
                    shots=SHOTS,
                    repeats=REPEATS,
                    initial_layout=layout,
                    seed_transpiler=SEED_TRANSPILER,
                )


                # ------------------------------------------------
                # PROJECT IDS
                # ------------------------------------------------

                result[
                    "data_row_index"
                ] = source_position

                result[
                    "layout_id"
                ] = int(layout_id)


                # ------------------------------------------------
                # CHECKPOINT-ONLY METADATA
                # These will NOT be model columns.
                # ------------------------------------------------

                result[
                    "backend"
                ] = backend_name

                result[
                    "source_row_position"
                ] = source_position

                result[
                    "source_row_index"
                ] = original_source_index

                result[
                    "layout_number"
                ] = int(layout_id)


                # ------------------------------------------------
                # WRITE IMMEDIATELY
                # ------------------------------------------------

                append_checkpoint(
                    result
                )

                completed_keys.add(
                    key
                )

                newly_completed += 1
                row_completed += 1


                # ------------------------------------------------
                # PROGRESS
                # ------------------------------------------------

                completed_total = (
                    len(completed_keys)
                )

                elapsed = (
                    time.time()
                    - start_time
                )

                rate = (
                    completed_total
                    / elapsed
                    if elapsed > 0
                    else 0
                )

                remaining = (
                    EXPECTED_TOTAL
                    - completed_total
                )

                eta_seconds = (
                    remaining / rate
                    if rate > 0
                    else 0
                )


                print(
                    f"[{completed_total:5d}/"
                    f"{EXPECTED_TOTAL}] "
                    f"{backend_name} | "
                    f"row={source_position:3d} | "
                    f"cfg=({reps},{entanglement}) | "
                    f"layout={layout_id} | "
                    f"TVD={result['mean_tvd']:.4f} | "
                    f"ETA={eta_seconds/60:.1f} min"
                )


        # --------------------------------------------------------
        # SOURCE-ROW/BACKEND CHECKPOINT CSV
        # --------------------------------------------------------

        if row_completed > 0:

            try:

                checkpoint_df = pd.DataFrame(
                    checkpoint_records
                    + [
                        r
                        for r in []
                    ]
                )

                # Read the JSONL checkpoint so that
                # CSV always reflects disk, not RAM.
                checkpoint_df = pd.read_json(
                    CHECKPOINT_JSONL,
                    lines=True
                )

                checkpoint_df.to_csv(
                    CHECKPOINT_CSV,
                    index=False
                )

            except Exception as e:

                print(
                    "CSV checkpoint warning:",
                    repr(e)
                )


        row_elapsed = (
            time.time()
            - row_start
        )

        print(
            f"Completed "
            f"{backend_name} source row "
            f"{source_position + 1}/{N_NEW_ROWS} "
            f"| experiments this row: "
            f"{row_completed} "
            f"| row time: "
            f"{row_elapsed/60:.2f} min"
        )


# ============================================================
# 24. LOAD COMPLETE CHECKPOINT FROM DISK
# ============================================================

print("\n")
print("=" * 75)
print("CHECKPOINT EXECUTION FINISHED")
print("=" * 75)

full_checkpoint_df = pd.read_json(
    CHECKPOINT_JSONL,
    lines=True
)

print(
    "Checkpoint rows:",
    len(full_checkpoint_df)
)


# ============================================================
# 25. FINAL ROW COUNT CHECK
# ============================================================

assert (
    len(full_checkpoint_df)
    == EXPECTED_TOTAL
), (
    f"Expected {EXPECTED_TOTAL} rows, "
    f"got {len(full_checkpoint_df)}"
)


# ============================================================
# 26. CHECK BACKEND DISTRIBUTION
# ============================================================

print("\nBackend distribution:")

print(
    full_checkpoint_df[
        "backend"
    ]
    .value_counts()
    .sort_index()
)

assert (
    full_checkpoint_df[
        "backend"
    ]
    .value_counts()
    .sort_index()
    .tolist()
    == [4800, 4800, 4800]
)


# ============================================================
# 27. CHECK LAYOUT DISTRIBUTION
# ============================================================

print("\nBackend × layout distribution:")

print(
    full_checkpoint_df
    .groupby(
        ["backend", "layout_id"]
    )
    .size()
)


# Every backend should have
# 200 × 6 = 1200 per layout.

for backend_name in (
    full_checkpoint_df[
        "backend"
    ].unique()
):

    counts = (
        full_checkpoint_df[
            full_checkpoint_df[
                "backend"
            ] == backend_name
        ]["layout_id"]
        .value_counts()
        .sort_index()
        .tolist()
    )

    assert counts == [
        1200,
        1200,
        1200,
        1200,
    ]


# ============================================================
# 28. CHECK CONFIGURATION DISTRIBUTION
# ============================================================

print(
    "\nBackend × configuration distribution:"
)

print(
    full_checkpoint_df
    .groupby(
        [
            "backend",
            "reps",
            "entanglement",
        ]
    )
    .size()
)


for backend_name in (
    full_checkpoint_df[
        "backend"
    ].unique()
):

    counts = (
        full_checkpoint_df[
            full_checkpoint_df[
                "backend"
            ] == backend_name
        ]
        .groupby(
            [
                "reps",
                "entanglement",
            ]
        )
        .size()
        .sort_index()
        .tolist()
    )

    assert counts == [
        800,
        800,
        800,
        800,
        800,
        800,
    ]


# ============================================================
# 29. CHECK 200 MATCHED SOURCE ROWS
# ============================================================

assert (
    full_checkpoint_df[
        "source_row_position"
    ].nunique()
    == 200
)

assert (
    full_checkpoint_df[
        "source_row_index"
    ].nunique()
    == 200
)


# ============================================================
# 30. EVERY SOURCE ROW SHOULD HAVE 72 EXPERIMENTS
# ============================================================

source_counts = (
    full_checkpoint_df
    .groupby(
        "source_row_position"
    )
    .size()
)

print(
    "\nExperiments per matched source row:"
)

print(
    source_counts.value_counts()
)

assert (
    source_counts.nunique()
    == 1
)

assert (
    source_counts.iloc[0]
    == 72
)


# ============================================================
# 31. CHECK DUPLICATES
# ============================================================

duplicate_keys = (
    full_checkpoint_df
    .duplicated(
        subset=[
            "backend",
            "source_row_position",
            "reps",
            "entanglement",
            "layout_id",
        ]
    )
    .sum()
)

print(
    "\nDuplicate experiment keys:",
    duplicate_keys
)

assert duplicate_keys == 0


# ============================================================
# 32. SAVE COMPLETE FULL CHECKPOINT CSV
# ============================================================

full_checkpoint_df.to_csv(
    FINAL_FULL_CSV,
    index=False
)

print(
    "\nSaved full dataset:"
)

print(
    FINAL_FULL_CSV
)

print(
    "Full shape:",
    full_checkpoint_df.shape
)


# ============================================================
# 33. CREATE EXACT 61-COLUMN MODEL DATASET
#
# Remove metadata-only columns:
# - backend
# - source_row_position
# - source_row_index
# - layout_number
#
# Keep:
# - data_row_index
# - layout_id
# ============================================================

metadata_only_columns = [
    "backend",
    "source_row_position",
    "source_row_index",
    "layout_number",
]

final_61_df = (
    full_checkpoint_df
    .drop(
        columns=metadata_only_columns
    )
    .copy()
)


# ============================================================
# 34. EXACT 61-COLUMN CHECK
# ============================================================

print(
    "\nFinal model dataset:"
)

print(
    "Rows:",
    len(final_61_df)
)

print(
    "Columns:",
    len(final_61_df.columns)
)

assert (
    len(final_61_df)
    == 14400
)

assert (
    len(final_61_df.columns)
    == 61
)


# ============================================================
# 35. SAVE FINAL 61-COLUMN DATASET
# ============================================================

final_61_df.to_csv(
    FINAL_61_CSV,
    index=False
)

print(
    "\nSaved FINAL 61-column dataset:"
)

print(
    FINAL_61_CSV
)


# ============================================================
# 36. SAVE METADATA
# ============================================================

metadata = {

    "dataset_type":
        "cross-backend matched-input",

    "source_rows":
        200,

    "backends": [
        "FakeKingston",
        "FakeMarrakesh",
        "FakeFez",
    ],

    "configurations": [
        {
            "reps": int(reps),
            "entanglement": entanglement,
        }
        for reps, entanglement
        in CONFIGS_MATCHED
    ],

    "layouts": {
        "FakeKingston":
            KINGSTON_MATCHED_LAYOUTS,

        "FakeMarrakesh":
            MARRAKESH_MATCHED_LAYOUTS,

        "FakeFez":
            FEZ_MATCHED_LAYOUTS,
    },

    "experiments_per_backend":
        4800,

    "total_experiments":
        14400,

    "shots":
        1024,

    "repeats":
        5,

    "seed_transpiler":
        42,

    "final_columns":
        61,

    "previous_source_rows_excluded":
        600,

    "new_source_selection_seed":
        2026,

    "backend_column_used_as_model_feature":
        False,

    "checkpoint_file":
        str(CHECKPOINT_JSONL),

    "final_file":
        str(FINAL_61_CSV),
}


with open(
    METADATA_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        default=str
    )


# ============================================================
# 37. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("MATCHED DATASET COMPLETE")
print("=" * 75)

print(
    "FINAL SHAPE:",
    final_61_df.shape
)

print(
    "Expected:",
    (14400, 61)
)

print(
    "\nFiles:"
)

print(
    "Checkpoint JSONL:",
    CHECKPOINT_JSONL
)

print(
    "Checkpoint CSV:",
    CHECKPOINT_CSV
)

print(
    "Full dataset:",
    FINAL_FULL_CSV
)

print(
    "Final 61-column dataset:",
    FINAL_61_CSV
)

print(
    "Metadata:",
    METADATA_JSON
)

print(
    "\nALL CHECKS PASSED."
)

print(
    "The 14,400 matched experiments are complete."
)

MATCHED 14,400-EXPERIMENT DATASET
Source rows: 200
Backends: 3
Configurations: 6
Layouts/backend: 4
Shots: 1024
Repeats: 5
Expected experiments: 14400

Clean source dataframe:
(5880, 36)
Scaler: [0, 2π]

Previous source rows:
Kingston : 200
Marrakesh: 200
Fez      : 200
Combined : 600

Rows available for NEW matched dataset: 5280

NEW matched source rows: 200
Overlap with previous 600: 0
Unique original source indices: 200

Matched raw shape: (200, 5)
Matched scaled shape: (200, 5)

Creating fake backends...

FakeKingston selected layouts:
[10, 11, 12, 13, 14] | mean = 0.001253 | max = 0.00158 | edges = [0.001341, 0.000873, 0.00158, 0.001218]
[61, 62, 63, 64, 65] | mean = 0.001791 | max = 0.002067 | edges = [0.001832, 0.002067, 0.001405, 0.00186]
[147, 148, 149, 150, 151] | mean = 0.002294 | max = 0.002585 | edges = [0.002434, 0.002585, 0.0022, 0.001958]
[83, 84, 85, 86, 87] | mean = 0.012769 | max = 0.035945 | edges = [0.002212, 0.035945, 0.008216, 0.004704]

FakeMarrakesh selected la

## Sanity checks before training a predictor
(PENDING)
Worth confirming before moving on:
- Does `mean_tvd` correlate with `depth` / `cx_count` the way we'd
  expect (more depth/entangling gates → more error)?
- Do `mean_cz_error` / `mean_t1` / `mean_t2` actually vary across the
  dataset (confirming the fully-crossed layout design worked), rather
  than being constant or confounded with row identity?

In [ ]:
print("Correlation of mean_tvd with key structural features:")
print(predictor_df[[
    "mean_tvd", "abstract_depth", "abstract_cx_count",
    "transpiled_depth", "transpiled_cz_count",
    "mean_cz_error", "mean_t1", "mean_t2",
]].corr()["mean_tvd"].sort_values(ascending=False))

print("\nVariation check -- hardware calibration features should NOT be constant:")
print(predictor_df[["mean_cz_error", "mean_t1", "mean_t2"]].describe())

print("\nConfound check -- layout should vary independently of data_row_index:")
print(pd.crosstab(predictor_df["data_row_index"] % 5, predictor_df["layout_id"]).head())


## Three-tier feature ablation

The central scientific question of this study: **does hardware-aware
information improve prediction of noise beyond ordinary circuit
complexity?**

Rather than training one model on the full feature table, we define
three nested feature tiers and compare predictive performance across
them.

In [ ]:
TIER_1_BASELINE = [
    "abstract_n_qubits",
    "abstract_depth",
    "abstract_two_qubit_gate_count",
    "reps",           # will need categorical/numeric encoding for entanglement
]
# entanglement is categorical -- handled via one-hot encoding below

TIER_2_STRUCTURAL = TIER_1_BASELINE + [
    "abstract_total_gate_count",
    "abstract_single_qubit_gate_count",
    "abstract_two_qubit_gate_ratio",
    "abstract_average_gates_per_qubit",
    "abstract_max_gates_per_qubit",
    "abstract_unique_2q_pairs",
    "abstract_average_qubit_degree",
    "abstract_max_qubit_degree",
    "abstract_repeated_2q_pairs",
    "transpiled_depth",
    "transpiled_total_gates",
    "transpiled_1q_gates",
    "transpiled_2q_gates",
    "transpiled_cz_count",
    "transpiled_sx_count",
    "transpiled_rz_count",
]

TIER_3_HARDWARE_AWARE = TIER_2_STRUCTURAL + [
    "mean_cz_error",
    "std_cz_error",
    "min_cz_error",
    "max_cz_error",
    "sum_cz_error",
    "cumulative_cz_error",
    "mean_cz_duration",
    "max_cz_duration",
    "total_cz_duration",
    "physical_qubit_count",
    "mean_t1",
    "min_t1",
    "max_t1",
    "mean_t2",
    "min_t2",
    "max_t2",
]

print("Tier 1 (baseline):", len(TIER_1_BASELINE), "features")
print("Tier 2 (structural):", len(TIER_2_STRUCTURAL), "features")
print("Tier 3 (hardware-aware):", len(TIER_3_HARDWARE_AWARE), "features")


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
import pandas as pd

def prepare_features(df, feature_columns):
    X = df[feature_columns].copy()
    # one-hot encode entanglement if present in this tier
    if "entanglement" in df.columns and "entanglement" not in X.columns:
        pass  # entanglement not in this tier's numeric list
    X = pd.get_dummies(X, columns=[c for c in ["entanglement"] if c in X.columns])
    return X.fillna(0)  # T1/T2 NaNs (if any) filled conservatively; revisit if this matters

def evaluate_tier(df, feature_columns, target="mean_tvd", n_folds=5):
    X = prepare_features(df, feature_columns)
    y = df[target]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    cv = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    scores_r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
    scores_neg_mae = cross_val_score(model, X, y, cv=cv, scoring="neg_mean_absolute_error")

    return {
        "n_features": X.shape[1],
        "mean_r2": scores_r2.mean(),
        "std_r2": scores_r2.std(),
        "mean_mae": -scores_neg_mae.mean(),
    }

# Include entanglement explicitly for tiers that should have it
tier_results = {
    "Tier 1: Baseline": evaluate_tier(predictor_df, TIER_1_BASELINE + ["entanglement"]),
    "Tier 2: Structural": evaluate_tier(predictor_df, TIER_2_STRUCTURAL + ["entanglement"]),
    "Tier 3: Hardware-aware": evaluate_tier(predictor_df, TIER_3_HARDWARE_AWARE + ["entanglement"]),
}

tier_comparison = pd.DataFrame(tier_results).T
print(tier_comparison)


**Interpreting this table:** if `mean_r2` increases meaningfully from
Tier 1 → Tier 2 → Tier 3, that's evidence hardware-aware calibration
data genuinely improves noise prediction beyond circuit structure
alone -- the paper's central claim. If Tier 3 doesn't improve over
Tier 2, that's still a valid, honest, reportable finding (structure
alone may be sufficient, or the hardware features may need
refinement/more data to show their value).

For feature importance *within* a tier (e.g. which hardware features
matter most in Tier 3), prefer **permutation importance** over default
impurity-based importance, since several hardware features (CZ count,
sum/mean/cumulative CZ error) are correlated and impurity-based
importance can split credit misleadingly across them.

In [ ]:
from sklearn.inspection import permutation_importance

X3 = prepare_features(predictor_df, TIER_3_HARDWARE_AWARE + ["entanglement"])
y3 = predictor_df["mean_tvd"]

model3 = RandomForestRegressor(n_estimators=200, random_state=42)
model3.fit(X3, y3)

perm_importance = permutation_importance(
    model3, X3, y3, n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X3.columns,
    "importance_mean": perm_importance.importances_mean,
    "importance_std": perm_importance.importances_std,
}).sort_values("importance_mean", ascending=False)

print(importance_df.head(15).to_string(index=False))
